In [1]:
# Cell 01 — Fresh Architecture-D initialization and persistent runtime configuration
# DECISION: REPLACE
#
# Purpose:
# 1. Restart Architecture-D cleanly.
# 2. Preserve automatic resume across Kaggle kernel restarts.
# 3. Keep exactly one implementation for fast and production modes.
# 4. Put mode-dependent budgets in configuration rather than branching
#    model/training implementations.
# 5. Establish all persistent directories before any expensive operation.
# 6. Keep architecture, training, generation, and submission artifacts
#    separated and reproducible.

import json
import os
import platform
import random
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


# =========================================================================
# 1. EXPERIMENT IDENTITY
# =========================================================================

ARCHITECTURE_NAME = (
    "Architecture_D_TCN_Language_Model"
)

ARCHITECTURE_VERSION = "D_v2"

EXPERIMENT_ID = (
    f"CS781_Phase1_{ARCHITECTURE_VERSION}"
)


# =========================================================================
# 2. RUN MODE
# =========================================================================
#
# fast:
#   Used to validate the COMPLETE pipeline quickly.
#
# production:
#   Uses the same implementation with larger budgets.
#
# IMPORTANT:
# The implementation itself will never branch into a separate
# "fast algorithm" and "production algorithm".
#
# Only configuration values such as token budget, validation budget,
# and generation count will change.

RUN_MODE = "production"

assert RUN_MODE in {
    "fast",
    "production",
}, (
    "RUN_MODE must be either 'fast' or 'production'."
)


# =========================================================================
# 3. ONE-TIME CLEAN RESTART
# =========================================================================
#
# The previous Architecture-D attempt may already have created:
#
#     /kaggle/working/architecture_D
#
# We archive that previous experiment exactly once.
#
# After the new experiment is initialized, ordinary notebook/kernel
# restarts will NEVER delete the active experiment directory.
#
# This gives us:
#
#     first execution
#         -> clean Architecture-D restart
#
#     later execution after interruption
#         -> automatic resume from existing artifacts

BASE_WORKING_ROOT = Path(
    "/kaggle/working"
)

ARCHIVE_ROOT = (
    BASE_WORKING_ROOT
    / "architecture_D_archive"
)

NOTEBOOK_DIR = (
    BASE_WORKING_ROOT
    / "architecture_D"
)

INITIALIZATION_MARKER = (
    NOTEBOOK_DIR
    / ".architecture_D_initialized"
)

if (
    NOTEBOOK_DIR.exists()
    and not INITIALIZATION_MARKER.exists()
):

    ARCHIVE_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    archive_timestamp = (
        datetime.now(
            timezone.utc
        )
        .strftime(
            "%Y%m%d_%H%M%S"
        )
    )

    previous_directory = (
        ARCHIVE_ROOT
        / f"pre_restart_{archive_timestamp}"
    )

    print(
        "Existing Architecture-D directory detected."
    )

    print(
        "Archiving previous D experiment to:"
    )

    print(
        f"  {previous_directory}"
    )

    shutil.move(
        str(NOTEBOOK_DIR),
        str(previous_directory),
    )

    print(
        "✅ Previous Architecture-D artifacts archived."
    )


# =========================================================================
# 4. PERSISTENT DIRECTORY STRUCTURE
# =========================================================================

CHECKPOINTS_DIR = (
    NOTEBOOK_DIR
    / "checkpoints"
)

CONFIGS_DIR = (
    NOTEBOOK_DIR
    / "configs"
)

METRICS_DIR = (
    NOTEBOOK_DIR
    / "metrics"
)

PREDICTIONS_DIR = (
    NOTEBOOK_DIR
    / "predictions"
)

SUBMISSION_DIR = (
    NOTEBOOK_DIR
    / "submission_package"
)

TEST_DATA_DIR = (
    NOTEBOOK_DIR
    / "test_data"
)

LOGS_DIR = (
    NOTEBOOK_DIR
    / "logs"
)


for directory in (
    NOTEBOOK_DIR,
    CHECKPOINTS_DIR,
    CONFIGS_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    SUBMISSION_DIR,
    TEST_DATA_DIR,
    LOGS_DIR,
):

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =========================================================================
# 5. CHECKPOINT PATHS
# =========================================================================

LATEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR
    / "latest.pt"
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR
    / "best.pt"
)

CHECKPOINT_BACKUP_PATH = (
    CHECKPOINTS_DIR
    / "latest_backup.pt"
)


# =========================================================================
# 6. TRAINING ARTIFACT PATHS
# =========================================================================

TRAINING_STATE_PATH = (
    METRICS_DIR
    / "training_state_D.json"
)

TRAINING_HISTORY_PATH = (
    METRICS_DIR
    / "training_history_D.json"
)

VALIDATION_HISTORY_PATH = (
    METRICS_DIR
    / "validation_history_D.json"
)

RUNTIME_LOG_PATH = (
    LOGS_DIR
    / "runtime_D.log"
)


# =========================================================================
# 7. GENERATION ARTIFACT PATHS
# =========================================================================

GENERATION_PROGRESS_PATH = (
    METRICS_DIR
    / "generation_progress_D.json"
)

PARTIAL_PREDICTION_PATH = (
    PREDICTIONS_DIR
    / "prediction_D_partial.json"
)

FINAL_PREDICTION_PATH = (
    PREDICTIONS_DIR
    / "prediction_D.json"
)

SUBMISSION_PREDICTION_PATH = (
    SUBMISSION_DIR
    / "prediction.json"
)

SUBMISSION_ZIP_PATH = (
    NOTEBOOK_DIR
    / "submission.zip"
)


# =========================================================================
# 8. RANDOM SEED
# =========================================================================

SEED = 42


def seed_everything(
    seed,
):
    """
    Establish the experiment seed.

    Full deterministic CUDA algorithms are not forced because that can
    significantly reduce GPU throughput on Kaggle. Exact RNG states will
    later be stored inside training checkpoints.
    """

    seed = int(
        seed
    )

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )


seed_everything(
    SEED
)


# =========================================================================
# 9. DEVICE
# =========================================================================

if torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda"
    )

    GPU_NAME = (
        torch.cuda.get_device_name(
            0
        )
    )

    GPU_MEMORY_GB = (
        torch.cuda.get_device_properties(
            0
        )
        .total_memory
        / (
            1024 ** 3
        )
    )

else:

    DEVICE = torch.device(
        "cpu"
    )

    GPU_NAME = "CPU"

    GPU_MEMORY_GB = 0.0


# =========================================================================
# 10. MODE-DEPENDENT TRAINING BUDGETS
# =========================================================================
#
# These are budgets only.
#
# The same:
#
#     dataset
#     sampler
#     model implementation
#     optimizer implementation
#     checkpoint implementation
#     validation implementation
#
# will be used in both modes.
#
# Production is configured to match the previous Phase-1 runs more
# closely: 100M training tokens and 5M validation tokens.

MODE_CONFIGURATION = {

    "fast": {

        # Small budget for end-to-end pipeline verification.
        "train_tokens": 2_000_000,

        "validation_tokens": 100_000,

        # Shorter sequence makes the first complete run practical.
        # The actual TCN model still supports the production context.
        "training_sequence_length": 512,

        # Microbatch remains one sequence because of the large
        # 1024-channel TCN.
        "micro_batch_size": 1,

        # Gradient accumulation preserves a useful effective batch.
        "gradient_accumulation_steps": 8,

        # Small number of validation batches.
        "validation_batches": 100,

        # Fast official-test smoke test.
        "active_test_count": 2,

        # Fast generation smoke-test target.
        "generation_words": 16,

    },

    "production": {

        "train_tokens": 100_000_000,

        "validation_tokens": 5_000_000,

        "training_sequence_length": 2048,

        "micro_batch_size": 1,

        "gradient_accumulation_steps": 8,

        "validation_batches": 1000,

        # Official Phase-1 test count.
        "active_test_count": 1000,

        # Official generation requirement.
        "generation_words": 128,

    },
}


ACTIVE_MODE_CONFIGURATION = dict(
    MODE_CONFIGURATION[
        RUN_MODE
    ]
)


# =========================================================================
# 11. GENERATION CONSTANTS
# =========================================================================
#
# These are kept here so the later generation code uses exactly the same
# implementation in fast and production.
#
# The production target is the official 128-word requirement.

GENERATION_TARGET_WORDS = int(
    ACTIVE_MODE_CONFIGURATION[
        "generation_words"
    ]
)

GENERATION_MAX_NEW_TOKENS = (
    512
    if RUN_MODE == "fast"
    else 4096
)


# =========================================================================
# 12. INITIAL CONFIGURATION ARTIFACT
# =========================================================================

RUNTIME_CONFIGURATION = {

    "experiment_id": EXPERIMENT_ID,

    "architecture_name": (
        ARCHITECTURE_NAME
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "run_mode": RUN_MODE,

    "seed": SEED,

    "device": str(
        DEVICE
    ),

    "gpu_name": GPU_NAME,

    "gpu_memory_gb": round(
        GPU_MEMORY_GB,
        4,
    ),

    "python_version": (
        sys.version.split()[0]
    ),

    "platform": platform.platform(),

    "pytorch_version": (
        torch.__version__
    ),

    "cuda_version": (
        torch.version.cuda
        if torch.cuda.is_available()
        else None
    ),

    "mode_configuration": (
        ACTIVE_MODE_CONFIGURATION
    ),

    "generation": {

        "target_words": (
            GENERATION_TARGET_WORDS
        ),

        "max_new_tokens": (
            GENERATION_MAX_NEW_TOKENS
        ),
    },

    "paths": {

        "notebook_dir": str(
            NOTEBOOK_DIR
        ),

        "checkpoints_dir": str(
            CHECKPOINTS_DIR
        ),

        "configs_dir": str(
            CONFIGS_DIR
        ),

        "metrics_dir": str(
            METRICS_DIR
        ),

        "predictions_dir": str(
            PREDICTIONS_DIR
        ),

        "submission_dir": str(
            SUBMISSION_DIR
        ),

        "latest_checkpoint": str(
            LATEST_CHECKPOINT_PATH
        ),

        "best_checkpoint": str(
            BEST_CHECKPOINT_PATH
        ),

        "partial_prediction": str(
            PARTIAL_PREDICTION_PATH
        ),

        "final_prediction": str(
            FINAL_PREDICTION_PATH
        ),

        "submission_prediction": str(
            SUBMISSION_PREDICTION_PATH
        ),

        "submission_zip": str(
            SUBMISSION_ZIP_PATH
        ),
    },
}


RUNTIME_CONFIGURATION_PATH = (
    CONFIGS_DIR
    / "runtime_configuration_D.json"
)

with RUNTIME_CONFIGURATION_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        RUNTIME_CONFIGURATION,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 13. INITIALIZATION MARKER
# =========================================================================

INITIALIZATION_MARKER.write_text(
    json.dumps(
        {
            "experiment_id": EXPERIMENT_ID,
            "architecture_version": (
                ARCHITECTURE_VERSION
            ),
            "initialized_utc": (
                datetime.now(
                    timezone.utc
                )
                .isoformat()
            ),
        },
        indent=2,
    ),
    encoding="utf-8",
)


# =========================================================================
# 14. RUNTIME LOG
# =========================================================================

with RUNTIME_LOG_PATH.open(
    "a",
    encoding="utf-8",
) as file:

    file.write(
        "\n"
        + "=" * 70
        + "\n"
    )

    file.write(
        f"Experiment: {EXPERIMENT_ID}\n"
    )

    file.write(
        f"Run mode  : {RUN_MODE}\n"
    )

    file.write(
        f"Device    : {DEVICE}\n"
    )

    file.write(
        f"GPU       : {GPU_NAME}\n"
    )


# =========================================================================
# 15. FINAL REPORT
# =========================================================================

print(
    "=" * 70
)

print(
    "ARCHITECTURE-D RESTART INITIALIZED"
)

print(
    "=" * 70
)

print(
    f"Architecture       : "
    f"{ARCHITECTURE_NAME}"
)

print(
    f"Version            : "
    f"{ARCHITECTURE_VERSION}"
)

print(
    f"Run mode           : "
    f"{RUN_MODE}"
)

print(
    f"Device             : "
    f"{DEVICE}"
)

print(
    f"GPU                : "
    f"{GPU_NAME}"
)

print(
    f"GPU memory         : "
    f"{GPU_MEMORY_GB:.2f} GB"
)

print(
    f"Seed               : "
    f"{SEED}"
)

print(
    f"Train token budget : "
    f"{ACTIVE_MODE_CONFIGURATION['train_tokens']:,}"
)

print(
    f"Validation budget  : "
    f"{ACTIVE_MODE_CONFIGURATION['validation_tokens']:,}"
)

print(
    f"Training seq len   : "
    f"{ACTIVE_MODE_CONFIGURATION['training_sequence_length']}"
)

print(
    f"Microbatch         : "
    f"{ACTIVE_MODE_CONFIGURATION['micro_batch_size']}"
)

print(
    f"Grad accumulation  : "
    f"{ACTIVE_MODE_CONFIGURATION['gradient_accumulation_steps']}"
)

print(
    f"Active test count  : "
    f"{ACTIVE_MODE_CONFIGURATION['active_test_count']}"
)

print(
    f"Generation words   : "
    f"{GENERATION_TARGET_WORDS}"
)

print(
    f"Working directory  : "
    f"{NOTEBOOK_DIR}"
)

print(
    f"Latest checkpoint  : "
    f"{LATEST_CHECKPOINT_PATH}"
)

print(
    f"Best checkpoint    : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"Generation partial : "
    f"{PARTIAL_PREDICTION_PATH}"
)

print(
    f"Runtime config     : "
    f"{RUNTIME_CONFIGURATION_PATH}"
)

print(
    f"Previous archives  : "
    f"{ARCHIVE_ROOT}"
)

print(
    "\n✅ Fresh Architecture-D experiment initialized."
)

print(
    "✅ Ordinary kernel restarts will preserve checkpoints."
)

print(
    "✅ Fast and production use the same implementation."
)

print(
    "✅ No model training started."
)

print(
    "=" * 70
)

ARCHITECTURE-D RESTART INITIALIZED
Architecture       : Architecture_D_TCN_Language_Model
Version            : D_v2
Run mode           : production
Device             : cuda
GPU                : Tesla T4
GPU memory         : 14.56 GB
Seed               : 42
Train token budget : 100,000,000
Validation budget  : 5,000,000
Training seq len   : 2048
Microbatch         : 1
Grad accumulation  : 8
Active test count  : 1000
Generation words   : 128
Working directory  : /kaggle/working/architecture_D
Latest checkpoint  : /kaggle/working/architecture_D/checkpoints/latest.pt
Best checkpoint    : /kaggle/working/architecture_D/checkpoints/best.pt
Generation partial : /kaggle/working/architecture_D/predictions/prediction_D_partial.json
Runtime config     : /kaggle/working/architecture_D/configs/runtime_configuration_D.json
Previous archives  : /kaggle/working/architecture_D_archive

✅ Fresh Architecture-D experiment initialized.
✅ Ordinary kernel restarts will preserve checkpoints.
✅ Fast and produ

In [2]:
# Cell 02 — Verify environment, GPU, dependencies, and Hugging Face authentication
# DECISION: ADD
#
# Purpose:
# 1. Verify the Kaggle runtime after the Architecture-D restart.
# 2. Verify CUDA/T4 availability.
# 3. Verify required Python packages.
# 4. Load the Hugging Face token from Kaggle Secrets using key "HF".
# 5. Authenticate with Hugging Face without ever printing the token.
# 6. Save a reproducible environment manifest.
#
# IMPORTANT:
# - This cell is identical in fast and production modes.
# - No model, corpus, or training setup is changed here.
# - The HF token is never written to disk.

import importlib
import json
import os
import platform
import sys
from pathlib import Path

import torch


# =========================================================================
# 1. BASIC RUNTIME INFORMATION
# =========================================================================

print("=" * 70)
print("ARCHITECTURE-D ENVIRONMENT CHECK")
print("=" * 70)

print(
    f"Python version     : "
    f"{sys.version.split()[0]}"
)

print(
    f"Platform           : "
    f"{platform.platform()}"
)

print(
    f"PyTorch version    : "
    f"{torch.__version__}"
)


# =========================================================================
# 2. CUDA VERIFICATION
# =========================================================================

CUDA_AVAILABLE = bool(
    torch.cuda.is_available()
)

print(
    f"CUDA available     : "
    f"{CUDA_AVAILABLE}"
)

if not CUDA_AVAILABLE:

    raise RuntimeError(
        "CUDA is not available. "
        "Architecture-D is configured for GPU execution."
    )


CUDA_DEVICE_INDEX = 0

DEVICE = torch.device(
    f"cuda:{CUDA_DEVICE_INDEX}"
)

GPU_NAME = (
    torch.cuda.get_device_name(
        CUDA_DEVICE_INDEX
    )
)

GPU_PROPERTIES = (
    torch.cuda.get_device_properties(
        CUDA_DEVICE_INDEX
    )
)

GPU_MEMORY_GB = (
    GPU_PROPERTIES.total_memory
    / (
        1024 ** 3
    )
)

GPU_COMPUTE_CAPABILITY = (
    f"{GPU_PROPERTIES.major}."
    f"{GPU_PROPERTIES.minor}"
)

CUDA_VERSION = (
    torch.version.cuda
)


print(
    f"GPU                : "
    f"{GPU_NAME}"
)

print(
    f"GPU memory         : "
    f"{GPU_MEMORY_GB:.2f} GB"
)

print(
    f"Compute capability : "
    f"{GPU_COMPUTE_CAPABILITY}"
)

print(
    f"CUDA version       : "
    f"{CUDA_VERSION}"
)


# =========================================================================
# 3. REQUIRED PACKAGE VERIFICATION
# =========================================================================

REQUIRED_PACKAGES = {

    "numpy": "numpy",

    "pandas": "pandas",

    "torch": "torch",

    "transformers": "transformers",

    "tokenizers": "tokenizers",

    "huggingface_hub": "huggingface_hub",
}


PACKAGE_VERSIONS = {}

print(
    "\n" + "-" * 70
)

print(
    "PACKAGE CHECK"
)

print(
    "-" * 70
)


for package_name, module_name in (
    REQUIRED_PACKAGES.items()
):

    try:

        module = importlib.import_module(
            module_name
        )

    except ImportError as exc:

        raise ImportError(
            f"Required package '{package_name}' "
            f"is not installed."
        ) from exc

    package_version = getattr(
        module,
        "__version__",
        "unknown",
    )

    PACKAGE_VERSIONS[
        package_name
    ] = package_version

    print(
        f"✅ {package_name:<20} "
        f"{package_version}"
    )


# =========================================================================
# 4. VERIFY PYTORCH CUDA TENSOR EXECUTION
# =========================================================================

print(
    "\n" + "-" * 70
)

print(
    "CUDA TENSOR TEST"
)

print(
    "-" * 70
)


cuda_test_tensor = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=DEVICE,
)

cuda_test_result = (
    cuda_test_tensor.sum()
)

assert cuda_test_result.device.type == (
    "cuda"
)

assert torch.isfinite(
    cuda_test_result
).item()

print(
    f"✅ CUDA tensor test passed on "
    f"{GPU_NAME}."
)


# =========================================================================
# 5. LOAD KAGGLE HUGGING FACE SECRET
# =========================================================================
#
# The secret name is fixed by the project setup:
#
#     HF
#
# Never print HF_TOKEN or write it to a configuration file.

print(
    "\n" + "-" * 70
)

print(
    "HUGGING FACE AUTHENTICATION"
)

print(
    "-" * 70
)


from kaggle_secrets import UserSecretsClient

SECRETS_CLIENT = (
    UserSecretsClient()
)

try:

    HF_TOKEN = (
        SECRETS_CLIENT.get_secret(
            "HF"
        )
    )

except Exception as exc:

    raise RuntimeError(
        "Unable to read Kaggle Secret 'HF'. "
        "Verify that the Kaggle secret exists "
        "and is named exactly 'HF'."
    ) from exc


if not isinstance(
    HF_TOKEN,
    str,
):

    raise RuntimeError(
        "Kaggle Secret 'HF' did not return a string."
    )


HF_TOKEN = HF_TOKEN.strip()


if not HF_TOKEN:

    raise RuntimeError(
        "Kaggle Secret 'HF' is empty."
    )


print(
    "✅ Kaggle secret 'HF' loaded successfully."
)

print(
    "✅ Secret value was not printed."
)


# =========================================================================
# 6. HUGGING FACE LOGIN
# =========================================================================
#
# Authentication is performed only in memory.
# add_to_git_credential=False prevents the token from being added
# to Git credentials.

from huggingface_hub import login

try:

    login(
        token=HF_TOKEN,
        add_to_git_credential=False,
    )

except Exception as exc:

    raise RuntimeError(
        "Hugging Face authentication failed "
        "using Kaggle Secret 'HF'."
    ) from exc


print(
    "✅ Hugging Face authentication successful."
)


# =========================================================================
# 7. DO NOT STORE THE TOKEN
# =========================================================================
#
# We deliberately do NOT put HF_TOKEN into any JSON artifact.
#
# The environment manifest records only:
#
#     hf_secret_key = "HF"
#
# This allows later cells to know which secret was used without leaking
# credentials.

HF_SECRET_KEY_NAME = "HF"


# =========================================================================
# 8. VERIFY PROJECT DIRECTORIES
# =========================================================================

REQUIRED_DIRECTORIES = {

    "notebook_dir": NOTEBOOK_DIR,

    "checkpoints_dir": CHECKPOINTS_DIR,

    "configs_dir": CONFIGS_DIR,

    "metrics_dir": METRICS_DIR,

    "predictions_dir": PREDICTIONS_DIR,

    "submission_dir": SUBMISSION_DIR,

    "test_data_dir": TEST_DATA_DIR,

    "logs_dir": LOGS_DIR,
}


print(
    "\n" + "-" * 70
)

print(
    "PROJECT DIRECTORY CHECK"
)

print(
    "-" * 70
)


for directory_name, directory_path in (
    REQUIRED_DIRECTORIES.items()
):

    directory_path = Path(
        directory_path
    )

    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    assert directory_path.is_dir()

    print(
        f"✅ {directory_name:<20} "
        f"{directory_path}"
    )


# =========================================================================
# 9. VERIFY CORE RUNTIME VARIABLES
# =========================================================================

assert "RUN_MODE" in globals()

assert RUN_MODE in {
    "fast",
    "production",
}

assert "EXPERIMENT_ID" in globals()

assert "ARCHITECTURE_NAME" in globals()

assert "ARCHITECTURE_VERSION" in globals()

assert "SEED" in globals()


# =========================================================================
# 10. SAVE ENVIRONMENT MANIFEST
# =========================================================================

ENVIRONMENT_MANIFEST = {

    "experiment_id": (
        EXPERIMENT_ID
    ),

    "architecture_name": (
        ARCHITECTURE_NAME
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "run_mode": (
        RUN_MODE
    ),

    "python_version": (
        sys.version.split()[0]
    ),

    "platform": (
        platform.platform()
    ),

    "pytorch_version": (
        torch.__version__
    ),

    "cuda_available": (
        CUDA_AVAILABLE
    ),

    "cuda_version": (
        CUDA_VERSION
    ),

    "device": str(
        DEVICE
    ),

    "gpu_name": (
        GPU_NAME
    ),

    "gpu_memory_gb": round(
        GPU_MEMORY_GB,
        4,
    ),

    "gpu_compute_capability": (
        GPU_COMPUTE_CAPABILITY
    ),

    "seed": int(
        SEED
    ),

    "hf_secret_key": (
        HF_SECRET_KEY_NAME
    ),

    "hf_token_loaded": True,

    "packages": (
        PACKAGE_VERSIONS
    ),
}


ENVIRONMENT_MANIFEST_PATH = (
    CONFIGS_DIR
    / "environment_manifest_D.json"
)


with ENVIRONMENT_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        ENVIRONMENT_MANIFEST,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 11. APPEND TO RUNTIME LOG
# =========================================================================

with RUNTIME_LOG_PATH.open(
    "a",
    encoding="utf-8",
) as file:

    file.write(
        "\n"
        + "=" * 70
        + "\n"
    )

    file.write(
        "ARCHITECTURE-D ENVIRONMENT VERIFIED\n"
    )

    file.write(
        f"Experiment : {EXPERIMENT_ID}\n"
    )

    file.write(
        f"Run mode   : {RUN_MODE}\n"
    )

    file.write(
        f"Device     : {DEVICE}\n"
    )

    file.write(
        f"GPU        : {GPU_NAME}\n"
    )

    file.write(
        f"PyTorch    : {torch.__version__}\n"
    )


# =========================================================================
# 12. CLEAN UP SENSITIVE TOKEN VARIABLE
# =========================================================================
#
# Later cells do not need the token directly. Hugging Face authentication
# has already been established.

HF_TOKEN = None


# =========================================================================
# 13. FINAL REPORT
# =========================================================================

print(
    "\n" + "=" * 70
)

print(
    "✅ CELL 02 COMPLETE"
)

print(
    "=" * 70
)

print(
    f"Device             : "
    f"{DEVICE}"
)

print(
    f"GPU                : "
    f"{GPU_NAME}"
)

print(
    f"GPU memory         : "
    f"{GPU_MEMORY_GB:.2f} GB"
)

print(
    f"PyTorch            : "
    f"{torch.__version__}"
)

print(
    f"CUDA               : "
    f"{CUDA_VERSION}"
)

print(
    f"HF secret key      : "
    f"{HF_SECRET_KEY_NAME}"
)

print(
    f"Environment manifest: "
    f"{ENVIRONMENT_MANIFEST_PATH}"
)

print(
    "✅ CUDA execution verified."
)

print(
    "✅ Required packages verified."
)

print(
    "✅ Hugging Face authentication verified."
)

print(
    "✅ Sensitive token removed from runtime variable."
)

print(
    "=" * 70
)

ARCHITECTURE-D ENVIRONMENT CHECK
Python version     : 3.12.13
Platform           : Linux-6.12.90+-x86_64-with-glibc2.35
PyTorch version    : 2.10.0+cu128
CUDA available     : True
GPU                : Tesla T4
GPU memory         : 14.56 GB
Compute capability : 7.5
CUDA version       : 12.8

----------------------------------------------------------------------
PACKAGE CHECK
----------------------------------------------------------------------
✅ numpy                2.0.2
✅ pandas               2.3.3
✅ torch                2.10.0+cu128
✅ transformers         5.0.0
✅ tokenizers           0.22.2
✅ huggingface_hub      1.11.0

----------------------------------------------------------------------
CUDA TENSOR TEST
----------------------------------------------------------------------
✅ CUDA tensor test passed on Tesla T4.

----------------------------------------------------------------------
HUGGING FACE AUTHENTICATION
----------------------------------------------------------------------

In [3]:
# Cell 03 — Load and validate the shared Phase-1 tokenizer and token shards
# DECISION: ADD
#
# Purpose:
# 1. Load the exact tokenizer already used by Architectures A/B/C.
# 2. Load the already-tokenized official Phase-1 corpus.
# 3. Validate vocabulary and special-token IDs.
# 4. Validate all four official training sources.
# 5. Use memory-mapped uint16 arrays so billions of tokens are never
#    loaded completely into RAM.
# 6. Define the common source mixture used by Architecture-D.
# 7. Save a persistent data manifest and fingerprint for checkpoint
#    compatibility and restart recovery.
#
# IMPORTANT:
# - No tokenizer training occurs.
# - No corpus retokenization occurs.
# - Fast and production use exactly the same data implementation.
# - Only the later training budget changes with RUN_MODE.

import hashlib
import json
from pathlib import Path

import numpy as np
import torch
from tokenizers import Tokenizer


# =========================================================================
# 1. DATASET ROOT DISCOVERY
# =========================================================================

KNOWN_DATASET_ROOT = Path(
    "/kaggle/input/datasets/"
    "rohitbatra2004/"
    "cs781-phase1-tokenized-corpus"
)

DATASET_ROOT = None

if KNOWN_DATASET_ROOT.exists():
    DATASET_ROOT = KNOWN_DATASET_ROOT


# -------------------------------------------------------------------------
# Fallback discovery
# -------------------------------------------------------------------------
#
# This makes the notebook robust if Kaggle mounts the dataset under a
# slightly different directory while preserving the exact same dataset.
# -------------------------------------------------------------------------

if DATASET_ROOT is None:

    INPUT_ROOT = Path(
        "/kaggle/input"
    )

    POSSIBLE_ROOTS = []

    try:

        for candidate in INPUT_ROOT.rglob(
            "tokenization_manifest.json"
        ):

            parent = candidate.parent

            # Expected structure:
            #
            # dataset_root/
            #     metadata/
            #         tokenization_manifest.json
            #
            if (
                parent.name.lower()
                == "metadata"
            ):

                possible_root = (
                    parent.parent
                )

                POSSIBLE_ROOTS.append(
                    possible_root
                )

    except Exception:
        POSSIBLE_ROOTS = []

    if POSSIBLE_ROOTS:

        POSSIBLE_ROOTS = sorted(
            set(POSSIBLE_ROOTS),
            key=lambda path: str(path).lower(),
        )

        DATASET_ROOT = (
            POSSIBLE_ROOTS[0]
        )


if DATASET_ROOT is None:

    raise FileNotFoundError(
        "The shared cs781 Phase-1 tokenized corpus dataset "
        "could not be located under /kaggle/input."
    )


# =========================================================================
# 2. DEFINE DATASET PATHS
# =========================================================================

TOKENIZER_PATH = (
    DATASET_ROOT
    / "tokenizer"
    / "tokenizer.json"
)

TOKENIZER_METADATA_PATH = (
    DATASET_ROOT
    / "metadata"
    / "tokenizer_metadata.json"
)

TOKENIZATION_MANIFEST_PATH = (
    DATASET_ROOT
    / "metadata"
    / "tokenization_manifest.json"
)

TOKENIZED_ROOT = (
    DATASET_ROOT
    / "tokenized"
)


assert TOKENIZER_PATH.is_file(), (
    f"Tokenizer not found: {TOKENIZER_PATH}"
)

assert TOKENIZATION_MANIFEST_PATH.is_file(), (
    "Tokenization manifest not found: "
    f"{TOKENIZATION_MANIFEST_PATH}"
)

assert TOKENIZED_ROOT.is_dir(), (
    f"Tokenized directory not found: {TOKENIZED_ROOT}"
)


# =========================================================================
# 3. LOAD TOKENIZER
# =========================================================================

print("=" * 70)
print("ARCHITECTURE-D SHARED DATA INITIALIZATION")
print("=" * 70)

print(
    f"Dataset root       : {DATASET_ROOT}"
)

print(
    f"Tokenizer          : {TOKENIZER_PATH}"
)

print(
    f"Tokenization manifest: "
    f"{TOKENIZATION_MANIFEST_PATH}"
)


tokenizer = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

VOCAB_SIZE = int(
    tokenizer.get_vocab_size()
)

assert VOCAB_SIZE > 0

print(
    f"\nVocabulary size    : {VOCAB_SIZE:,}"
)


# =========================================================================
# 4. SPECIAL TOKEN IDS
# =========================================================================

def resolve_token_id(
    *candidates,
):
    """
    Resolve the first token that exists in the tokenizer vocabulary.
    """

    for token in candidates:

        try:

            token_id = tokenizer.token_to_id(
                token
            )

        except Exception:

            token_id = None

        if token_id is not None:

            return int(
                token_id
            )

    return None


PAD_ID = resolve_token_id(
    "[PAD]",
    "<pad>",
)

UNK_ID = resolve_token_id(
    "[UNK]",
    "<unk>",
)

BOS_ID = resolve_token_id(
    "[BOS]",
    "<bos>",
)

EOS_ID = resolve_token_id(
    "[EOS]",
    "<eos>",
)


assert UNK_ID is not None, (
    "Unable to resolve the tokenizer UNK token."
)

assert EOS_ID is not None, (
    "Unable to resolve the tokenizer EOS token."
)


print(
    "\n" + "-" * 70
)

print(
    "SPECIAL TOKENS"
)

print(
    "-" * 70
)

print(
    f"PAD ID             : {PAD_ID}"
)

print(
    f"UNK ID             : {UNK_ID}"
)

print(
    f"BOS ID             : {BOS_ID}"
)

print(
    f"EOS ID             : {EOS_ID}"
)


# =========================================================================
# 5. LOAD TOKENIZATION MANIFEST
# =========================================================================

with TOKENIZATION_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file:

    TOKENIZATION_MANIFEST = json.load(
        file
    )


assert isinstance(
    TOKENIZATION_MANIFEST,
    dict,
), (
    "Tokenization manifest must be a JSON object."
)


# =========================================================================
# 6. OFFICIAL SOURCE DEFINITIONS
# =========================================================================
#
# These correspond to the four official Phase-1 training sources:
#
#   Acts
#   High Courts
#   Supreme Court
#   Tribunals
#
# The capstone defines these as the official Phase-1 training sources.

SOURCE_NAMES = [
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
]


TRAIN_SHARD_PATHS = {}

VALIDATION_SHARD_PATHS = {}

for source_name in SOURCE_NAMES:

    train_path = (
        TOKENIZED_ROOT
        / f"{source_name}_train.bin"
    )

    validation_path = (
        TOKENIZED_ROOT
        / f"{source_name}_val.bin"
    )

    assert train_path.is_file(), (
        f"Training shard missing: {train_path}"
    )

    assert validation_path.is_file(), (
        f"Validation shard missing: {validation_path}"
    )

    TRAIN_SHARD_PATHS[
        source_name
    ] = train_path

    VALIDATION_SHARD_PATHS[
        source_name
    ] = validation_path


# =========================================================================
# 7. TOKEN STORAGE VALIDATION
# =========================================================================

TOKEN_DTYPE = np.uint16

TOKEN_DTYPE_BYTES = np.dtype(
    TOKEN_DTYPE
).itemsize

assert TOKEN_DTYPE_BYTES == 2

assert VOCAB_SIZE <= (
    np.iinfo(
        TOKEN_DTYPE
    ).max
    + 1
), (
    "The vocabulary cannot be represented by uint16."
)


# =========================================================================
# 8. CREATE MEMORY-MAPPED SHARDS
# =========================================================================
#
# np.memmap provides random-access views over the large binary files.
#
# Only the portions actually accessed by a training batch are brought into
# memory.

TRAIN_MEMMAPS = {}

VALIDATION_MEMMAPS = {}

SHARD_METADATA = {}


print(
    "\n" + "-" * 70
)

print(
    "TOKEN SHARD INVENTORY"
)

print(
    "-" * 70
)


for source_name in SOURCE_NAMES:

    train_path = (
        TRAIN_SHARD_PATHS[
            source_name
        ]
    )

    validation_path = (
        VALIDATION_SHARD_PATHS[
            source_name
        ]
    )

    train_size_bytes = (
        train_path.stat().st_size
    )

    validation_size_bytes = (
        validation_path.stat().st_size
    )

    assert (
        train_size_bytes
        % TOKEN_DTYPE_BYTES
        == 0
    )

    assert (
        validation_size_bytes
        % TOKEN_DTYPE_BYTES
        == 0
    )

    train_token_count = (
        train_size_bytes
        // TOKEN_DTYPE_BYTES
    )

    validation_token_count = (
        validation_size_bytes
        // TOKEN_DTYPE_BYTES
    )

    assert train_token_count > 0

    assert validation_token_count > 0

    TRAIN_MEMMAPS[
        source_name
    ] = np.memmap(
        train_path,
        dtype=TOKEN_DTYPE,
        mode="r",
    )

    VALIDATION_MEMMAPS[
        source_name
    ] = np.memmap(
        validation_path,
        dtype=TOKEN_DTYPE,
        mode="r",
    )

    assert len(
        TRAIN_MEMMAPS[
            source_name
        ]
    ) == train_token_count

    assert len(
        VALIDATION_MEMMAPS[
            source_name
        ]
    ) == validation_token_count

    SHARD_METADATA[
        source_name
    ] = {

        "train": {

            "path": str(
                train_path
            ),

            "tokens": int(
                train_token_count
            ),

            "bytes": int(
                train_size_bytes
            ),

            "dtype": "uint16",
        },

        "validation": {

            "path": str(
                validation_path
            ),

            "tokens": int(
                validation_token_count
            ),

            "bytes": int(
                validation_size_bytes
            ),

            "dtype": "uint16",
        },
    }

    print(
        f"{source_name:<16} "
        f"train={train_token_count:>15,} "
        f"val={validation_token_count:>12,}"
    )


# =========================================================================
# 9. TOTAL TOKEN COUNTS
# =========================================================================

TOTAL_TRAIN_TOKENS = sum(
    SHARD_METADATA[
        source_name
    ]["train"]["tokens"]
    for source_name in SOURCE_NAMES
)

TOTAL_VALIDATION_TOKENS = sum(
    SHARD_METADATA[
        source_name
    ]["validation"]["tokens"]
    for source_name in SOURCE_NAMES
)


print(
    "\n" + "-" * 70
)

print(
    "TOTAL AVAILABLE TOKENS"
)

print(
    "-" * 70
)

print(
    f"Training tokens    : "
    f"{TOTAL_TRAIN_TOKENS:,}"
)

print(
    f"Validation tokens  : "
    f"{TOTAL_VALIDATION_TOKENS:,}"
)


# =========================================================================
# 10. QUICK TOKEN-ID VALIDATION
# =========================================================================
#
# We validate a bounded prefix of every shard. This catches corruption,
# wrong dtype, and out-of-range IDs without scanning billions of tokens.

SAMPLE_TOKEN_COUNT = 4096


def validate_memmap_sample(
    array,
    source_name,
    split_name,
):
    """
    Validate a small prefix of a token shard.
    """

    count = min(
        SAMPLE_TOKEN_COUNT,
        len(array),
    )

    if count <= 0:

        raise RuntimeError(
            f"{source_name}/{split_name} is empty."
        )

    sample = np.asarray(
        array[:count],
        dtype=np.int64,
    )

    if np.any(
        sample < 0
    ):

        raise RuntimeError(
            f"{source_name}/{split_name}: "
            "negative token ID detected."
        )

    if np.any(
        sample >= VOCAB_SIZE
    ):

        raise RuntimeError(
            f"{source_name}/{split_name}: "
            "token ID exceeds vocabulary."
        )

    return (
        int(sample.min()),
        int(sample.max()),
        int(count),
    )


for source_name in SOURCE_NAMES:

    train_min, train_max, train_count = (
        validate_memmap_sample(
            TRAIN_MEMMAPS[
                source_name
            ],
            source_name,
            "train",
        )
    )

    val_min, val_max, val_count = (
        validate_memmap_sample(
            VALIDATION_MEMMAPS[
                source_name
            ],
            source_name,
            "validation",
        )
    )

    print(
        f"✅ {source_name:<16} "
        f"train IDs={train_min}..{train_max} | "
        f"val IDs={val_min}..{val_max}"
    )


# =========================================================================
# 11. DEFAULT FOUR-SOURCE MIXTURE
# =========================================================================
#
# This is a baseline source mixture, not a result of tuning.
#
# Every official source contributes equally to the sampling distribution.
# The same sampler implementation is later used for fast and production.

SOURCE_PROBABILITIES = np.full(
    len(SOURCE_NAMES),
    1.0 / len(SOURCE_NAMES),
    dtype=np.float64,
)

SOURCE_PROBABILITIES /= (
    SOURCE_PROBABILITIES.sum()
)

SOURCE_NAMES_LIST = list(
    SOURCE_NAMES
)


assert np.isclose(
    SOURCE_PROBABILITIES.sum(),
    1.0,
)


# =========================================================================
# 12. DATA FINGERPRINT
# =========================================================================
#
# A fingerprint detects accidental changes to the tokenizer or source
# shards between notebook sessions.

DATA_FINGERPRINT_PAYLOAD = {

    "dataset_root": str(
        DATASET_ROOT
    ),

    "tokenizer_path": str(
        TOKENIZER_PATH
    ),

    "vocab_size": int(
        VOCAB_SIZE
    ),

    "special_token_ids": {

        "pad": PAD_ID,

        "unk": UNK_ID,

        "bos": BOS_ID,

        "eos": EOS_ID,
    },

    "token_dtype": "uint16",

    "sources": SOURCE_NAMES_LIST,

    "shards": SHARD_METADATA,
}


DATA_FINGERPRINT = hashlib.sha256(
    json.dumps(
        DATA_FINGERPRINT_PAYLOAD,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


# =========================================================================
# 13. PERSISTENT DATA MANIFEST
# =========================================================================

DATA_MANIFEST_D = {

    "experiment_id": (
        EXPERIMENT_ID
    ),

    "architecture_name": (
        ARCHITECTURE_NAME
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "dataset_root": str(
        DATASET_ROOT
    ),

    "tokenizer_path": str(
        TOKENIZER_PATH
    ),

    "tokenizer_metadata_path": str(
        TOKENIZER_METADATA_PATH
    ),

    "tokenization_manifest_path": str(
        TOKENIZATION_MANIFEST_PATH
    ),

    "vocab_size": int(
        VOCAB_SIZE
    ),

    "special_token_ids": {

        "pad": PAD_ID,

        "unk": UNK_ID,

        "bos": BOS_ID,

        "eos": EOS_ID,
    },

    "token_dtype": "uint16",

    "source_names": list(
        SOURCE_NAMES_LIST
    ),

    "source_probabilities": (
        SOURCE_PROBABILITIES.tolist()
    ),

    "total_train_tokens": int(
        TOTAL_TRAIN_TOKENS
    ),

    "total_validation_tokens": int(
        TOTAL_VALIDATION_TOKENS
    ),

    "shards": SHARD_METADATA,

    "data_fingerprint": (
        DATA_FINGERPRINT
    ),
}


DATA_MANIFEST_PATH = (
    CONFIGS_DIR
    / "data_manifest_D.json"
)


with DATA_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        DATA_MANIFEST_D,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 14. SAVE TOKENIZER INFORMATION
# =========================================================================

TOKENIZER_INFO = {

    "tokenizer_path": str(
        TOKENIZER_PATH
    ),

    "vocab_size": int(
        VOCAB_SIZE
    ),

    "pad_id": PAD_ID,

    "unk_id": UNK_ID,

    "bos_id": BOS_ID,

    "eos_id": EOS_ID,

    "data_fingerprint": (
        DATA_FINGERPRINT
    ),
}


TOKENIZER_INFO_PATH = (
    CONFIGS_DIR
    / "tokenizer_info_D.json"
)


with TOKENIZER_INFO_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        TOKENIZER_INFO,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 15. FINAL STATUS
# =========================================================================

print(
    "\n" + "=" * 70
)

print(
    "✅ CELL 03 COMPLETE"
)

print(
    "=" * 70
)

print(
    f"Dataset root       : "
    f"{DATASET_ROOT}"
)

print(
    f"Tokenizer          : "
    f"{TOKENIZER_PATH}"
)

print(
    f"Vocabulary         : "
    f"{VOCAB_SIZE:,}"
)

print(
    f"Training tokens    : "
    f"{TOTAL_TRAIN_TOKENS:,}"
)

print(
    f"Validation tokens  : "
    f"{TOTAL_VALIDATION_TOKENS:,}"
)

print(
    f"Sources            : "
    f"{SOURCE_NAMES_LIST}"
)

print(
    f"Source probabilities: "
    f"{SOURCE_PROBABILITIES.tolist()}"
)

print(
    f"Token dtype        : "
    f"{TOKEN_DTYPE}"
)

print(
    f"Data fingerprint   : "
    f"{DATA_FINGERPRINT[:20]}..."
)

print(
    f"Data manifest      : "
    f"{DATA_MANIFEST_PATH}"
)

print(
    f"Tokenizer info     : "
    f"{TOKENIZER_INFO_PATH}"
)

print(
    "\n✅ Shared tokenizer locked."
)

print(
    "✅ All four official Phase-1 sources locked."
)

print(
    "✅ Memory-mapped token access ready."
)

print(
    "✅ No retokenization performed."
)

print(
    "✅ No corpus loaded fully into RAM."
)

print(
    "=" * 70
)

ARCHITECTURE-D SHARED DATA INITIALIZATION
Dataset root       : /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus
Tokenizer          : /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/tokenizer/tokenizer.json
Tokenization manifest: /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/metadata/tokenization_manifest.json

Vocabulary size    : 50,000

----------------------------------------------------------------------
SPECIAL TOKENS
----------------------------------------------------------------------
PAD ID             : 0
UNK ID             : 1
BOS ID             : 2
EOS ID             : 3

----------------------------------------------------------------------
TOKEN SHARD INVENTORY
----------------------------------------------------------------------
acts             train=    149,598,391 val=   3,405,295
high_courts      train=  2,327,101,667 val=  42,068,902
supreme_court    train=    251,833,809 val=   5,313,054
tribunals       

In [4]:
# Cell 04 — Build the checkpointable multi-source sequence sampler
# DECISION: ADD
#
# Purpose:
# 1. Provide memory-safe random access to the official token corpus.
# 2. Sample across Acts, High Courts, Supreme Court, and Tribunals.
# 3. Produce autoregressive language-model pairs:
#
#       input  = x[t : t + T]
#       target = x[t+1 : t + T + 1]
#
# 4. Preserve exact sampler RNG state in checkpoints.
# 5. Resume from the exact sampling stream after a Kaggle restart.
# 6. Support both fast and production with the same implementation.
# 7. Never load the complete corpus into RAM.
#
# IMPORTANT:
# - The only mode-dependent values come from RUN_MODE configuration.
# - The sampler implementation itself never branches by mode.
# - Existing tokenized shards are read-only.

import copy
import json
from pathlib import Path

import numpy as np
import torch


# =========================================================================
# 1. RESOLVE TRAINING SEQUENCE CONFIGURATION
# =========================================================================

assert "ACTIVE_MODE_CONFIGURATION" in globals(), (
    "ACTIVE_MODE_CONFIGURATION is unavailable. "
    "Run Cell 01 first."
)

assert "SOURCE_NAMES_LIST" in globals(), (
    "SOURCE_NAMES_LIST is unavailable. "
    "Run Cell 03 first."
)

assert "SOURCE_PROBABILITIES" in globals(), (
    "SOURCE_PROBABILITIES is unavailable. "
    "Run Cell 03 first."
)

assert "TRAIN_MEMMAPS" in globals(), (
    "TRAIN_MEMMAPS is unavailable. "
    "Run Cell 03 first."
)

assert "VALIDATION_MEMMAPS" in globals(), (
    "VALIDATION_MEMMAPS is unavailable. "
    "Run Cell 03 first."
)


CONTEXT_LENGTH = int(
    ACTIVE_MODE_CONFIGURATION[
        "training_sequence_length"
    ]
)

MICRO_BATCH_SIZE = int(
    ACTIVE_MODE_CONFIGURATION[
        "micro_batch_size"
    ]
)

GRADIENT_ACCUMULATION_STEPS = int(
    ACTIVE_MODE_CONFIGURATION[
        "gradient_accumulation_steps"
    ]
)

TRAIN_TOKEN_BUDGET = int(
    ACTIVE_MODE_CONFIGURATION[
        "train_tokens"
    ]
)

VALIDATION_TOKEN_BUDGET = int(
    ACTIVE_MODE_CONFIGURATION[
        "validation_tokens"
    ]
)

VALIDATION_BATCHES = int(
    ACTIVE_MODE_CONFIGURATION[
        "validation_batches"
    ]
)


assert CONTEXT_LENGTH > 0
assert MICRO_BATCH_SIZE > 0
assert GRADIENT_ACCUMULATION_STEPS > 0
assert TRAIN_TOKEN_BUDGET > 0
assert VALIDATION_TOKEN_BUDGET > 0
assert VALIDATION_BATCHES > 0


# =========================================================================
# 2. DERIVE SAMPLING LENGTHS
# =========================================================================
#
# Each sample consumes:
#
#     context_length input tokens
#     context_length target tokens
#
# But the underlying shard requires:
#
#     context_length + 1
#
# consecutive tokens.

SAMPLING_WINDOW_LENGTH = (
    CONTEXT_LENGTH + 1
)


# =========================================================================
# 3. CHECK THAT EVERY SOURCE CAN SUPPORT THE CONTEXT
# =========================================================================

for source_name in SOURCE_NAMES_LIST:

    train_length = len(
        TRAIN_MEMMAPS[
            source_name
        ]
    )

    validation_length = len(
        VALIDATION_MEMMAPS[
            source_name
        ]
    )

    if train_length < (
        CONTEXT_LENGTH + 1
    ):

        raise RuntimeError(
            f"Training source '{source_name}' contains "
            f"{train_length:,} tokens, but at least "
            f"{CONTEXT_LENGTH + 1:,} are required."
        )

    if validation_length < (
        CONTEXT_LENGTH + 1
    ):

        raise RuntimeError(
            f"Validation source '{source_name}' contains "
            f"{validation_length:,} tokens, but at least "
            f"{CONTEXT_LENGTH + 1:,} are required."
        )


# =========================================================================
# 4. CHECKPOINTABLE SEQUENCE SAMPLER
# =========================================================================

class MultiSourceSequenceSampler:
    """
    Memory-mapped, deterministic, checkpointable language-model sampler.

    The sampler state includes the NumPy bit-generator state, so restoring
    the state reproduces the exact future source/position sampling stream.
    """

    SCHEMA_VERSION = 1

    def __init__(
        self,
        *,
        source_names,
        source_probabilities,
        split,
        context_length,
        seed,
    ):

        self.source_names = list(
            source_names
        )

        self.source_probabilities = np.asarray(
            source_probabilities,
            dtype=np.float64,
        ).copy()

        if len(
            self.source_names
        ) != len(
            self.source_probabilities
        ):

            raise ValueError(
                "source_names and source_probabilities "
                "must have the same length."
            )

        if not np.all(
            np.isfinite(
                self.source_probabilities
            )
        ):

            raise ValueError(
                "Source probabilities contain "
                "NaN or Inf."
            )

        if np.any(
            self.source_probabilities < 0
        ):

            raise ValueError(
                "Source probabilities cannot be negative."
            )

        probability_sum = float(
            self.source_probabilities.sum()
        )

        if probability_sum <= 0:

            raise ValueError(
                "Source probabilities must sum "
                "to a positive value."
            )

        self.source_probabilities /= (
            probability_sum
        )

        if split not in {
            "train",
            "validation",
        }:

            raise ValueError(
                "split must be 'train' or 'validation'."
            )

        self.split = str(
            split
        )

        self.context_length = int(
            context_length
        )

        if self.context_length <= 0:

            raise ValueError(
                "context_length must be positive."
            )

        self.seed = int(
            seed
        )

        self.rng = np.random.default_rng(
            self.seed
        )

        # Number of samples produced so far.
        self.samples_drawn = 0

        # Number of full batches produced through get_batch().
        self.batches_drawn = 0

    # ---------------------------------------------------------------------
    # Return the correct memory-map dictionary.
    # ---------------------------------------------------------------------

    def _memmaps(
        self,
    ):

        if self.split == "train":

            return TRAIN_MEMMAPS

        return VALIDATION_MEMMAPS

    # ---------------------------------------------------------------------
    # Number of valid starting positions for a source.
    # ---------------------------------------------------------------------

    def _valid_start_count(
        self,
        source_name,
    ):

        token_count = len(
            self._memmaps()[
                source_name
            ]
        )

        return max(
            0,
            token_count
            - self.context_length,
        )

    # ---------------------------------------------------------------------
    # Select one source + one random sequence position.
    # ---------------------------------------------------------------------

    def sample_one(
        self,
    ):

        source_index = int(
            self.rng.choice(
                len(
                    self.source_names
                ),
                p=self.source_probabilities,
            )
        )

        source_name = (
            self.source_names[
                source_index
            ]
        )

        valid_start_count = (
            self._valid_start_count(
                source_name
            )
        )

        if valid_start_count <= 0:

            raise RuntimeError(
                f"{self.split}/{source_name} does not "
                f"contain enough tokens for context length "
                f"{self.context_length}."
            )

        start = int(
            self.rng.integers(
                low=0,
                high=valid_start_count,
            )
        )

        self.samples_drawn += 1

        return (
            source_name,
            start,
        )

    # ---------------------------------------------------------------------
    # Produce one batch.
    # ---------------------------------------------------------------------

    def get_batch(
        self,
        batch_size,
    ):

        batch_size = int(
            batch_size
        )

        if batch_size <= 0:

            raise ValueError(
                "batch_size must be positive."
            )

        memmaps = self._memmaps()

        inputs = np.empty(
            (
                batch_size,
                self.context_length,
            ),
            dtype=np.int64,
        )

        targets = np.empty(
            (
                batch_size,
                self.context_length,
            ),
            dtype=np.int64,
        )

        selected_sources = []

        selected_positions = []

        for row in range(
            batch_size
        ):

            source_name, start = (
                self.sample_one()
            )

            values = np.asarray(
                memmaps[
                    source_name
                ][
                    start
                    :
                    start
                    + SAMPLING_WINDOW_LENGTH
                ],
                dtype=np.int64,
            )

            if len(values) != (
                SAMPLING_WINDOW_LENGTH
            ):

                raise RuntimeError(
                    f"Sampler retrieved {len(values)} tokens "
                    f"instead of {SAMPLING_WINDOW_LENGTH}."
                )

            # -------------------------------------------------------------
            # Autoregressive shift:
            #
            # input  = x[t ... t+T-1]
            # target = x[t+1 ... t+T]
            # -------------------------------------------------------------

            inputs[row] = (
                values[:-1]
            )

            targets[row] = (
                values[1:]
            )

            selected_sources.append(
                source_name
            )

            selected_positions.append(
                int(start)
            )

        self.batches_drawn += 1

        input_tensor = torch.from_numpy(
            inputs
        )

        target_tensor = torch.from_numpy(
            targets
        )

        return (
            input_tensor,
            target_tensor,
            selected_sources,
            selected_positions,
        )

    # ---------------------------------------------------------------------
    # Serialize sampler state.
    # ---------------------------------------------------------------------

    def state_dict(
        self,
    ):

        return {

            "schema_version": (
                self.SCHEMA_VERSION
            ),

            "source_names": list(
                self.source_names
            ),

            "source_probabilities": (
                self.source_probabilities.tolist()
            ),

            "split": (
                self.split
            ),

            "context_length": (
                self.context_length
            ),

            "seed": (
                self.seed
            ),

            "samples_drawn": (
                self.samples_drawn
            ),

            "batches_drawn": (
                self.batches_drawn
            ),

            "rng_state": copy.deepcopy(
                self.rng.bit_generator.state
            ),
        }

    # ---------------------------------------------------------------------
    # Restore sampler state.
    # ---------------------------------------------------------------------

    def load_state_dict(
        self,
        state,
    ):

        if int(
            state["schema_version"]
        ) != self.SCHEMA_VERSION:

            raise ValueError(
                "Unsupported sampler checkpoint schema."
            )

        checkpoint_sources = list(
            state[
                "source_names"
            ]
        )

        if checkpoint_sources != (
            self.source_names
        ):

            raise ValueError(
                "Checkpoint source ordering does not "
                "match the current sampler."
            )

        checkpoint_split = str(
            state[
                "split"
            ]
        )

        if checkpoint_split != (
            self.split
        ):

            raise ValueError(
                "Checkpoint split does not match "
                "the current sampler."
            )

        checkpoint_context_length = int(
            state[
                "context_length"
            ]
        )

        if checkpoint_context_length != (
            self.context_length
        ):

            raise ValueError(
                "Checkpoint context length does not "
                "match the current sampler."
            )

        checkpoint_probabilities = np.asarray(
            state[
                "source_probabilities"
            ],
            dtype=np.float64,
        )

        if not np.all(
            np.isfinite(
                checkpoint_probabilities
            )
        ):

            raise ValueError(
                "Checkpoint source probabilities "
                "contain NaN or Inf."
            )

        if np.any(
            checkpoint_probabilities < 0
        ):

            raise ValueError(
                "Checkpoint source probabilities "
                "contain negative values."
            )

        probability_sum = float(
            checkpoint_probabilities.sum()
        )

        if probability_sum <= 0:

            raise ValueError(
                "Checkpoint source probabilities "
                "sum to zero."
            )

        checkpoint_probabilities /= (
            probability_sum
        )

        self.source_probabilities = (
            checkpoint_probabilities
        )

        self.seed = int(
            state[
                "seed"
            ]
        )

        self.samples_drawn = int(
            state[
                "samples_drawn"
            ]
        )

        self.batches_drawn = int(
            state[
                "batches_drawn"
            ]
        )

        self.rng = np.random.default_rng(
            self.seed
        )

        self.rng.bit_generator.state = (
            copy.deepcopy(
                state[
                    "rng_state"
                ]
            )
        )


# =========================================================================
# 5. CREATE TRAIN + VALIDATION SAMPLERS
# =========================================================================

TRAIN_SAMPLER = (
    MultiSourceSequenceSampler(
        source_names=SOURCE_NAMES_LIST,
        source_probabilities=SOURCE_PROBABILITIES,
        split="train",
        context_length=CONTEXT_LENGTH,
        seed=SEED,
    )
)

VALIDATION_SAMPLER = (
    MultiSourceSequenceSampler(
        source_names=SOURCE_NAMES_LIST,
        source_probabilities=SOURCE_PROBABILITIES,
        split="validation",
        context_length=CONTEXT_LENGTH,
        seed=SEED + 1,
    )
)


# =========================================================================
# 6. SAMPLER CONFIGURATION
# =========================================================================

SAMPLER_CONFIG = {

    "schema_version": 1,

    "source_names": list(
        SOURCE_NAMES_LIST
    ),

    "source_probabilities": (
        SOURCE_PROBABILITIES.tolist()
    ),

    "context_length": (
        CONTEXT_LENGTH
    ),

    "training_batch_size": (
        MICRO_BATCH_SIZE
    ),

    "gradient_accumulation_steps": (
        GRADIENT_ACCUMULATION_STEPS
    ),

    "train_token_budget": (
        TRAIN_TOKEN_BUDGET
    ),

    "validation_token_budget": (
        VALIDATION_TOKEN_BUDGET
    ),

    "validation_batches": (
        VALIDATION_BATCHES
    ),

    "train_seed": (
        SEED
    ),

    "validation_seed": (
        SEED + 1
    ),

    "dataset_fingerprint": (
        DATA_FINGERPRINT
    ),
}


SAMPLER_CONFIG_PATH = (
    CONFIGS_DIR
    / "sampler_config_D.json"
)


with SAMPLER_CONFIG_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        SAMPLER_CONFIG,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 7. SAMPLER SANITY TEST
# =========================================================================

print("=" * 70)
print("ARCHITECTURE-D SAMPLER TEST")
print("=" * 70)

print(
    f"Split                : train"
)

print(
    f"Context length       : "
    f"{CONTEXT_LENGTH}"
)

print(
    f"Micro batch size     : "
    f"{MICRO_BATCH_SIZE}"
)

print(
    f"Gradient accumulation: "
    f"{GRADIENT_ACCUMULATION_STEPS}"
)


test_inputs, test_targets, test_sources, test_positions = (
    TRAIN_SAMPLER.get_batch(
        MICRO_BATCH_SIZE
    )
)


# -------------------------------------------------------------------------
# Shape checks
# -------------------------------------------------------------------------

EXPECTED_BATCH_SHAPE = (
    MICRO_BATCH_SIZE,
    CONTEXT_LENGTH,
)

assert test_inputs.shape == (
    EXPECTED_BATCH_SHAPE
)

assert test_targets.shape == (
    EXPECTED_BATCH_SHAPE
)


# -------------------------------------------------------------------------
# dtype checks
# -------------------------------------------------------------------------

assert test_inputs.dtype == torch.int64

assert test_targets.dtype == torch.int64


# -------------------------------------------------------------------------
# Vocabulary-range checks
# -------------------------------------------------------------------------

assert bool(
    torch.all(
        test_inputs >= 0
    )
)

assert bool(
    torch.all(
        test_inputs < VOCAB_SIZE
    )
)

assert bool(
    torch.all(
        test_targets >= 0
    )
)

assert bool(
    torch.all(
        test_targets < VOCAB_SIZE
    )
)


# -------------------------------------------------------------------------
# Autoregressive shift check
# -------------------------------------------------------------------------
#
# For each sampled example:
#
#     targets correspond exactly to the token immediately after each
#     input token in the source shard.
#
# We verify this directly against the memory map.

for row in range(
    MICRO_BATCH_SIZE
):

    source_name = (
        test_sources[row]
    )

    start = (
        test_positions[row]
    )

    raw_values = np.asarray(
        TRAIN_MEMMAPS[
            source_name
        ][
            start
            :
            start
            + CONTEXT_LENGTH
            + 1
        ],
        dtype=np.int64,
    )

    assert np.array_equal(
        test_inputs[
            row
        ].cpu().numpy(),
        raw_values[:-1],
    )

    assert np.array_equal(
        test_targets[
            row
        ].cpu().numpy(),
        raw_values[1:],
    )


# =========================================================================
# 8. CHECKPOINT/RESTORE SANITY TEST
# =========================================================================
#
# We test that saving and restoring the sampler state reproduces the exact
# next sample.

SAMPLER_STATE = (
    TRAIN_SAMPLER.state_dict()
)

next_sample_a = (
    TRAIN_SAMPLER.sample_one()
)

RESTORE_TEST_SAMPLER = (
    MultiSourceSequenceSampler(
        source_names=SOURCE_NAMES_LIST,
        source_probabilities=SOURCE_PROBABILITIES,
        split="train",
        context_length=CONTEXT_LENGTH,
        seed=SEED,
    )
)

RESTORE_TEST_SAMPLER.load_state_dict(
    SAMPLER_STATE
)

next_sample_b = (
    RESTORE_TEST_SAMPLER.sample_one()
)

assert next_sample_a == (
    next_sample_b
), (
    "Sampler restore test failed: "
    "restored RNG did not reproduce the next sample."
)


# =========================================================================
# 9. SOURCE COVERAGE TEST
# =========================================================================
#
# Draw a modest number of samples to verify the multi-source mechanism.
# This is not training.

SOURCE_COVERAGE_SAMPLES = 200

source_counts = {
    source_name: 0
    for source_name in SOURCE_NAMES_LIST
}

for _ in range(
    SOURCE_COVERAGE_SAMPLES
):

    sampled_source, _ = (
        TRAIN_SAMPLER.sample_one()
    )

    source_counts[
        sampled_source
    ] += 1


assert sum(
    source_counts.values()
) == SOURCE_COVERAGE_SAMPLES


# =========================================================================
# 10. SAVE INITIAL SAMPLER STATES
# =========================================================================
#
# These are diagnostic artifacts only. The authoritative training sampler
# state will be stored inside the training checkpoint.

INITIAL_SAMPLER_STATE = {

    "train": (
        TRAIN_SAMPLER.state_dict()
    ),

    "validation": (
        VALIDATION_SAMPLER.state_dict()
    ),
}


INITIAL_SAMPLER_STATE_PATH = (
    CONFIGS_DIR
    / "initial_sampler_state_D.json"
)


with INITIAL_SAMPLER_STATE_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        INITIAL_SAMPLER_STATE,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 11. FINAL REPORT
# =========================================================================

print("\n" + "-" * 70)

print(
    "SAMPLER RESULTS"
)

print(
    "-" * 70
)

print(
    f"Train samples drawn : "
    f"{TRAIN_SAMPLER.samples_drawn:,}"
)

print(
    f"Train batches drawn : "
    f"{TRAIN_SAMPLER.batches_drawn:,}"
)

print(
    f"Source coverage     : "
    f"{source_counts}"
)

print(
    "\n✅ Input shape       : "
    f"{tuple(test_inputs.shape)}"
)

print(
    "✅ Target shape      : "
    f"{tuple(test_targets.shape)}"
)

print(
    "✅ Input dtype       : "
    f"{test_inputs.dtype}"
)

print(
    "✅ Target dtype      : "
    f"{test_targets.dtype}"
)

print(
    "✅ Autoregressive shift verified."
)

print(
    "✅ Vocabulary bounds verified."
)

print(
    "✅ Sampler checkpoint/restore verified."
)

print(
    f"Sampler config      : "
    f"{SAMPLER_CONFIG_PATH}"
)

print(
    f"Initial sampler state: "
    f"{INITIAL_SAMPLER_STATE_PATH}"
)

print(
    "\n" + "=" * 70
)

print(
    "✅ CELL 04 COMPLETE"
)

print(
    "=" * 70
)

print(
    "Memory-mapped training sampler : READY"
)

print(
    "Memory-mapped validation sampler: READY"
)

print(
    "Deterministic multi-source sampling: READY"
)

print(
    "Checkpointable RNG state       : READY"
)

print(
    "Automatic sampler restoration  : READY"
)

print(
    "Fast/production implementation : IDENTICAL"
)

print(
    "=" * 70
)

ARCHITECTURE-D SAMPLER TEST
Split                : train
Context length       : 2048
Micro batch size     : 1
Gradient accumulation: 8

----------------------------------------------------------------------
SAMPLER RESULTS
----------------------------------------------------------------------
Train samples drawn : 202
Train batches drawn : 1
Source coverage     : {'acts': 50, 'high_courts': 52, 'supreme_court': 50, 'tribunals': 48}

✅ Input shape       : (1, 2048)
✅ Target shape      : (1, 2048)
✅ Input dtype       : torch.int64
✅ Target dtype      : torch.int64
✅ Autoregressive shift verified.
✅ Vocabulary bounds verified.
✅ Sampler checkpoint/restore verified.
Sampler config      : /kaggle/working/architecture_D/configs/sampler_config_D.json
Initial sampler state: /kaggle/working/architecture_D/configs/initial_sampler_state_D.json

✅ CELL 04 COMPLETE
Memory-mapped training sampler : READY
Memory-mapped validation sampler: READY
Deterministic multi-source sampling: READY
Checkpointabl

In [5]:
# Cell 05 — Freeze the complete Architecture-D TCN configuration
# DECISION: ADD
#
# Purpose:
# 1. Define the complete TCN architecture in one place.
# 2. Keep architecture configuration independent from the training loop.
# 3. Use the same architecture implementation in fast and production.
# 4. Compute and validate the nominal receptive field.
# 5. Compute an architecture-level parameter estimate.
# 6. Persist the complete configuration for checkpoint compatibility.
#
# Architecture:
#   Vocabulary embedding
#       ↓
#   10 residual TCN blocks
#       ↓
#   2 causal dilated convolutions per block
#       ↓
#   dilations = 1,2,4,...,512
#       ↓
#   tied vocabulary projection
#
# Baseline configuration:
#   vocabulary          = 50,000
#   embedding           = 1,024
#   channels            = 1,024
#   residual blocks     = 10
#   convolutions/block  = 2
#   kernel size         = 3
#   dilations           = powers of two
#   residual connections= enabled
#   weight normalization= enabled
#   dropout             = 0.10
#   tied output         = enabled
#   model context       = 2,048
#
# IMPORTANT:
# - These are baseline architecture choices, not results of tuning.
# - The actual instantiated parameter count is verified in Cell 06.

import copy
import hashlib
import json
from pathlib import Path


print("=" * 70)
print("ARCHITECTURE-D TCN CONFIGURATION")
print("=" * 70)


# =========================================================================
# 1. COMPLETE MODEL CONFIGURATION
# =========================================================================

TCN_ARCHITECTURE = {

    # ---------------------------------------------------------------------
    # Identity
    # ---------------------------------------------------------------------

    "architecture": (
        "causal_dilated_residual_tcn_language_model"
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    # ---------------------------------------------------------------------
    # Token representation
    # ---------------------------------------------------------------------

    "vocab_size": int(
        VOCAB_SIZE
    ),

    "embedding_dim": 1024,

    # ---------------------------------------------------------------------
    # TCN channels
    # ---------------------------------------------------------------------

    "channels": 1024,

    # ---------------------------------------------------------------------
    # Residual-stack structure
    # ---------------------------------------------------------------------

    "num_blocks": 10,

    "convs_per_block": 2,

    "kernel_size": 3,

    "dilations": [
        1,
        2,
        4,
        8,
        16,
        32,
        64,
        128,
        256,
        512,
    ],

    # ---------------------------------------------------------------------
    # Block options
    # ---------------------------------------------------------------------

    "use_residual_connections": True,

    "use_weight_normalization": True,

    "dropout": 0.10,

    # ---------------------------------------------------------------------
    # Language-model output
    # ---------------------------------------------------------------------

    "tie_output_embedding": True,

    # ---------------------------------------------------------------------
    # Maximum sequence length supported by the architecture
    # ---------------------------------------------------------------------

    "context_length": 2048,
}


# =========================================================================
# 2. FUNDAMENTAL VALIDATION
# =========================================================================

assert (
    TCN_ARCHITECTURE["vocab_size"]
    == VOCAB_SIZE
), (
    "TCN vocabulary size does not match the locked tokenizer."
)

assert (
    TCN_ARCHITECTURE["embedding_dim"]
    > 0
)

assert (
    TCN_ARCHITECTURE["channels"]
    > 0
)

assert (
    TCN_ARCHITECTURE["num_blocks"]
    > 0
)

assert (
    TCN_ARCHITECTURE["convs_per_block"]
    == 2
), (
    "Architecture-D is configured for two convolutions per residual block."
)

assert (
    TCN_ARCHITECTURE["kernel_size"]
    >= 2
)

assert (
    TCN_ARCHITECTURE["context_length"]
    > 0
)

assert (
    0.0
    <= TCN_ARCHITECTURE["dropout"]
    < 1.0
)

assert isinstance(
    TCN_ARCHITECTURE[
        "use_residual_connections"
    ],
    bool,
)

assert isinstance(
    TCN_ARCHITECTURE[
        "use_weight_normalization"
    ],
    bool,
)

assert isinstance(
    TCN_ARCHITECTURE[
        "tie_output_embedding"
    ],
    bool,
)


# =========================================================================
# 3. DILATION VALIDATION
# =========================================================================

DILATIONS = [
    int(value)
    for value in TCN_ARCHITECTURE[
        "dilations"
    ]
]

assert len(
    DILATIONS
) == TCN_ARCHITECTURE[
    "num_blocks"
], (
    "The number of dilation values must equal the number of blocks."
)

assert all(
    dilation > 0
    for dilation in DILATIONS
), (
    "All dilations must be positive."
)

assert DILATIONS == sorted(
    DILATIONS
), (
    "Dilations must be monotonically increasing."
)


# =========================================================================
# 4. RECEPTIVE FIELD
# =========================================================================
#
# Each residual block has two causal convolutions.
#
# For one causal convolution:
#
#     receptive-field increment
#         = (kernel_size - 1) * dilation
#
# With two convolutions per block:
#
#     block increment
#         = 2 * (kernel_size - 1) * dilation
#
# Therefore:
#
#     RF = 1 + 2 * (kernel_size - 1) * sum(dilations)

KERNEL_SIZE = int(
    TCN_ARCHITECTURE[
        "kernel_size"
    ]
)

NOMINAL_RECEPTIVE_FIELD = (
    1
    + TCN_ARCHITECTURE[
        "convs_per_block"
    ]
    * (
        KERNEL_SIZE - 1
    )
    * sum(
        DILATIONS
    )
)


TCN_ARCHITECTURE[
    "nominal_receptive_field"
] = int(
    NOMINAL_RECEPTIVE_FIELD
)


assert (
    NOMINAL_RECEPTIVE_FIELD
    >= TCN_ARCHITECTURE[
        "context_length"
    ]
), (
    "The nominal receptive field does not cover the configured "
    "model context."
)


# =========================================================================
# 5. ARCHITECTURE-LEVEL PARAMETER ESTIMATE
# =========================================================================
#
# Input embedding:
#
#     V × E
#
# Each convolution:
#
#     C × C × K + C
#
# There are:
#
#     blocks × convolutions_per_block
#
# convolutions.
#
# Weight normalization adds one additional scalar per output channel
# for every normalized convolution.
#
# The output vocabulary projection is tied to the input embedding, so
# it contributes NO additional vocabulary matrix.

VOCAB_SIZE_ARCH = int(
    TCN_ARCHITECTURE[
        "vocab_size"
    ]
)

EMBEDDING_DIM_ARCH = int(
    TCN_ARCHITECTURE[
        "embedding_dim"
    ]
)

CHANNELS_ARCH = int(
    TCN_ARCHITECTURE[
        "channels"
    ]
)

NUM_BLOCKS_ARCH = int(
    TCN_ARCHITECTURE[
        "num_blocks"
    ]
)

CONVS_PER_BLOCK_ARCH = int(
    TCN_ARCHITECTURE[
        "convs_per_block"
    ]
)

DROPOUT_ARCH = float(
    TCN_ARCHITECTURE[
        "dropout"
    ])


EMBEDDING_PARAMETER_COUNT = (
    VOCAB_SIZE_ARCH
    * EMBEDDING_DIM_ARCH
)


SINGLE_CONV_PARAMETER_COUNT = (
    CHANNELS_ARCH
    * CHANNELS_ARCH
    * KERNEL_SIZE
    + CHANNELS_ARCH
)


TOTAL_CONV_COUNT = (
    NUM_BLOCKS_ARCH
    * CONVS_PER_BLOCK_ARCH
)


CONV_PARAMETER_COUNT = (
    TOTAL_CONV_COUNT
    * SINGLE_CONV_PARAMETER_COUNT
)


WEIGHT_NORM_PARAMETER_COUNT = (
    TOTAL_CONV_COUNT
    * CHANNELS_ARCH
    if TCN_ARCHITECTURE[
        "use_weight_normalization"
    ]
    else 0
)


if (
    TCN_ARCHITECTURE[
        "tie_output_embedding"
    ]
):

    OUTPUT_PROJECTION_PARAMETER_COUNT = 0

else:

    OUTPUT_PROJECTION_PARAMETER_COUNT = (
        CHANNELS_ARCH
        * VOCAB_SIZE_ARCH
    )


ESTIMATED_PARAMETER_COUNT = (
    EMBEDDING_PARAMETER_COUNT
    + CONV_PARAMETER_COUNT
    + WEIGHT_NORM_PARAMETER_COUNT
    + OUTPUT_PROJECTION_PARAMETER_COUNT
)


TCN_ARCHITECTURE[
    "estimated_parameter_count"
] = int(
    ESTIMATED_PARAMETER_COUNT
)


# =========================================================================
# 6. PHASE-1 MODEL SIZE VALIDATION
# =========================================================================

PHASE1_MIN_PARAMETERS = 50_000_000

PHASE1_MAX_PARAMETERS = 300_000_000


assert (
    PHASE1_MIN_PARAMETERS
    <= ESTIMATED_PARAMETER_COUNT
    <= PHASE1_MAX_PARAMETERS
), (
    "The estimated model size is outside the Phase-1 "
    "50M–300M parameter requirement."
)


# =========================================================================
# 7. CONFIGURATION FINGERPRINT
# =========================================================================
#
# The fingerprint will later be stored in every training checkpoint.
# It prevents accidentally loading a checkpoint belonging to a different
# Architecture-D configuration.

CONFIGURATION_FOR_HASH = copy.deepcopy(
    TCN_ARCHITECTURE
)

CONFIGURATION_JSON = json.dumps(
    CONFIGURATION_FOR_HASH,
    sort_keys=True,
    separators=(
        ",",
        ":",
    ),
)

TCN_CONFIGURATION_FINGERPRINT = (
    hashlib.sha256(
        CONFIGURATION_JSON.encode(
            "utf-8"
        )
    ).hexdigest()
)


TCN_ARCHITECTURE[
    "configuration_fingerprint"
] = (
    TCN_CONFIGURATION_FINGERPRINT
)


# =========================================================================
# 8. SAVE CONFIGURATION
# =========================================================================

TCN_SPEC_PATH = (
    CONFIGS_DIR
    / "tcn_architecture_spec_D.json"
)

with TCN_SPEC_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        TCN_ARCHITECTURE,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 9. UPDATE RUNTIME CONFIGURATION
# =========================================================================

RUNTIME_CONFIGURATION[
    "tcn_architecture"
] = copy.deepcopy(
    TCN_ARCHITECTURE
)

RUNTIME_CONFIGURATION[
    "data_fingerprint"
] = (
    DATA_FINGERPRINT
)

RUNTIME_CONFIGURATION[
    "tokenizer_vocab_size"
] = int(
    VOCAB_SIZE
)


with RUNTIME_CONFIGURATION_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        RUNTIME_CONFIGURATION,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 10. FINAL REPORT
# =========================================================================

print(
    f"Architecture        : "
    f"{TCN_ARCHITECTURE['architecture']}"
)

print(
    f"Vocabulary          : "
    f"{VOCAB_SIZE_ARCH:,}"
)

print(
    f"Embedding dimension : "
    f"{EMBEDDING_DIM_ARCH}"
)

print(
    f"Channels            : "
    f"{CHANNELS_ARCH}"
)

print(
    f"Residual blocks     : "
    f"{NUM_BLOCKS_ARCH}"
)

print(
    f"Convolutions/block  : "
    f"{CONVS_PER_BLOCK_ARCH}"
)

print(
    f"Kernel size         : "
    f"{KERNEL_SIZE}"
)

print(
    f"Dilations           : "
    f"{DILATIONS}"
)

print(
    f"Residual connections: "
    f"{TCN_ARCHITECTURE['use_residual_connections']}"
)

print(
    f"Weight normalization: "
    f"{TCN_ARCHITECTURE['use_weight_normalization']}"
)

print(
    f"Dropout             : "
    f"{DROPOUT_ARCH}"
)

print(
    f"Tied output         : "
    f"{TCN_ARCHITECTURE['tie_output_embedding']}"
)

print(
    f"Context length      : "
    f"{TCN_ARCHITECTURE['context_length']}"
)

print(
    f"Nominal receptive field: "
    f"{NOMINAL_RECEPTIVE_FIELD:,}"
)

print(
    f"Embedding params    : "
    f"{EMBEDDING_PARAMETER_COUNT:,}"
)

print(
    f"Convolution params  : "
    f"{CONV_PARAMETER_COUNT:,}"
)

print(
    f"Weight-norm params  : "
    f"{WEIGHT_NORM_PARAMETER_COUNT:,}"
)

print(
    f"Output projection   : "
    f"{OUTPUT_PROJECTION_PARAMETER_COUNT:,}"
)

print(
    f"Estimated parameters: "
    f"{ESTIMATED_PARAMETER_COUNT:,} "
    f"({ESTIMATED_PARAMETER_COUNT / 1e6:.3f}M)"
)

print(
    f"Configuration hash  : "
    f"{TCN_CONFIGURATION_FINGERPRINT[:20]}..."
)

print(
    f"Specification file  : "
    f"{TCN_SPEC_PATH}"
)

print(
    "\n✅ Receptive field covers the full model context."
)

print(
    "✅ Estimated model size is inside 50M–300M."
)

print(
    "✅ TCN architecture configuration frozen."
)

print(
    "\n" + "=" * 70
)

print(
    "✅ CELL 05 COMPLETE"
)

print(
    "=" * 70
)

ARCHITECTURE-D TCN CONFIGURATION
Architecture        : causal_dilated_residual_tcn_language_model
Vocabulary          : 50,000
Embedding dimension : 1024
Channels            : 1024
Residual blocks     : 10
Convolutions/block  : 2
Kernel size         : 3
Dilations           : [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
Residual connections: True
Weight normalization: True
Dropout             : 0.1
Tied output         : True
Context length      : 2048
Nominal receptive field: 4,093
Embedding params    : 51,200,000
Convolution params  : 62,935,040
Weight-norm params  : 20,480
Output projection   : 0
Estimated parameters: 114,155,520 (114.156M)
Configuration hash  : 584c1d762e6fc5514a0c...
Specification file  : /kaggle/working/architecture_D/configs/tcn_architecture_spec_D.json

✅ Receptive field covers the full model context.
✅ Estimated model size is inside 50M–300M.
✅ TCN architecture configuration frozen.

✅ CELL 05 COMPLETE


In [6]:
# Cell 06 — Complete Architecture-D TCN model implementation
# DECISION: REPLACE
#
# Purpose:
# 1. Implement the complete Architecture-D TCN.
# 2. Define all supporting causal-convolution and residual-block classes.
# 3. Instantiate MODEL for use by all later cells.
# 4. Use the frozen configuration from Cell 05.
# 5. Keep the exact same implementation for fast and production.
# 6. Do not perform training in this cell.
#
# Architecture:
#
#   Token IDs
#       ↓
#   Token Embedding
#       ↓
#   Causal Dilated Residual Block × 10
#       ↓
#   Weight-tied vocabulary projection
#       ↓
#   Logits [B, T, V]
#
# Configuration:
#   Vocab       = 50,000
#   Embedding   = 1,024
#   Channels    = 1,024
#   Blocks      = 10
#   Conv/block  = 2
#   Kernel      = 3
#   Dilations   = 1,2,4,8,16,32,64,128,256,512
#   Dropout     = 0.10
#   Residuals   = enabled
#   Weight norm = enabled
#   Tied output = enabled
#   Context     = 2,048

import gc
import json
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm


# =========================================================================
# 1. PRECONDITION CHECKS
# =========================================================================

assert "TCN_ARCHITECTURE" in globals(), (
    "TCN_ARCHITECTURE is unavailable. "
    "Run Cell 05 first."
)

assert "VOCAB_SIZE" in globals(), (
    "VOCAB_SIZE is unavailable. "
    "Run Cell 03 first."
)

assert "DEVICE" in globals(), (
    "DEVICE is unavailable. "
    "Run Cell 02 first."
)

assert (
    int(TCN_ARCHITECTURE["vocab_size"])
    == int(VOCAB_SIZE)
), (
    "TCN vocabulary size does not match tokenizer vocabulary."
)


print("=" * 70)
print("ARCHITECTURE-D TCN IMPLEMENTATION")
print("=" * 70)


# =========================================================================
# 2. CAUSAL CONVOLUTION
# =========================================================================
#
# Conv1d normally pads symmetrically when padding > 0.
# Symmetric padding would expose future positions.
#
# Instead:
#
#     left_padding = (kernel_size - 1) * dilation
#
# followed by Conv1d with padding=0.
#
# Thus output[t] cannot use x[t+1], x[t+2], ...
#
# Sequence length is preserved.

class CausalConv1d(
    nn.Module
):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        dilation=1,
        bias=True,
        use_weight_normalization=True,
    ):

        super().__init__()

        self.in_channels = int(
            in_channels
        )

        self.out_channels = int(
            out_channels
        )

        self.kernel_size = int(
            kernel_size
        )

        self.dilation = int(
            dilation
        )

        if (
            self.in_channels <= 0
            or self.out_channels <= 0
        ):

            raise ValueError(
                "Convolution channel counts must be positive."
            )

        if self.kernel_size <= 0:

            raise ValueError(
                "Kernel size must be positive."
            )

        if self.dilation <= 0:

            raise ValueError(
                "Dilation must be positive."
            )

        self.left_padding = (
            self.kernel_size - 1
        ) * self.dilation

        convolution = nn.Conv1d(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            kernel_size=self.kernel_size,
            dilation=self.dilation,
            padding=0,
            bias=bias,
        )

        if use_weight_normalization:

            convolution = weight_norm(
                convolution
            )

        self.conv = convolution

    def forward(
        self,
        x,
    ):

        if x.ndim != 3:

            raise ValueError(
                "CausalConv1d expects "
                "[B, C, T]."
            )

        if (
            x.shape[1]
            != self.in_channels
        ):

            raise ValueError(
                f"Expected {self.in_channels} channels, "
                f"received {x.shape[1]}."
            )

        if self.left_padding > 0:

            x = F.pad(
                x,
                (
                    self.left_padding,
                    0,
                ),
            )

        output = self.conv(
            x
        )

        if output.shape[-1] != x.shape[-1] - self.left_padding:

            raise RuntimeError(
                "Causal convolution unexpectedly changed "
                "the temporal dimension."
            )

        return output


# =========================================================================
# 3. TCN RESIDUAL BLOCK
# =========================================================================
#
# Each block has:
#
#     causal dilated conv
#         ↓
#     ReLU
#         ↓
#     Dropout
#         ↓
#     causal dilated conv
#         ↓
#     ReLU
#         ↓
#     Dropout
#         ↓
#     residual addition
#         ↓
#     ReLU
#
# Input/output channels are equal in this Architecture-D configuration.

class TCNResidualBlock(
    nn.Module
):

    def __init__(
        self,
        channels,
        kernel_size,
        dilation,
        dropout,
        use_residual_connections=True,
        use_weight_normalization=True,
    ):

        super().__init__()

        self.channels = int(
            channels
        )

        self.kernel_size = int(
            kernel_size
        )

        self.dilation = int(
            dilation
        )

        self.dropout_probability = float(
            dropout
        )

        self.use_residual_connections = bool(
            use_residual_connections
        )

        if self.dropout_probability < 0.0:

            raise ValueError(
                "Dropout cannot be negative."
            )

        if self.dropout_probability >= 1.0:

            raise ValueError(
                "Dropout must be less than 1."
            )

        # First causal dilated convolution.
        self.conv1 = CausalConv1d(
            in_channels=self.channels,
            out_channels=self.channels,
            kernel_size=self.kernel_size,
            dilation=self.dilation,
            bias=True,
            use_weight_normalization=(
                use_weight_normalization
            ),
        )

        self.activation1 = nn.ReLU()

        self.dropout1 = nn.Dropout(
            p=self.dropout_probability
        )

        # Second causal dilated convolution.
        self.conv2 = CausalConv1d(
            in_channels=self.channels,
            out_channels=self.channels,
            kernel_size=self.kernel_size,
            dilation=self.dilation,
            bias=True,
            use_weight_normalization=(
                use_weight_normalization
            ),
        )

        self.activation2 = nn.ReLU()

        self.dropout2 = nn.Dropout(
            p=self.dropout_probability
        )

        # Residual projection.
        #
        # Current Architecture-D has equal input/output channels,
        # so this is simply Identity.

        if self.use_residual_connections:

            self.residual_projection = (
                nn.Identity()
            )

        else:

            self.residual_projection = None

    def forward(
        self,
        x,
    ):

        if x.ndim != 3:

            raise ValueError(
                "TCNResidualBlock expects "
                "[B, C, T]."
            )

        if x.shape[1] != self.channels:

            raise ValueError(
                f"Expected {self.channels} channels, "
                f"received {x.shape[1]}."
            )

        residual = x

        output = self.conv1(
            x
        )

        output = self.activation1(
            output
        )

        output = self.dropout1(
            output
        )

        output = self.conv2(
            output
        )

        output = self.activation2(
            output
        )

        output = self.dropout2(
            output
        )

        if self.use_residual_connections:

            residual = self.residual_projection(
                residual
            )

            output = output + residual

            output = F.relu(
                output
            )

        return output


# =========================================================================
# 4. COMPLETE TCN LANGUAGE MODEL
# =========================================================================

class TCNLanguageModel(
    nn.Module
):

    def __init__(
        self,
        config,
    ):

        super().__init__()

        self.config = dict(
            config
        )

        # -------------------------------------------------------------
        # Core configuration
        # -------------------------------------------------------------

        self.vocab_size = int(
            config["vocab_size"]
        )

        self.embedding_dim = int(
            config["embedding_dim"]
        )

        self.channels = int(
            config["channels"]
        )

        self.num_blocks = int(
            config["num_blocks"]
        )

        self.convs_per_block = int(
            config["convs_per_block"]
        )

        self.kernel_size = int(
            config["kernel_size"]
        )

        self.dilations = [
            int(value)
            for value in config[
                "dilations"
            ]
        ]

        self.context_length = int(
            config["context_length"]
        )

        self.dropout = float(
            config["dropout"]
        )

        self.use_residual_connections = bool(
            config[
                "use_residual_connections"
            ]
        )

        self.use_weight_normalization = bool(
            config[
                "use_weight_normalization"
            ]
        )

        self.tie_output_embedding = bool(
            config[
                "tie_output_embedding"
            ]
        )

        # -------------------------------------------------------------
        # Validate architecture
        # -------------------------------------------------------------

        if self.num_blocks != len(
            self.dilations
        ):

            raise ValueError(
                "num_blocks must equal "
                "len(dilations)."
            )

        if self.convs_per_block != 2:

            raise ValueError(
                "Architecture-D requires "
                "two convolutions per block."
            )

        if self.embedding_dim != self.channels:

            raise ValueError(
                "embedding_dim must equal channels "
                "for the configured Architecture-D."
            )

        if self.kernel_size < 2:

            raise ValueError(
                "kernel_size must be at least 2."
            )

        if self.context_length <= 0:

            raise ValueError(
                "context_length must be positive."
            )

        if not (
            0.0
            <= self.dropout
            < 1.0
        ):

            raise ValueError(
                "dropout must be in [0, 1)."
            )

        if not all(
            dilation > 0
            for dilation in self.dilations
        ):

            raise ValueError(
                "All dilations must be positive."
            )

        # -------------------------------------------------------------
        # Token embedding
        # -------------------------------------------------------------

        self.token_embedding = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=self.embedding_dim,
        )

        # -------------------------------------------------------------
        # TCN residual stack
        # -------------------------------------------------------------

        self.tcn_blocks = nn.ModuleList(
            [
                TCNResidualBlock(
                    channels=self.channels,
                    kernel_size=self.kernel_size,
                    dilation=dilation,
                    dropout=self.dropout,
                    use_residual_connections=(
                        self.use_residual_connections
                    ),
                    use_weight_normalization=(
                        self.use_weight_normalization
                    ),
                )
                for dilation in self.dilations
            ]
        )

        # -------------------------------------------------------------
        # Output representation normalization
        # -------------------------------------------------------------
        #
        # The registered architecture does not add an extra normalization
        # layer beyond the configured weight normalization.

        self.output_norm = nn.Identity()

        # -------------------------------------------------------------
        # Vocabulary projection
        # -------------------------------------------------------------
        #
        # Tied output:
        #
        #     logits = hidden @ embedding_weight.T
        #
        # No second V × E matrix is created.

        if self.tie_output_embedding:

            self.output_projection = None

        else:

            self.output_projection = nn.Linear(
                self.embedding_dim,
                self.vocab_size,
                bias=False,
            )

        # -------------------------------------------------------------
        # Initialization
        # -------------------------------------------------------------

        self._initialize_parameters()

    # =========================================================================
    # INITIALIZATION
    # =========================================================================

    def _initialize_parameters(
        self,
    ):

        # -------------------------------------------------------------
        # Token embeddings
        # -------------------------------------------------------------

        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02,
        )

        # -------------------------------------------------------------
        # Convolution parameters
        # -------------------------------------------------------------

        for module in self.modules():

            if not isinstance(
                module,
                nn.Conv1d,
            ):

                continue

            # Weight-normalized Conv1d contains:
            #
            #     weight_v
            #     weight_g
            #
            # instead of a normal weight parameter.

            if hasattr(
                module,
                "weight_v",
            ):

                nn.init.kaiming_normal_(
                    module.weight_v,
                    mode="fan_in",
                    nonlinearity="relu",
                )

                if hasattr(
                    module,
                    "weight_g",
                ):

                    with torch.no_grad():

                        # Norm across all dimensions except output
                        # channel.

                        dimensions = tuple(
                            range(
                                1,
                                module.weight_v.ndim,
                            )
                        )

                        norm = (
                            module.weight_v
                            .norm(
                                dim=dimensions,
                                keepdim=True,
                            )
                        )

                        module.weight_g.copy_(
                            norm
                        )

            else:

                nn.init.kaiming_normal_(
                    module.weight,
                    mode="fan_in",
                    nonlinearity="relu",
                )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )

        # -------------------------------------------------------------
        # Untied output projection
        # -------------------------------------------------------------

        if (
            self.output_projection
            is not None
        ):

            nn.init.normal_(
                self.output_projection.weight,
                mean=0.0,
                std=0.02,
            )

    # =========================================================================
    # FORWARD
    # =========================================================================

    def forward(
        self,
        input_ids,
    ):

        # -------------------------------------------------------------
        # Tensor validation
        # -------------------------------------------------------------

        if not isinstance(
            input_ids,
            torch.Tensor,
        ):

            raise TypeError(
                "input_ids must be a torch.Tensor."
            )

        if input_ids.dtype != torch.long:

            raise TypeError(
                "input_ids must have dtype torch.long."
            )

        if input_ids.ndim != 2:

            raise ValueError(
                "input_ids must have shape [B, T]."
            )

        batch_size, sequence_length = (
            input_ids.shape
        )

        if batch_size <= 0:

            raise ValueError(
                "Batch size must be positive."
            )

        if sequence_length <= 0:

            raise ValueError(
                "Sequence length must be positive."
            )

        if sequence_length > self.context_length:

            raise ValueError(
                f"Sequence length {sequence_length} exceeds "
                f"context length {self.context_length}."
            )

        # -------------------------------------------------------------
        # Token ID range
        # -------------------------------------------------------------

        if torch.any(
            input_ids < 0
        ):

            raise ValueError(
                "input_ids contains negative token IDs."
            )

        if torch.any(
            input_ids >= self.vocab_size
        ):

            raise ValueError(
                "input_ids contains token IDs outside "
                "the vocabulary."
            )

        # -------------------------------------------------------------
        # Token embedding
        #
        # [B, T]
        #    ↓
        # [B, T, E]
        # -------------------------------------------------------------

        hidden = self.token_embedding(
            input_ids
        )

        # -------------------------------------------------------------
        # Conv1d representation
        #
        # [B, T, E]
        #    ↓ transpose
        # [B, E, T]
        # -------------------------------------------------------------

        hidden = hidden.transpose(
            1,
            2,
        ).contiguous()

        # -------------------------------------------------------------
        # TCN residual stack
        # -------------------------------------------------------------

        for block in self.tcn_blocks:

            hidden = block(
                hidden
            )

        # -------------------------------------------------------------
        # Output normalization
        # -------------------------------------------------------------

        hidden = self.output_norm(
            hidden
        )

        # -------------------------------------------------------------
        # Back to language-model layout
        #
        # [B, C, T]
        #    ↓
        # [B, T, C]
        # -------------------------------------------------------------

        hidden = hidden.transpose(
            1,
            2,
        ).contiguous()

        # -------------------------------------------------------------
        # Vocabulary projection
        # -------------------------------------------------------------

        if self.tie_output_embedding:

            logits = F.linear(
                hidden,
                self.token_embedding.weight,
            )

        else:

            logits = self.output_projection(
                hidden
            )

        # -------------------------------------------------------------
        # Output shape safety check
        # -------------------------------------------------------------

        expected_shape = (
            batch_size,
            sequence_length,
            self.vocab_size,
        )

        if tuple(
            logits.shape
        ) != expected_shape:

            raise RuntimeError(
                f"Unexpected logits shape "
                f"{tuple(logits.shape)}; "
                f"expected {expected_shape}."
            )

        return logits


# =========================================================================
# 5. INSTANTIATE THE MODEL
# =========================================================================
#
# Cell 06 is responsible for creating MODEL.
# Cell 07 will only validate it.

# Remove an old model reference if the cell is being rerun.

if "MODEL" in globals():

    try:
        del MODEL
    except Exception:
        pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


MODEL_CONFIG = dict(
    TCN_ARCHITECTURE
)

MODEL = TCNLanguageModel(
    MODEL_CONFIG
)

MODEL = MODEL.to(
    DEVICE
)


# =========================================================================
# 6. BASIC PARAMETER REPORT
# =========================================================================

TOTAL_PARAMETERS = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
)

TRAINABLE_PARAMETERS = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
    if parameter.requires_grad
)


assert (
    50_000_000
    <= TOTAL_PARAMETERS
    <= 300_000_000
), (
    f"Actual parameter count "
    f"{TOTAL_PARAMETERS:,} is outside "
    f"the Phase-1 50M–300M range."
)


# =========================================================================
# 7. MODEL STATUS
# =========================================================================

TCN_MODEL_IMPLEMENTED = True

TCN_IMPLEMENTATION_STATUS = {

    "status": "IMPLEMENTED",

    "model_class": (
        "TCNLanguageModel"
    ),

    "architecture": (
        TCN_ARCHITECTURE[
            "architecture"
        ]
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "total_parameters": int(
        TOTAL_PARAMETERS
    ),

    "trainable_parameters": int(
        TRAINABLE_PARAMETERS
    ),

    "context_length": int(
        TCN_ARCHITECTURE[
            "context_length"
        ]
    ),

    "vocab_size": int(
        TCN_ARCHITECTURE[
            "vocab_size"
        ]
    ),

    "dilations": list(
        TCN_ARCHITECTURE[
            "dilations"
        ]
    ),

    "weight_normalization": bool(
        TCN_ARCHITECTURE[
            "use_weight_normalization"
        ]
    ),

    "residual_connections": bool(
        TCN_ARCHITECTURE[
            "use_residual_connections"
        ]
    ),

    "tied_output_embedding": bool(
        TCN_ARCHITECTURE[
            "tie_output_embedding"
        ]
    ),

    "data_fingerprint": (
        DATA_FINGERPRINT
    ),
}


TCN_IMPLEMENTATION_STATUS_PATH = (
    CONFIGS_DIR
    / "tcn_implementation_status_D.json"
)


with TCN_IMPLEMENTATION_STATUS_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        TCN_IMPLEMENTATION_STATUS,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 8. UPDATE RUNTIME CONFIGURATION
# =========================================================================

RUNTIME_CONFIGURATION[
    "model"
] = {

    "implemented": True,

    "total_parameters": int(
        TOTAL_PARAMETERS
    ),

    "trainable_parameters": int(
        TRAINABLE_PARAMETERS
    ),

    "implementation_status_file": (
        str(
            TCN_IMPLEMENTATION_STATUS_PATH
        )
    ),
}


with RUNTIME_CONFIGURATION_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        RUNTIME_CONFIGURATION,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 9. FINAL REPORT
# =========================================================================

print(
    f"\nArchitecture       : "
    f"{TCN_ARCHITECTURE['architecture']}"
)

print(
    f"Vocabulary         : "
    f"{VOCAB_SIZE:,}"
)

print(
    f"Embedding dim      : "
    f"{TCN_ARCHITECTURE['embedding_dim']}"
)

print(
    f"Channels           : "
    f"{TCN_ARCHITECTURE['channels']}"
)

print(
    f"Residual blocks    : "
    f"{TCN_ARCHITECTURE['num_blocks']}"
)

print(
    f"Convs / block      : "
    f"{TCN_ARCHITECTURE['convs_per_block']}"
)

print(
    f"Kernel size        : "
    f"{TCN_ARCHITECTURE['kernel_size']}"
)

print(
    f"Dilations          : "
    f"{TCN_ARCHITECTURE['dilations']}"
)

print(
    f"Dropout            : "
    f"{TCN_ARCHITECTURE['dropout']}"
)

print(
    f"Residuals          : "
    f"{TCN_ARCHITECTURE['use_residual_connections']}"
)

print(
    f"Weight normalization: "
    f"{TCN_ARCHITECTURE['use_weight_normalization']}"
)

print(
    f"Tied output        : "
    f"{TCN_ARCHITECTURE['tie_output_embedding']}"
)

print(
    f"Context length     : "
    f"{TCN_ARCHITECTURE['context_length']}"
)

print(
    f"Nominal receptive field: "
    f"{TCN_ARCHITECTURE['nominal_receptive_field']:,}"
)

print(
    f"Actual parameters   : "
    f"{TOTAL_PARAMETERS:,} "
    f"({TOTAL_PARAMETERS / 1e6:.3f}M)"
)

print(
    f"Trainable parameters: "
    f"{TRAINABLE_PARAMETERS:,} "
    f"({TRAINABLE_PARAMETERS / 1e6:.3f}M)"
)

print(
    f"Status file         : "
    f"{TCN_IMPLEMENTATION_STATUS_PATH}"
)

print(
    "\n✅ Complete TCN implementation loaded."
)

print(
    "✅ MODEL instantiated and moved to DEVICE."
)

print(
    "✅ Actual parameter count is inside 50M–300M."
)

print(
    "✅ Cell 07 can now run without dependency-order errors."
)

print(
    "\n" + "=" * 70
)

print(
    "✅ CELL 06 COMPLETE"
)

print(
    "=" * 70
)

ARCHITECTURE-D TCN IMPLEMENTATION


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Architecture       : causal_dilated_residual_tcn_language_model
Vocabulary         : 50,000
Embedding dim      : 1024
Channels           : 1024
Residual blocks    : 10
Convs / block      : 2
Kernel size        : 3
Dilations          : [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
Dropout            : 0.1
Residuals          : True
Weight normalization: True
Tied output        : True
Context length     : 2048
Nominal receptive field: 4,093
Actual parameters   : 114,155,520 (114.156M)
Trainable parameters: 114,155,520 (114.156M)
Status file         : /kaggle/working/architecture_D/configs/tcn_implementation_status_D.json

✅ Complete TCN implementation loaded.
✅ MODEL instantiated and moved to DEVICE.
✅ Actual parameter count is inside 50M–300M.
✅ Cell 07 can now run without dependency-order errors.

✅ CELL 06 COMPLETE


In [7]:
# Cell 07 — Complete Architecture-D TCN validation and safety gate
# DECISION: ADD
#
# Purpose:
# 1. Validate the model created by Cell 06.
# 2. Verify the actual parameter count.
# 3. Verify architecture/configuration consistency.
# 4. Verify tied vocabulary projection.
# 5. Verify forward-pass shapes and numerical stability.
# 6. Verify next-token cross-entropy loss.
# 7. Verify backward propagation and gradient stability.
# 8. Directly verify causal behavior.
# 9. Verify the full 2048-token model context.
# 10. Save a persistent validation report.
#
# IMPORTANT:
# - No optimizer is created.
# - No training is started.
# - No checkpoint is created.
# - This is a hard safety gate before expensive training.
# - The same model is used for fast and production.

import gc
import hashlib
import json
import math
from pathlib import Path

import torch
import torch.nn as nn


# =========================================================================
# 1. PRECONDITIONS
# =========================================================================

assert "MODEL" in globals(), (
    "MODEL is unavailable. Run Cell 06 first."
)

assert isinstance(
    MODEL,
    nn.Module,
), (
    "MODEL is not a PyTorch nn.Module."
)

assert "MODEL_CONFIG" in globals(), (
    "MODEL_CONFIG is unavailable. Run Cell 05 and Cell 06 first."
)

assert "TCN_ARCHITECTURE" in globals(), (
    "TCN_ARCHITECTURE is unavailable. Run Cell 05 first."
)

assert "VOCAB_SIZE" in globals(), (
    "VOCAB_SIZE is unavailable. Run Cell 03 first."
)

assert "DEVICE" in globals(), (
    "DEVICE is unavailable. Run Cell 02 first."
)

assert "DATA_FINGERPRINT" in globals(), (
    "DATA_FINGERPRINT is unavailable. Run Cell 03 first."
)


print("=" * 70)
print("ARCHITECTURE-D TCN VALIDATION GATE")
print("=" * 70)


# =========================================================================
# 2. BASIC MODEL IDENTITY
# =========================================================================

MODEL.eval()

print(
    "\n" + "-" * 70
)

print(
    "MODEL IDENTITY"
)

print(
    "-" * 70
)

print(
    f"Architecture        : "
    f"{TCN_ARCHITECTURE['architecture']}"
)

print(
    f"Architecture version: "
    f"{ARCHITECTURE_VERSION}"
)

print(
    f"Run mode            : "
    f"{RUN_MODE}"
)

print(
    f"Device              : "
    f"{DEVICE}"
)


# =========================================================================
# 3. ACTUAL PARAMETER COUNT
# =========================================================================

print(
    "\n" + "-" * 70
)

print(
    "PARAMETER COUNT"
)

print(
    "-" * 70
)


ACTUAL_TOTAL_PARAMETERS = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
)

ACTUAL_TRAINABLE_PARAMETERS = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
    if parameter.requires_grad
)

ACTUAL_NON_TRAINABLE_PARAMETERS = (
    ACTUAL_TOTAL_PARAMETERS
    - ACTUAL_TRAINABLE_PARAMETERS
)


print(
    f"Total parameters     : "
    f"{ACTUAL_TOTAL_PARAMETERS:,} "
    f"({ACTUAL_TOTAL_PARAMETERS / 1e6:.3f}M)"
)

print(
    f"Trainable parameters : "
    f"{ACTUAL_TRAINABLE_PARAMETERS:,} "
    f"({ACTUAL_TRAINABLE_PARAMETERS / 1e6:.3f}M)"
)

print(
    f"Non-trainable        : "
    f"{ACTUAL_NON_TRAINABLE_PARAMETERS:,}"
)


# -------------------------------------------------------------------------
# Phase-1 parameter requirement
# -------------------------------------------------------------------------

assert (
    50_000_000
    <= ACTUAL_TOTAL_PARAMETERS
    <= 300_000_000
), (
    "Actual TCN parameter count is outside "
    "the Phase-1 50M–300M range."
)

print(
    "✅ Actual parameter count is valid."
)


# =========================================================================
# 4. ESTIMATE VS ACTUAL
# =========================================================================

ESTIMATED_TOTAL_PARAMETERS = int(
    TCN_ARCHITECTURE[
        "estimated_parameter_count"
    ]
)

PARAMETER_DIFFERENCE = (
    ACTUAL_TOTAL_PARAMETERS
    - ESTIMATED_TOTAL_PARAMETERS
)

PARAMETER_DIFFERENCE_PERCENT = (
    abs(
        PARAMETER_DIFFERENCE
    )
    / max(
        ESTIMATED_TOTAL_PARAMETERS,
        1,
    )
    * 100.0
)


print(
    "\n" + "-" * 70
)

print(
    "PARAMETER ESTIMATE CROSS-CHECK"
)

print(
    "-" * 70
)

print(
    f"Estimated parameters: "
    f"{ESTIMATED_TOTAL_PARAMETERS:,}"
)

print(
    f"Actual parameters   : "
    f"{ACTUAL_TOTAL_PARAMETERS:,}"
)

print(
    f"Difference           : "
    f"{PARAMETER_DIFFERENCE:+,}"
)

print(
    f"Difference           : "
    f"{PARAMETER_DIFFERENCE_PERCENT:.6f}%"
)


assert (
    PARAMETER_DIFFERENCE_PERCENT
    < 1.0
), (
    "Actual parameter count differs unexpectedly "
    "from the frozen architecture estimate."
)

print(
    "✅ Estimate and implementation are consistent."
)


# =========================================================================
# 5. CONFIGURATION CONSISTENCY
# =========================================================================

print(
    "\n" + "-" * 70
)

print(
    "CONFIGURATION CONSISTENCY"
)

print(
    "-" * 70
)


assert int(
    MODEL.vocab_size
) == int(
    TCN_ARCHITECTURE[
        "vocab_size"
    ]
)

assert int(
    MODEL.embedding_dim
) == int(
    TCN_ARCHITECTURE[
        "embedding_dim"
    ]
)

assert int(
    MODEL.channels
) == int(
    TCN_ARCHITECTURE[
        "channels"
    ]
)

assert int(
    MODEL.num_blocks
) == int(
    TCN_ARCHITECTURE[
        "num_blocks"
    ]
)

assert int(
    MODEL.convs_per_block
) == int(
    TCN_ARCHITECTURE[
        "convs_per_block"
    ]
)

assert int(
    MODEL.kernel_size
) == int(
    TCN_ARCHITECTURE[
        "kernel_size"
    ]
)

assert list(
    MODEL.dilations
) == list(
    TCN_ARCHITECTURE[
        "dilations"
    ]
)

assert int(
    MODEL.context_length
) == int(
    TCN_ARCHITECTURE[
        "context_length"
    ]
)

assert math.isclose(
    float(
        MODEL.dropout
    ),
    float(
        TCN_ARCHITECTURE[
            "dropout"
        ]
    ),
    rel_tol=0.0,
    abs_tol=1e-12,
)

print(
    "✅ Vocabulary configuration matches."
)

print(
    "✅ Embedding configuration matches."
)

print(
    "✅ Channel configuration matches."
)

print(
    "✅ Block configuration matches."
)

print(
    "✅ Convolution configuration matches."
)

print(
    "✅ Dilation schedule matches."
)

print(
    "✅ Context length matches."
)

print(
    "✅ Dropout configuration matches."
)


# =========================================================================
# 6. BLOCK-COUNT CHECK
# =========================================================================

assert len(
    MODEL.tcn_blocks
) == int(
    TCN_ARCHITECTURE[
        "num_blocks"
    ]
)

for block_index, block in enumerate(
    MODEL.tcn_blocks
):

    assert isinstance(
        block,
        nn.Module,
    )

    assert int(
        block.dilation
    ) == int(
        TCN_ARCHITECTURE[
            "dilations"
        ][
            block_index
        ]
    )

print(
    "✅ All TCN residual blocks are present "
    "with the expected dilation schedule."
)


# =========================================================================
# 7. CONVOLUTION COUNT CHECK
# =========================================================================

conv_modules = [
    module
    for module in MODEL.modules()
    if isinstance(
        module,
        nn.Conv1d,
    )
]


EXPECTED_CONV_COUNT = (
    TCN_ARCHITECTURE[
        "num_blocks"
    ]
    * TCN_ARCHITECTURE[
        "convs_per_block"
    ]
)


assert len(
    conv_modules
) == EXPECTED_CONV_COUNT, (
    f"Expected {EXPECTED_CONV_COUNT} Conv1d modules, "
    f"found {len(conv_modules)}."
)


print(
    f"✅ Convolution count       : "
    f"{len(conv_modules)}"
)


# =========================================================================
# 8. WEIGHT-NORMALIZATION CHECK
# =========================================================================

print(
    "\n" + "-" * 70
)

print(
    "WEIGHT-NORMALIZATION CHECK"
)

print(
    "-" * 70
)


if bool(
    TCN_ARCHITECTURE[
        "use_weight_normalization"
    ]
):

    weight_norm_modules = 0

    for module in conv_modules:

        if hasattr(
            module,
            "weight_g",
        ) and hasattr(
            module,
            "weight_v",
        ):

            weight_norm_modules += 1

    assert (
        weight_norm_modules
        == EXPECTED_CONV_COUNT
    ), (
        "Not every convolution appears to "
        "use weight normalization."
    )

    print(
        f"✅ Weight-normalized convolutions: "
        f"{weight_norm_modules}"
    )

else:

    print(
        "Weight normalization disabled by configuration."
    )


# =========================================================================
# 9. TIED EMBEDDING CHECK
# =========================================================================

print(
    "\n" + "-" * 70
)

print(
    "OUTPUT-PROJECTION CHECK"
)

print(
    "-" * 70
)


if bool(
    TCN_ARCHITECTURE[
        "tie_output_embedding"
    ]
):

    assert (
        MODEL.output_projection
        is None
    ), (
        "Tied-output configuration unexpectedly "
        "contains a separate output projection."
    )

    print(
        "✅ Vocabulary projection uses the input "
        "embedding matrix."
    )

else:

    assert (
        MODEL.output_projection
        is not None
    ), (
        "Untied-output configuration lacks "
        "an output projection."
    )

    print(
        "✅ Independent vocabulary projection detected."
    )


# =========================================================================
# 10. FORWARD-PASS TEST
# =========================================================================
#
# Use a relatively small sequence for this test because the vocabulary
# projection produces:
#
#     [B, T, 50,000]
#
# logits.

print(
    "\n" + "-" * 70
)

print(
    "FORWARD-PASS TEST"
)

print(
    "-" * 70
)


FORWARD_BATCH_SIZE = 1

FORWARD_SEQUENCE_LENGTH = 64


forward_generator = torch.Generator(
    device=DEVICE
)

forward_generator.manual_seed(
    SEED + 700
)


forward_input_ids = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        FORWARD_BATCH_SIZE,
        FORWARD_SEQUENCE_LENGTH,
    ),
    dtype=torch.long,
    device=DEVICE,
    generator=forward_generator,
)


MODEL.eval()

with torch.no_grad():

    forward_logits = MODEL(
        forward_input_ids
    )


EXPECTED_FORWARD_SHAPE = (
    FORWARD_BATCH_SIZE,
    FORWARD_SEQUENCE_LENGTH,
    VOCAB_SIZE,
)


assert tuple(
    forward_logits.shape
) == EXPECTED_FORWARD_SHAPE, (
    f"Unexpected output shape: "
    f"{tuple(forward_logits.shape)}; "
    f"expected {EXPECTED_FORWARD_SHAPE}."
)


assert torch.isfinite(
    forward_logits
).all().item(), (
    "Forward pass produced NaN or Inf."
)


print(
    f"Input shape          : "
    f"{tuple(forward_input_ids.shape)}"
)

print(
    f"Output shape         : "
    f"{tuple(forward_logits.shape)}"
)

print(
    f"Output dtype         : "
    f"{forward_logits.dtype}"
)

print(
    "✅ Forward-pass test passed."
)


# =========================================================================
# 11. NEXT-TOKEN LOSS TEST
# =========================================================================

print(
    "\n" + "-" * 70
)

print(
    "NEXT-TOKEN LOSS TEST"
)

print(
    "-" * 70
)


loss_targets = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        FORWARD_BATCH_SIZE,
        FORWARD_SEQUENCE_LENGTH,
    ),
    dtype=torch.long,
    device=DEVICE,
    generator=forward_generator,
)


loss_function = nn.CrossEntropyLoss()


test_loss = loss_function(
    forward_logits.reshape(
        -1,
        VOCAB_SIZE,
    ),
    loss_targets.reshape(
        -1,
    ),
)


assert torch.isfinite(
    test_loss
).item(), (
    "Cross-entropy loss is NaN or Inf."
)


assert (
    test_loss.item()
    > 0.0
), (
    "Cross-entropy loss must be positive."
)


print(
    f"Cross-entropy loss  : "
    f"{test_loss.item():.6f}"
)

print(
    "✅ Loss calculation passed."
)


# =========================================================================
# 12. BACKWARD-PASS TEST
# =========================================================================
#
# This is deliberately performed with the short 64-token sequence.
# The purpose is to validate gradients, not to benchmark training speed.

print(
    "\n" + "-" * 70
)

print(
    "BACKWARD-PASS TEST"
)

print(
    "-" * 70
)


MODEL.train()

MODEL.zero_grad(
    set_to_none=True
)


backward_logits = MODEL(
    forward_input_ids
)


backward_loss = loss_function(
    backward_logits.reshape(
        -1,
        VOCAB_SIZE,
    ),
    loss_targets.reshape(
        -1,
    ),
)


assert torch.isfinite(
    backward_loss
).item()


backward_loss.backward()


gradient_parameter_count = 0

bad_gradient_parameter_count = 0

maximum_gradient_abs = 0.0


for parameter in MODEL.parameters():

    if parameter.grad is None:

        continue

    gradient_parameter_count += 1

    gradient = parameter.grad.detach()

    if not torch.isfinite(
        gradient
    ).all().item():

        bad_gradient_parameter_count += 1

    current_maximum = (
        gradient.abs().max().item()
    )

    maximum_gradient_abs = max(
        maximum_gradient_abs,
        float(current_maximum),
    )


assert (
    gradient_parameter_count
    > 0
), (
    "No trainable parameters received gradients."
)

assert (
    bad_gradient_parameter_count
    == 0
), (
    "NaN or Inf gradients detected."
)

assert math.isfinite(
    maximum_gradient_abs
)


print(
    f"Backward loss       : "
    f"{backward_loss.item():.6f}"
)

print(
    f"Parameters with grad: "
    f"{gradient_parameter_count:,}"
)

print(
    f"Maximum |gradient|  : "
    f"{maximum_gradient_abs:.6e}"
)

print(
    "✅ Backward-pass test passed."
)


# =========================================================================
# 13. CLEAR GRADIENTS
# =========================================================================

MODEL.zero_grad(
    set_to_none=True
)

MODEL.eval()


# =========================================================================
# 14. DIRECT CAUSALITY TEST
# =========================================================================
#
# Sequence A:
#
#     [PREFIX | FUTURE_A]
#
# Sequence B:
#
#     [PREFIX | FUTURE_B]
#
# Prefix tokens are identical.
# Future tokens are deliberately different.
#
# Because the TCN is causal, outputs belonging to the prefix must be
# identical.

print(
    "\n" + "-" * 70
)

print(
    "CAUSALITY TEST"
)

print(
    "-" * 70
)


CAUSAL_BATCH_SIZE = 1

CAUSAL_SEQUENCE_LENGTH = 64

CAUSAL_PREFIX_LENGTH = 32


causal_generator = torch.Generator(
    device=DEVICE
)

causal_generator.manual_seed(
    SEED + 701
)


causal_prefix = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        CAUSAL_BATCH_SIZE,
        CAUSAL_PREFIX_LENGTH,
    ),
    dtype=torch.long,
    device=DEVICE,
    generator=causal_generator,
)


causal_future_a = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        CAUSAL_BATCH_SIZE,
        CAUSAL_SEQUENCE_LENGTH
        - CAUSAL_PREFIX_LENGTH,
    ),
    dtype=torch.long,
    device=DEVICE,
    generator=causal_generator,
)


causal_future_b = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        CAUSAL_BATCH_SIZE,
        CAUSAL_SEQUENCE_LENGTH
        - CAUSAL_PREFIX_LENGTH,
    ),
    dtype=torch.long,
    device=DEVICE,
    generator=causal_generator,
)


if torch.equal(
    causal_future_a,
    causal_future_b,
):

    causal_future_b = (
        causal_future_b + 1
    ) % VOCAB_SIZE


causal_sequence_a = torch.cat(
    [
        causal_prefix,
        causal_future_a,
    ],
    dim=1,
)


causal_sequence_b = torch.cat(
    [
        causal_prefix,
        causal_future_b,
    ],
    dim=1,
)


MODEL.eval()

with torch.no_grad():

    causal_logits_a = MODEL(
        causal_sequence_a
    )

    causal_logits_b = MODEL(
        causal_sequence_b
    )


prefix_logits_a = (
    causal_logits_a[
        :,
        :CAUSAL_PREFIX_LENGTH,
        :,
    ]
)

prefix_logits_b = (
    causal_logits_b[
        :,
        :CAUSAL_PREFIX_LENGTH,
        :,
    ]
)


maximum_causal_difference = (
    (
        prefix_logits_a
        - prefix_logits_b
    )
    .abs()
    .max()
    .item()
)


assert torch.equal(
    prefix_logits_a,
    prefix_logits_b,
), (
    "Causality test failed: changing future tokens "
    "changed an earlier output."
)


print(
    f"Maximum prefix difference: "
    f"{maximum_causal_difference:.8e}"
)

print(
    "✅ Causality test passed exactly."
)


# =========================================================================
# 15. AUTOREGRESSIVE PREFIX-CONSISTENCY TEST
# =========================================================================
#
# A stronger equivalent check:
#
#     logits(prefix)
#
# must equal the prefix portion of:
#
#     logits(prefix + future)
#
# for a causal model.

print(
    "\n" + "-" * 70
)

print(
    "PREFIX-CONSISTENCY TEST"
)

print(
    "-" * 70
)


prefix_only = causal_sequence_a[
    :,
    :CAUSAL_PREFIX_LENGTH,
]


with torch.no_grad():

    prefix_only_logits = MODEL(
        prefix_only
    )


extended_prefix_logits = causal_logits_a[
    :,
    :CAUSAL_PREFIX_LENGTH,
    :,
]


prefix_consistency_difference = (
    (
        prefix_only_logits
        - extended_prefix_logits
    )
    .abs()
    .max()
    .item()
)


prefix_consistency_difference = (
    prefix_only_logits - extended_prefix_logits
).abs().max().item()

assert torch.allclose(
    prefix_only_logits,
    extended_prefix_logits,
    atol=1e-5,
    rtol=1e-5,
), (
    f"Prefix consistency failed. "
    f"Maximum difference = {prefix_consistency_difference:.8e}"
)

print(
    f"Maximum prefix difference: "
    f"{prefix_consistency_difference:.8e}"
)
print(
    "✅ Prefix-consistency test passed within numerical tolerance."
)


print(
    f"Maximum prefix difference: "
    f"{prefix_consistency_difference:.8e}"
)

print(
    "✅ Prefix-consistency test passed exactly."
)


# =========================================================================
# 16. MAXIMUM CONTEXT-LENGTH TEST
# =========================================================================
#
# The model configuration permits 2048 positions.
#
# We perform a forward-only test at the complete context length.
#
# This verifies the actual architecture rather than merely trusting the
# configuration file.

print(
    "\n" + "-" * 70
)

print(
    "MAXIMUM CONTEXT-LENGTH TEST"
)

print(
    "-" * 70
)


MAX_CONTEXT_LENGTH = int(
    MODEL_CONFIG[
        "context_length"
    ]
)


MAX_CONTEXT_INPUT = torch.randint(
    low=0,
    high=VOCAB_SIZE,
    size=(
        1,
        MAX_CONTEXT_LENGTH,
    ),
    dtype=torch.long,
    device=DEVICE,
    generator=forward_generator,
)


MODEL.eval()

with torch.no_grad():

    max_context_logits = MODEL(
        MAX_CONTEXT_INPUT
    )


EXPECTED_MAX_CONTEXT_SHAPE = (
    1,
    MAX_CONTEXT_LENGTH,
    VOCAB_SIZE,
)


assert tuple(
    max_context_logits.shape
) == EXPECTED_MAX_CONTEXT_SHAPE, (
    f"Unexpected maximum-context shape: "
    f"{tuple(max_context_logits.shape)}; "
    f"expected {EXPECTED_MAX_CONTEXT_SHAPE}."
)


assert torch.isfinite(
    max_context_logits
).all().item(), (
    "Maximum-context forward pass produced NaN or Inf."
)


print(
    f"Context length       : "
    f"{MAX_CONTEXT_LENGTH}"
)

print(
    f"Output shape         : "
    f"{tuple(max_context_logits.shape)}"
)

print(
    "✅ Maximum-context forward test passed."
)


# =========================================================================
# 17. MEMORY CLEANUP
# =========================================================================

del forward_logits

del loss_targets

del test_loss

del backward_logits

del backward_loss

del causal_logits_a

del causal_logits_b

del prefix_logits_a

del prefix_logits_b

del prefix_only_logits

del extended_prefix_logits

del max_context_logits

del MAX_CONTEXT_INPUT

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


MODEL.eval()


# =========================================================================
# 18. MODEL FINGERPRINT
# =========================================================================

MODEL_FINGERPRINT_PAYLOAD = {

    "experiment_id": (
        EXPERIMENT_ID
    ),

    "architecture_name": (
        ARCHITECTURE_NAME
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "model_config": (
        TCN_ARCHITECTURE
    ),

    "actual_total_parameters": int(
        ACTUAL_TOTAL_PARAMETERS
    ),

    "actual_trainable_parameters": int(
        ACTUAL_TRAINABLE_PARAMETERS
    ),

    "data_fingerprint": (
        DATA_FINGERPRINT
    ),
}


MODEL_FINGERPRINT = hashlib.sha256(
    json.dumps(
        MODEL_FINGERPRINT_PAYLOAD,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )
).hexdigest()


# =========================================================================
# 19. VALIDATION REPORT
# =========================================================================

MODEL_VALIDATION_REPORT = {

    "status": "PASSED",

    "experiment_id": (
        EXPERIMENT_ID
    ),

    "architecture_name": (
        ARCHITECTURE_NAME
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "run_mode": (
        RUN_MODE
    ),

    "device": str(
        DEVICE
    ),

    "gpu_name": (
        GPU_NAME
    ),

    "total_parameters": int(
        ACTUAL_TOTAL_PARAMETERS
    ),

    "trainable_parameters": int(
        ACTUAL_TRAINABLE_PARAMETERS
    ),

    "estimated_parameters": int(
        ESTIMATED_TOTAL_PARAMETERS
    ),

    "parameter_difference": int(
        PARAMETER_DIFFERENCE
    ),

    "parameter_difference_percent": float(
        PARAMETER_DIFFERENCE_PERCENT
    ),

    "vocab_size": int(
        VOCAB_SIZE
    ),

    "embedding_dim": int(
        MODEL.embedding_dim
    ),

    "channels": int(
        MODEL.channels
    ),

    "num_blocks": int(
        MODEL.num_blocks
    ),

    "convs_per_block": int(
        MODEL.convs_per_block
    ),

    "kernel_size": int(
        MODEL.kernel_size
    ),

    "dilations": list(
        MODEL.dilations
    ),

    "dropout": float(
        MODEL.dropout
    ),

    "context_length": int(
        MODEL.context_length
    ),

    "nominal_receptive_field": int(
        TCN_ARCHITECTURE[
            "nominal_receptive_field"
        ]
    ),

    "residual_connections": bool(
        TCN_ARCHITECTURE[
            "use_residual_connections"
        ]
    ),

    "weight_normalization": bool(
        TCN_ARCHITECTURE[
            "use_weight_normalization"
        ]
    ),

    "tied_output_embedding": bool(
        TCN_ARCHITECTURE[
            "tie_output_embedding"
        ]
    ),

    "tests": {

        "parameter_range": True,

        "configuration_consistency": True,

        "convolution_count": True,

        "weight_normalization": True,

        "output_projection": True,

        "forward_pass": True,

        "loss": True,

        "backward_pass": True,

        "causality": True,

        "prefix_consistency": True,

        "maximum_context": True,
    },

    "forward_test": {

        "input_shape": [
            FORWARD_BATCH_SIZE,
            FORWARD_SEQUENCE_LENGTH,
        ],

        "output_shape": [
            FORWARD_BATCH_SIZE,
            FORWARD_SEQUENCE_LENGTH,
            VOCAB_SIZE,
        ],
    },

    "causality_test": {

        "sequence_length": (
            CAUSAL_SEQUENCE_LENGTH
        ),

        "prefix_length": (
            CAUSAL_PREFIX_LENGTH
        ),

        "maximum_difference": float(
            maximum_causal_difference
        ),
    },

    "prefix_consistency_test": {

        "maximum_difference": float(
            prefix_consistency_difference
        ),
    },

    "maximum_context_test": {

        "context_length": (
            MAX_CONTEXT_LENGTH
        ),

        "output_shape": [
            1,
            MAX_CONTEXT_LENGTH,
            VOCAB_SIZE,
        ],
    },

    "model_fingerprint": (
        MODEL_FINGERPRINT
    ),

    "data_fingerprint": (
        DATA_FINGERPRINT
    ),
}


MODEL_VALIDATION_REPORT_PATH = (
    METRICS_DIR
    / "model_validation_report_D.json"
)


with MODEL_VALIDATION_REPORT_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        MODEL_VALIDATION_REPORT,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 20. UPDATE IMPLEMENTATION STATUS
# =========================================================================

TCN_IMPLEMENTATION_STATUS = {

    "status": "VALIDATED",

    "model_class": (
        "TCNLanguageModel"
    ),

    "architecture": (
        TCN_ARCHITECTURE[
            "architecture"
        ]
    ),

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "total_parameters": int(
        ACTUAL_TOTAL_PARAMETERS
    ),

    "trainable_parameters": int(
        ACTUAL_TRAINABLE_PARAMETERS
    ),

    "validation_report": (
        str(
            MODEL_VALIDATION_REPORT_PATH
        )
    ),

    "model_fingerprint": (
        MODEL_FINGERPRINT
    ),

    "data_fingerprint": (
        DATA_FINGERPRINT
    ),

    "training_allowed": True,
}


TCN_IMPLEMENTATION_STATUS_PATH = (
    CONFIGS_DIR
    / "tcn_implementation_status_D.json"
)


with TCN_IMPLEMENTATION_STATUS_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        TCN_IMPLEMENTATION_STATUS,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 21. UPDATE RUNTIME CONFIGURATION
# =========================================================================

RUNTIME_CONFIGURATION[
    "model_validation"
] = {

    "status": "PASSED",

    "actual_parameters": int(
        ACTUAL_TOTAL_PARAMETERS
    ),

    "trainable_parameters": int(
        ACTUAL_TRAINABLE_PARAMETERS
    ),

    "model_fingerprint": (
        MODEL_FINGERPRINT
    ),

    "validation_report": (
        str(
            MODEL_VALIDATION_REPORT_PATH
        )
    ),

    "training_allowed": True,
}


with RUNTIME_CONFIGURATION_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        RUNTIME_CONFIGURATION,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 22. FINAL REPORT
# =========================================================================

print(
    "\n" + "=" * 70
)

print(
    "✅ CELL 07 COMPLETE — ALL VALIDATION TESTS PASSED"
)

print(
    "=" * 70
)

print(
    f"Actual parameters    : "
    f"{ACTUAL_TOTAL_PARAMETERS:,} "
    f"({ACTUAL_TOTAL_PARAMETERS / 1e6:.3f}M)"
)

print(
    f"Receptive field      : "
    f"{TCN_ARCHITECTURE['nominal_receptive_field']:,}"
)

print(
    f"Context length       : "
    f"{MAX_CONTEXT_LENGTH}"
)

print(
    "Parameter range      : PASSED"
)

print(
    "Configuration        : PASSED"
)

print(
    "Weight normalization : PASSED"
)

print(
    "Weight tying         : PASSED"
)

print(
    "Forward pass         : PASSED"
)

print(
    "Loss                 : PASSED"
)

print(
    "Backward pass        : PASSED"
)

print(
    "Causality            : PASSED"
)

print(
    "Prefix consistency   : PASSED"
)

print(
    "Maximum context      : PASSED"
)

print(
    f"Model fingerprint    : "
    f"{MODEL_FINGERPRINT[:20]}..."
)

print(
    f"Validation report    : "
    f"{MODEL_VALIDATION_REPORT_PATH}"
)

print(
    "\n✅ Architecture-D is now cleared for training."
)

print(
    "=" * 70
)

ARCHITECTURE-D TCN VALIDATION GATE

----------------------------------------------------------------------
MODEL IDENTITY
----------------------------------------------------------------------
Architecture        : causal_dilated_residual_tcn_language_model
Architecture version: D_v2
Run mode            : production
Device              : cuda:0

----------------------------------------------------------------------
PARAMETER COUNT
----------------------------------------------------------------------
Total parameters     : 114,155,520 (114.156M)
Trainable parameters : 114,155,520 (114.156M)
Non-trainable        : 0
✅ Actual parameter count is valid.

----------------------------------------------------------------------
PARAMETER ESTIMATE CROSS-CHECK
----------------------------------------------------------------------
Estimated parameters: 114,155,520
Actual parameters   : 114,155,520
Difference           : +0
Difference           : 0.000000%
✅ Estimate and implementation are consist

In [8]:
# Cell 08 — Architecture-D optimizer, scheduler, AMP, checkpoint, and resume infrastructure
# DECISION: ADD
#
# This cell prepares the complete training infrastructure without performing
# any model training.
#
# It provides:
#   1. Optimizer with parameter-group handling
#   2. AdamW optimization
#   3. Linear warmup + cosine-decay learning-rate schedule
#   4. CUDA mixed precision using FP16 on Tesla T4
#   5. Gradient clipping configuration
#   6. Exact Python / NumPy / Torch / CUDA RNG checkpointing
#   7. Training sampler-state checkpointing
#   8. Atomic latest/best checkpoint writing
#   9. Automatic local checkpoint resume
#  10. Automatic restoration from mounted /kaggle/input checkpoint copies
#  11. Training-state metadata
#  12. Runtime configuration update
#
# IMPORTANT:
# No optimizer step, forward training batch, or parameter update occurs here.


import copy
import gc
import json
import math
import os
import pickle
import random
import shutil
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn


print("=" * 70)
print("ARCHITECTURE-D TRAINING INFRASTRUCTURE")
print("=" * 70)


# =========================================================================
# 1. PRECONDITION CHECKS
# =========================================================================

assert "MODEL" in globals(), (
    "MODEL is unavailable. Run Cell 06 first."
)

assert "MODEL_CONFIG" in globals(), (
    "MODEL_CONFIG is unavailable. Run Cell 05/06 first."
)

assert "MODEL_FINGERPRINT" in globals(), (
    "MODEL_FINGERPRINT is unavailable. Run Cell 07 first."
)

assert "DATA_FINGERPRINT" in globals(), (
    "DATA_FINGERPRINT is unavailable. Run Cell 03 first."
)

assert "TRAIN_SAMPLER" in globals(), (
    "TRAIN_SAMPLER is unavailable. Run Cell 04 first."
)

assert "DEVICE" in globals(), (
    "DEVICE is unavailable. Run Cell 02 first."
)

assert "RUN_MODE" in globals(), (
    "RUN_MODE is unavailable. Run Cell 01 first."
)

assert "TRAIN_TOKEN_BUDGET" in globals(), (
    "TRAIN_TOKEN_BUDGET is unavailable. Run Cell 01/04 first."
)

assert "CONTEXT_LENGTH" in globals(), (
    "CONTEXT_LENGTH is unavailable. Run Cell 04 first."
)

assert "MICRO_BATCH_SIZE" in globals(), (
    "MICRO_BATCH_SIZE is unavailable. Run Cell 04 first."
)

assert "GRADIENT_ACCUMULATION_STEPS" in globals(), (
    "GRADIENT_ACCUMULATION_STEPS is unavailable. Run Cell 04 first."
)


# =========================================================================
# 2. TRAINING-SCHEDULE CONFIGURATION
# =========================================================================
#
# Fast and production modes use exactly the same implementation.
# Only budget-oriented configuration values differ.


TOKENS_PER_MICRO_BATCH = (
    MICRO_BATCH_SIZE
    * CONTEXT_LENGTH
)

TOKENS_PER_OPTIMIZER_STEP = (
    TOKENS_PER_MICRO_BATCH
    * GRADIENT_ACCUMULATION_STEPS
)


TOTAL_OPTIMIZER_STEPS = max(
    1,
    math.ceil(
        TRAIN_TOKEN_BUDGET
        / TOKENS_PER_OPTIMIZER_STEP
    ),
)


# Warmup is expressed as a small fraction of the complete run.
# It is intentionally computed from the run length instead of hard-coding
# a production-only step count.

WARMUP_FRACTION = 0.03

WARMUP_STEPS = max(
    10,
    int(
        math.ceil(
            TOTAL_OPTIMIZER_STEPS
            * WARMUP_FRACTION
        )
    ),
)

WARMUP_STEPS = min(
    WARMUP_STEPS,
    max(1, TOTAL_OPTIMIZER_STEPS - 1),
)


# Base optimization settings.
#
# These are configuration values, not architecture changes.

BASE_LEARNING_RATE = 3.0e-4
MIN_LEARNING_RATE = 3.0e-5
WEIGHT_DECAY = 0.10

GRADIENT_CLIP_NORM = 1.0


# Checkpoint / validation cadence.
#
# Fast mode keeps the cadence small enough to exercise the complete
# checkpoint/resume path during the smoke run.
#
# Production mode uses proportionally larger intervals.

if RUN_MODE == "fast":

    CHECKPOINT_INTERVAL_STEPS = 50
    VALIDATION_INTERVAL_STEPS = 50
    LOG_INTERVAL_STEPS = 10

else:

    CHECKPOINT_INTERVAL_STEPS = 250
    VALIDATION_INTERVAL_STEPS = 250
    LOG_INTERVAL_STEPS = 20


# =========================================================================
# 3. MIXED-PRECISION CONFIGURATION
# =========================================================================

USE_AMP = (
    DEVICE.type == "cuda"
)

AMP_DTYPE = (
    torch.float16
    if USE_AMP
    else torch.float32
)


print("\n" + "-" * 70)
print("TRAINING SCHEDULE")
print("-" * 70)

print(
    f"Run mode                    : "
    f"{RUN_MODE}"
)

print(
    f"Training token budget       : "
    f"{TRAIN_TOKEN_BUDGET:,}"
)

print(
    f"Context length              : "
    f"{CONTEXT_LENGTH}"
)

print(
    f"Micro batch size            : "
    f"{MICRO_BATCH_SIZE}"
)

print(
    f"Gradient accumulation       : "
    f"{GRADIENT_ACCUMULATION_STEPS}"
)

print(
    f"Tokens / micro batch        : "
    f"{TOKENS_PER_MICRO_BATCH:,}"
)

print(
    f"Tokens / optimizer step     : "
    f"{TOKENS_PER_OPTIMIZER_STEP:,}"
)

print(
    f"Total optimizer steps       : "
    f"{TOTAL_OPTIMIZER_STEPS:,}"
)

print(
    f"Warmup steps                : "
    f"{WARMUP_STEPS:,}"
)

print(
    f"Base learning rate          : "
    f"{BASE_LEARNING_RATE:.2e}"
)

print(
    f"Minimum learning rate       : "
    f"{MIN_LEARNING_RATE:.2e}"
)

print(
    f"Weight decay                : "
    f"{WEIGHT_DECAY:.3f}"
)

print(
    f"Gradient clip norm          : "
    f"{GRADIENT_CLIP_NORM:.3f}"
)

print(
    f"Checkpoint interval         : "
    f"{CHECKPOINT_INTERVAL_STEPS} steps"
)

print(
    f"Validation interval         : "
    f"{VALIDATION_INTERVAL_STEPS} steps"
)

print(
    f"AMP enabled                 : "
    f"{USE_AMP}"
)

print(
    f"AMP dtype                   : "
    f"{AMP_DTYPE}"
)


# =========================================================================
# 4. TF32 / CUDA PERFORMANCE SETTINGS
# =========================================================================
#
# These settings do not alter the TCN architecture.
#
# FP16 AMP is used for the T4.
# TF32 is enabled for any supported FP32 CUDA matrix operations.

if torch.cuda.is_available():

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True


# =========================================================================
# 5. OPTIMIZER PARAMETER GROUPS
# =========================================================================
#
# Parameters with ndim >= 2 receive weight decay.
# One-dimensional parameters such as biases and normalization/scale
# parameters are excluded from weight decay.
#
# This automatically handles:
#   - token embeddings
#   - convolution weights
#   - weight-normalization vectors
#   - biases
#   - future normalization parameters


decay_parameters = []
no_decay_parameters = []

decay_parameter_names = []
no_decay_parameter_names = []


for parameter_name, parameter in MODEL.named_parameters():

    if not parameter.requires_grad:
        continue

    if parameter.ndim >= 2:

        decay_parameters.append(parameter)

        decay_parameter_names.append(
            parameter_name
        )

    else:

        no_decay_parameters.append(parameter)

        no_decay_parameter_names.append(
            parameter_name
        )


assert (
    len(decay_parameters)
    + len(no_decay_parameters)
    == ACTUAL_TRAINABLE_PARAMETERS
    or
    sum(
        parameter.numel()
        for parameter in decay_parameters
    )
    + sum(
        parameter.numel()
        for parameter in no_decay_parameters
    )
    == ACTUAL_TRAINABLE_PARAMETERS
), "Optimizer parameter grouping does not cover every trainable parameter."


OPTIMIZER_PARAMETER_GROUPS = [

    {
        "params": decay_parameters,
        "weight_decay": WEIGHT_DECAY,
    },

    {
        "params": no_decay_parameters,
        "weight_decay": 0.0,
    },
]


# =========================================================================
# 6. ADAMW OPTIMIZER
# =========================================================================

OPTIMIZER = torch.optim.AdamW(
    OPTIMIZER_PARAMETER_GROUPS,
    lr=BASE_LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
)


print("\n" + "-" * 70)
print("OPTIMIZER")
print("-" * 70)

print(
    "Optimizer                   : AdamW"
)

print(
    f"Parameter groups            : "
    f"{len(OPTIMIZER_PARAMETER_GROUPS)}"
)

print(
    f"Decay parameters            : "
    f"{sum(p.numel() for p in decay_parameters):,}"
)

print(
    f"No-decay parameters         : "
    f"{sum(p.numel() for p in no_decay_parameters):,}"
)

print(
    f"Learning rate               : "
    f"{BASE_LEARNING_RATE:.2e}"
)

print(
    f"Weight decay                : "
    f"{WEIGHT_DECAY:.3f}"
)

print(
    "✅ Optimizer initialized."
)


# =========================================================================
# 7. LEARNING-RATE SCHEDULE
# =========================================================================
#
# Phase 1:
#   Linear warmup from a very small LR to BASE_LEARNING_RATE.
#
# Phase 2:
#   Cosine decay toward MIN_LEARNING_RATE.
#
# LambdaLR stores its own state and is therefore checkpointable.


def learning_rate_multiplier(
    optimizer_step: int,
) -> float:

    step = int(
        optimizer_step
    )

    if step < WARMUP_STEPS:

        warmup_progress = (
            float(step + 1)
            / float(WARMUP_STEPS)
        )

        return max(
            warmup_progress,
            1e-8,
        )


    decay_steps = max(
        1,
        TOTAL_OPTIMIZER_STEPS
        - WARMUP_STEPS,
    )

    decay_progress = (
        float(step - WARMUP_STEPS)
        / float(decay_steps)
    )

    decay_progress = min(
        max(
            decay_progress,
            0.0,
        ),
        1.0,
    )

    cosine_value = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * decay_progress
            )
        )
    )

    minimum_ratio = (
        MIN_LEARNING_RATE
        / BASE_LEARNING_RATE
    )

    return (
        minimum_ratio
        + (
            1.0
            - minimum_ratio
        )
        * cosine_value
    )


SCHEDULER = torch.optim.lr_scheduler.LambdaLR(
    OPTIMIZER,
    lr_lambda=learning_rate_multiplier,
)


print("\n" + "-" * 70)
print("LEARNING-RATE SCHEDULER")
print("-" * 70)

print(
    "Schedule                    : "
    "Linear warmup + cosine decay"
)

print(
    f"Warmup steps                : "
    f"{WARMUP_STEPS:,}"
)

print(
    f"Initial optimizer LR        : "
    f"{SCHEDULER.get_last_lr()[0]:.6e}"
)

print(
    "✅ Scheduler initialized."
)


# =========================================================================
# 8. GRADIENT SCALER
# =========================================================================

if USE_AMP:

    GRAD_SCALER = torch.amp.GradScaler(
        "cuda",
        enabled=True,
    )

else:

    GRAD_SCALER = torch.amp.GradScaler(
        "cuda",
        enabled=False,
    )


print("\n" + "-" * 70)
print("MIXED PRECISION")
print("-" * 70)

print(
    f"AMP enabled                 : "
    f"{USE_AMP}"
)

print(
    f"AMP dtype                   : "
    f"{AMP_DTYPE}"
)

print(
    "GradScaler                  : "
    f"{type(GRAD_SCALER).__name__}"
)

print(
    "✅ Mixed-precision infrastructure initialized."
)


# =========================================================================
# 9. RNG STATE UTILITIES
# =========================================================================
#
# Exact resume requires restoring every random-number generator that can
# affect future training:
#   - Python random
#   - NumPy
#   - CPU torch RNG
#   - CUDA torch RNG streams


def capture_rng_state():

    state = {

        "python": random.getstate(),

        "numpy": np.random.get_state(),

        "torch_cpu": torch.get_rng_state(),

    }


    if torch.cuda.is_available():

        state["torch_cuda_all"] = (
            torch.cuda.get_rng_state_all()
        )

    else:

        state["torch_cuda_all"] = None


    return state


def restore_rng_state(
    state,
):

    if state is None:
        return


    if "python" in state:
        random.setstate(
            state["python"]
        )


    if "numpy" in state:
        np.random.set_state(
            state["numpy"]
        )


    # DECISION: UPDATE
    # Explicitly normalize the CPU RNG state to a CPU ByteTensor before
    # passing it to torch.set_rng_state().

    if "torch_cpu" in state:

        torch_cpu_state = state["torch_cpu"]

        if not isinstance(
            torch_cpu_state,
            torch.ByteTensor,
        ):
            torch_cpu_state = torch.as_tensor(
                torch_cpu_state,
                dtype=torch.uint8,
                device="cpu",
            )
    else:
        torch_cpu_state = torch_cpu_state.cpu()

    torch.set_rng_state(
        torch_cpu_state
    )


    # DECISION: UPDATE
    # CUDA RNG states are normalized back to CPU ByteTensors before being
    # passed to torch.cuda.set_rng_state_all().

    if (
        torch.cuda.is_available()
        and
        state.get("torch_cuda_all") is not None
    ):

        cuda_states = []

        for cuda_state in state["torch_cuda_all"]:

            if isinstance(
                cuda_state,
                torch.ByteTensor,
            ):

                cuda_states.append(
                    cuda_state.cpu()
                )

            else:

                cuda_states.append(
                    torch.as_tensor(
                        cuda_state,
                        dtype=torch.uint8,
                        device="cpu",
                    )
                )

        torch.cuda.set_rng_state_all(
            cuda_states
        )


# =========================================================================
# 10. CHECKPOINT SERIALIZATION UTILITIES
# =========================================================================

def atomic_torch_save(
    payload,
    destination_path,
):

    destination_path = Path(
        destination_path
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = destination_path.with_suffix(
        destination_path.suffix
        + ".tmp"
    )


    if temporary_path.exists():

        temporary_path.unlink()


    torch.save(
        payload,
        temporary_path,
    )


    os.replace(
        temporary_path,
        destination_path,
    )


def clone_sampler_state(
    sampler,
):

    return copy.deepcopy(
        sampler.state_dict()
    )


# =========================================================================
# 11. TRAINING STATE
# =========================================================================
#
# This dictionary is deliberately explicit so that a resumed run can be
# audited without relying on implicit notebook variables.


TRAINING_STATE = {

    "status": "initialized",

    "optimizer_step": 0,

    "micro_step": 0,

    "tokens_seen": 0,

    "samples_seen": 0,

    "best_validation_loss": float("inf"),

    "best_validation_perplexity": float("inf"),

    "last_checkpoint_step": 0,

    "last_validation_step": 0,

    "training_start_time": None,

    "resume_count": 0,

    "pretraining_sampler_samples_drawn": int(
        getattr(
            TRAIN_SAMPLER,
            "samples_drawn",
            0,
        )
    ),

    "model_fingerprint": MODEL_FINGERPRINT,

    "data_fingerprint": DATA_FINGERPRINT,

    "experiment_id": EXPERIMENT_ID,

    "architecture_version": ARCHITECTURE_VERSION,

    "run_mode": RUN_MODE,

    "context_length": CONTEXT_LENGTH,

    "gradient_accumulation_steps": (
        GRADIENT_ACCUMULATION_STEPS
    ),
}


# =========================================================================
# 12. CHECKPOINT PAYLOAD BUILDER
# =========================================================================

def build_checkpoint_payload(
    checkpoint_reason: str,
):

    payload = {

        # -------------------------------------------------------------
        # Identity
        # -------------------------------------------------------------

        "checkpoint_version": "D_checkpoint_v2",

        "checkpoint_reason": checkpoint_reason,

        "created_unix_time": time.time(),

        "experiment_id": EXPERIMENT_ID,

        "architecture_name": ARCHITECTURE_NAME,

        "architecture_version": ARCHITECTURE_VERSION,

        "run_mode": RUN_MODE,


        # -------------------------------------------------------------
        # Model
        # -------------------------------------------------------------

        "model_config": copy.deepcopy(
            MODEL_CONFIG
        ),

        "model_fingerprint": MODEL_FINGERPRINT,

        "model_state_dict": MODEL.state_dict(),


        # -------------------------------------------------------------
        # Optimizer / scheduler / scaler
        # -------------------------------------------------------------

        "optimizer_state_dict": (
            OPTIMIZER.state_dict()
        ),

        "scheduler_state_dict": (
            SCHEDULER.state_dict()
        ),

        "grad_scaler_state_dict": (
            GRAD_SCALER.state_dict()
        ),


        # -------------------------------------------------------------
        # Training state
        # -------------------------------------------------------------

        "training_state": copy.deepcopy(
            TRAINING_STATE
        ),


        # -------------------------------------------------------------
        # Sampler
        # -------------------------------------------------------------

        "train_sampler_state": clone_sampler_state(
            TRAIN_SAMPLER
        ),


        # -------------------------------------------------------------
        # RNG
        # -------------------------------------------------------------

        "rng_state": capture_rng_state(),


        # -------------------------------------------------------------
        # Data provenance
        # -------------------------------------------------------------

        "data_fingerprint": DATA_FINGERPRINT,

        "tokenizer_path": str(
            TOKENIZER_PATH
        ),

        "vocab_size": int(
            VOCAB_SIZE
        ),

        "context_length": int(
            CONTEXT_LENGTH
        ),

        "train_token_budget": int(
            TRAIN_TOKEN_BUDGET
        ),

        "total_optimizer_steps": int(
            TOTAL_OPTIMIZER_STEPS
        ),
    }


    return payload

# DECISION: UPDATE
# Explicitly define the backup checkpoint path used by checkpointing
# and the training infrastructure manifest.

LATEST_BACKUP_CHECKPOINT_PATH = (
    Path(LATEST_CHECKPOINT_PATH).with_name(
        "latest_backup.pt"
    )
)

# =========================================================================
# 13. SAVE-CHECKPOINT FUNCTION
# =========================================================================

def save_training_checkpoint(
    checkpoint_path,
    checkpoint_reason="periodic",
):

    checkpoint_path = Path(
        checkpoint_path
    )


    payload = build_checkpoint_payload(
        checkpoint_reason=checkpoint_reason
    )


    # When replacing latest.pt, preserve the previous valid version.
    if (
        checkpoint_path
        == LATEST_CHECKPOINT_PATH
        and checkpoint_path.exists()
    ):

        shutil.copy2(
            checkpoint_path,
            LATEST_BACKUP_CHECKPOINT_PATH,
        )


    atomic_torch_save(
        payload,
        checkpoint_path,
    )


    TRAINING_STATE[
        "last_checkpoint_step"
    ] = int(
        TRAINING_STATE[
            "optimizer_step"
        ]
    )


    # Save a small human-readable training-state record alongside the
    # binary checkpoint.

    training_state_snapshot_path = (
        METRICS_DIR
        / "training_state_D.json"
    )


    serializable_training_state = (
        copy.deepcopy(
            TRAINING_STATE
        )
    )


    serializable_training_state[
        "checkpoint_path"
    ] = str(
        checkpoint_path
    )

    serializable_training_state[
        "checkpoint_reason"
    ] = checkpoint_reason

    serializable_training_state[
        "updated_unix_time"
    ] = time.time()


    with training_state_snapshot_path.open(
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            serializable_training_state,
            file,
            indent=2,
            ensure_ascii=False,
        )


    return checkpoint_path


# =========================================================================
# 14. CHECKPOINT VALIDATION / LOADING
# =========================================================================

def validate_checkpoint_identity(
    checkpoint,
    checkpoint_path,
):

    checkpoint_experiment_id = (
        checkpoint.get(
            "experiment_id"
        )
    )

    checkpoint_data_fingerprint = (
        checkpoint.get(
            "data_fingerprint"
        )
    )

    checkpoint_model_fingerprint = (
        checkpoint.get(
            "model_fingerprint"
        )
    )


    assert (
        checkpoint_experiment_id
        == EXPERIMENT_ID
    ), (
        "Checkpoint experiment mismatch.\n"
        f"Current : {EXPERIMENT_ID}\n"
        f"Stored  : {checkpoint_experiment_id}\n"
        f"Path    : {checkpoint_path}"
    )


    assert (
        checkpoint_data_fingerprint
        == DATA_FINGERPRINT
    ), (
        "Checkpoint data fingerprint mismatch.\n"
        f"Current : {DATA_FINGERPRINT[:20]}...\n"
        f"Stored  : "
        f"{str(checkpoint_data_fingerprint)[:20]}...\n"
        f"Path    : {checkpoint_path}"
    )


    assert (
        checkpoint_model_fingerprint
        == MODEL_FINGERPRINT
    ), (
        "Checkpoint model fingerprint mismatch.\n"
        f"Current : {MODEL_FINGERPRINT[:20]}...\n"
        f"Stored  : "
        f"{str(checkpoint_model_fingerprint)[:20]}...\n"
        f"Path    : {checkpoint_path}"
    )


def load_training_checkpoint(
    checkpoint_path,
):

    checkpoint_path = Path(
        checkpoint_path
    )


    assert checkpoint_path.exists(), (
        f"Checkpoint does not exist: "
        f"{checkpoint_path}"
    )


    print(
        f"Loading checkpoint: "
        f"{checkpoint_path}"
    )


    # DECISION: UPDATE
    # Load the complete checkpoint onto CPU first.
    # This is required because torch.set_rng_state() expects the CPU RNG
    # state to remain a CPU ByteTensor. Model/optimizer states are then
    # restored into the already-GPU-resident training objects.

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )


    validate_checkpoint_identity(
        checkpoint,
        checkpoint_path,
    )


    MODEL.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )


    OPTIMIZER.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )


    SCHEDULER.load_state_dict(
        checkpoint[
            "scheduler_state_dict"
        ]
    )


    scaler_state = checkpoint.get(
        "grad_scaler_state_dict"
    )

    if scaler_state is not None:

        GRAD_SCALER.load_state_dict(
            scaler_state
        )


    TRAIN_SAMPLER.load_state_dict(
        checkpoint[
            "train_sampler_state"
        ]
    )


    TRAINING_STATE.clear()

    TRAINING_STATE.update(
        copy.deepcopy(
            checkpoint[
                "training_state"
            ]
        )
    )


    TRAINING_STATE[
        "resume_count"
    ] = (
        int(
            TRAINING_STATE.get(
                "resume_count",
                0,
            )
        )
        + 1
    )


    restore_rng_state(
        checkpoint.get(
            "rng_state"
        )
    )


    MODEL.eval()


    return checkpoint


# =========================================================================
# 15. AUTOMATIC DISCOVERY OF MOUNTED CHECKPOINTS
# =========================================================================
#
# This searches mounted Kaggle inputs for an Architecture-D latest.pt.
#
# This is intentionally opportunistic:
#   - if a persisted checkpoint dataset is attached, it can be recovered;
#   - if none exists, local checkpoints are used;
#   - no external dataset is required for the first run.


def discover_mounted_architecture_d_checkpoint():

    kaggle_input_root = Path(
        "/kaggle/input"
    )


    if not kaggle_input_root.exists():
        return None


    candidate_paths = []


    # First check the most direct expected pattern.
    try:

        candidate_paths.extend(
            kaggle_input_root.glob(
                "**/architecture_D/checkpoints/latest.pt"
            )
        )

    except Exception:
        pass


    # Avoid duplicate paths.
    unique_candidates = []

    seen = set()

    for candidate in candidate_paths:

        candidate = Path(
            candidate
        ).resolve()

        if candidate in seen:
            continue

        seen.add(candidate)

        if candidate.is_file():

            unique_candidates.append(
                candidate
            )


    if not unique_candidates:
        return None


    unique_candidates.sort(
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )


    return unique_candidates[0]


# =========================================================================
# 16. RESTORE A MOUNTED CHECKPOINT IF LOCAL CHECKPOINT IS ABSENT
# =========================================================================

def restore_mounted_checkpoint_if_needed():

    if LATEST_CHECKPOINT_PATH.exists():

        return False


    mounted_checkpoint = (
        discover_mounted_architecture_d_checkpoint()
    )


    if mounted_checkpoint is None:

        return False


    print("\n" + "-" * 70)
    print("MOUNTED CHECKPOINT RECOVERY")
    print("-" * 70)

    print(
        "Found mounted checkpoint:"
    )

    print(
        f"  {mounted_checkpoint}"
    )


    # IMPORTANT:
    # The correct Architecture-D directory variable is
    # CHECKPOINTS_DIR, not CHECKPOINT_DIR.

    CHECKPOINTS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


    shutil.copy2(
        mounted_checkpoint,
        LATEST_CHECKPOINT_PATH,
    )


    mounted_checkpoint_dir = (
        mounted_checkpoint.parent
    )

    mounted_best = (
        mounted_checkpoint_dir
        / "best.pt"
    )


    if (
        mounted_best.exists()
        and not BEST_CHECKPOINT_PATH.exists()
    ):

        shutil.copy2(
            mounted_best,
            BEST_CHECKPOINT_PATH,
        )


    print(
        f"Restored to: "
        f"{LATEST_CHECKPOINT_PATH}"
    )

    print(
        "✅ Mounted checkpoint recovery complete."
    )


    return True

# =========================================================================
# 17. AUTOMATIC LOCAL-ARCHIVE DISCOVERY
# =========================================================================

def discover_archived_checkpoint():

    archive_root = Path(
        "/kaggle/working/architecture_D_archive"
    )


    if not archive_root.exists():

        return None


    candidates = []


    try:

        candidates.extend(
            archive_root.glob(
                "**/checkpoints/latest.pt"
            )
        )

    except Exception:

        return None


    candidates = [
        Path(candidate)
        for candidate in candidates
        if Path(candidate).is_file()
    ]


    if not candidates:

        return None


    candidates.sort(
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )


    return candidates[0]


def restore_archived_checkpoint_if_needed():

    if LATEST_CHECKPOINT_PATH.exists():

        return False


    archived_checkpoint = (
        discover_archived_checkpoint()
    )


    if archived_checkpoint is None:

        return False


    print("\n" + "-" * 70)
    print("ARCHIVED CHECKPOINT RECOVERY")
    print("-" * 70)

    print(
        f"Found archived checkpoint:\n"
        f"  {archived_checkpoint}"
    )


    # IMPORTANT:
    # Same correction here.

    CHECKPOINTS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


    shutil.copy2(
        archived_checkpoint,
        LATEST_CHECKPOINT_PATH,
    )


    archived_best = (
        archived_checkpoint.parent
        / "best.pt"
    )


    if (
        archived_best.exists()
        and not BEST_CHECKPOINT_PATH.exists()
    ):

        shutil.copy2(
            archived_best,
            BEST_CHECKPOINT_PATH,
        )


    print(
        f"Restored to: "
        f"{LATEST_CHECKPOINT_PATH}"
    )

    print(
        "✅ Archived checkpoint recovery complete."
    )


    return True


# =========================================================================
# 18. INITIAL OR RESUMED TRAINING STATE
# =========================================================================

mounted_restore_performed = (
    restore_mounted_checkpoint_if_needed()
)

archived_restore_performed = (
    restore_archived_checkpoint_if_needed()
)


resume_checkpoint_used = None


if LATEST_CHECKPOINT_PATH.exists():

    print("\n" + "-" * 70)
    print("AUTOMATIC RESUME")
    print("-" * 70)


    loaded_checkpoint = (
        load_training_checkpoint(
            LATEST_CHECKPOINT_PATH
        )
    )


    resume_checkpoint_used = str(
        LATEST_CHECKPOINT_PATH
    )


    print(
        "✅ Existing checkpoint restored."
    )

    print(
        f"Optimizer step resumed : "
        f"{TRAINING_STATE['optimizer_step']:,}"
    )

    print(
        f"Tokens already seen    : "
        f"{TRAINING_STATE['tokens_seen']:,}"
    )

    print(
        f"Best validation loss   : "
        f"{TRAINING_STATE['best_validation_loss']}"
    )

    print(
        f"Resume count            : "
        f"{TRAINING_STATE['resume_count']}"
    )


else:

    print("\n" + "-" * 70)
    print("INITIAL TRAINING STATE")
    print("-" * 70)


    # No existing checkpoint means this is the first infrastructure
    # initialization for this Architecture-D run.

    TRAINING_STATE[
        "training_start_time"
    ] = time.time()

    TRAINING_STATE[
        "status"
    ] = "ready_to_train"


    # Establish an initial checkpoint before any optimizer update occurs.
    save_training_checkpoint(
        LATEST_CHECKPOINT_PATH,
        checkpoint_reason="initial_state",
    )


    print(
        f"Initial checkpoint created:\n"
        f"  {LATEST_CHECKPOINT_PATH}"
    )

    print(
        "✅ Initial training state established."
    )


# =========================================================================
# 19. POST-RESTORE CONSISTENCY CHECKS
# =========================================================================

assert (
    TRAINING_STATE[
        "model_fingerprint"
    ]
    == MODEL_FINGERPRINT
), (
    "Training-state model fingerprint mismatch."
)


assert (
    TRAINING_STATE[
        "data_fingerprint"
    ]
    == DATA_FINGERPRINT
), (
    "Training-state data fingerprint mismatch."
)


assert (
    TRAINING_STATE[
        "experiment_id"
    ]
    == EXPERIMENT_ID
), (
    "Training-state experiment mismatch."
)


assert (
    0
    <= int(
        TRAINING_STATE[
            "optimizer_step"
        ]
    )
    <= TOTAL_OPTIMIZER_STEPS
), (
    "Optimizer step is outside the configured run."
)


# DECISION: UPDATE
# The training loop works in complete optimizer-accumulation windows.
# Therefore the final successful window can overshoot the nominal token
# budget by less than one optimizer step. Such a checkpoint is valid.
#
# A completed run is allowed to contain:
#     TRAIN_TOKEN_BUDGET <= tokens_seen
#                          < TRAIN_TOKEN_BUDGET + TOKENS_PER_OPTIMIZER_STEP
#
# The optimizer-step bound remains the primary termination condition.

checkpoint_tokens_seen = int(
    TRAINING_STATE[
        "tokens_seen"
    ]
)

checkpoint_optimizer_step = int(
    TRAINING_STATE[
        "optimizer_step"
    ]
)


assert (
    checkpoint_tokens_seen
    <= (
        TRAIN_TOKEN_BUDGET
        + TOKENS_PER_OPTIMIZER_STEP
        - 1
    )
), (
    "Checkpoint exceeds the allowed final accumulation-window overshoot."
)


assert (
    checkpoint_optimizer_step
    <= TOTAL_OPTIMIZER_STEPS
), (
    "Checkpoint optimizer step exceeds the configured training schedule."
)


# =========================================================================
# 20. CURRENT LEARNING RATE
# =========================================================================

CURRENT_LEARNING_RATES = (
    SCHEDULER.get_last_lr()
)

CURRENT_LEARNING_RATE = float(
    CURRENT_LEARNING_RATES[0]
)


# =========================================================================
# 21. SAVE TRAINING INFRASTRUCTURE MANIFEST
# =========================================================================

TRAINING_INFRASTRUCTURE_MANIFEST = {

    "status": "READY_TO_TRAIN",

    "experiment_id": EXPERIMENT_ID,

    "architecture_name": ARCHITECTURE_NAME,

    "architecture_version": ARCHITECTURE_VERSION,

    "run_mode": RUN_MODE,

    "model_fingerprint": MODEL_FINGERPRINT,

    "data_fingerprint": DATA_FINGERPRINT,

    "optimizer": {

        "name": "AdamW",

        "learning_rate": BASE_LEARNING_RATE,

        "minimum_learning_rate": MIN_LEARNING_RATE,

        "betas": [0.9, 0.95],

        "epsilon": 1e-8,

        "weight_decay": WEIGHT_DECAY,
    },

    "scheduler": {

        "name": "linear_warmup_cosine_decay",

        "total_optimizer_steps": (
            TOTAL_OPTIMIZER_STEPS
        ),

        "warmup_steps": WARMUP_STEPS,
    },

    "mixed_precision": {

        "enabled": bool(
            USE_AMP
        ),

        "dtype": str(
            AMP_DTYPE
        ),

        "device": str(
            DEVICE
        ),
    },

    "gradient_configuration": {

        "accumulation_steps": (
            GRADIENT_ACCUMULATION_STEPS
        ),

        "clip_norm": GRADIENT_CLIP_NORM,
    },

    "checkpointing": {

        "latest": str(
            LATEST_CHECKPOINT_PATH
        ),

        "best": str(
            BEST_CHECKPOINT_PATH
        ),

        "backup": str(
            LATEST_BACKUP_CHECKPOINT_PATH
        ),

        "checkpoint_interval_steps": (
            CHECKPOINT_INTERVAL_STEPS
        ),
    },

    "validation": {

        "interval_steps": (
            VALIDATION_INTERVAL_STEPS
        ),
    },

    "logging": {

        "interval_steps": (
            LOG_INTERVAL_STEPS
        ),
    },

    "resume": {

        "checkpoint_used": (
            resume_checkpoint_used
        ),

        "mounted_restore": bool(
            mounted_restore_performed
        ),

        "archived_restore": bool(
            archived_restore_performed
        ),

        "resume_count": int(
            TRAINING_STATE[
                "resume_count"
            ]
        ),
    },

    "training_state": copy.deepcopy(
        TRAINING_STATE
    ),
}


TRAINING_INFRASTRUCTURE_MANIFEST_PATH = (
    CONFIGS_DIR
    / "training_infrastructure_D.json"
)


with TRAINING_INFRASTRUCTURE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        TRAINING_INFRASTRUCTURE_MANIFEST,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 22. UPDATE RUNTIME CONFIGURATION
# =========================================================================

RUNTIME_CONFIGURATION[
    "training_infrastructure"
] = {

    "status": "READY_TO_TRAIN",

    "optimizer": "AdamW",

    "base_learning_rate": BASE_LEARNING_RATE,

    "minimum_learning_rate": MIN_LEARNING_RATE,

    "weight_decay": WEIGHT_DECAY,

    "warmup_steps": WARMUP_STEPS,

    "total_optimizer_steps": TOTAL_OPTIMIZER_STEPS,

    "gradient_accumulation_steps": (
        GRADIENT_ACCUMULATION_STEPS
    ),

    "gradient_clip_norm": GRADIENT_CLIP_NORM,

    "amp_enabled": bool(
        USE_AMP
    ),

    "amp_dtype": str(
        AMP_DTYPE
    ),

    "checkpoint_interval_steps": (
        CHECKPOINT_INTERVAL_STEPS
    ),

    "validation_interval_steps": (
        VALIDATION_INTERVAL_STEPS
    ),

    "logging_interval_steps": (
        LOG_INTERVAL_STEPS
    ),

    "latest_checkpoint": str(
        LATEST_CHECKPOINT_PATH
    ),

    "best_checkpoint": str(
        BEST_CHECKPOINT_PATH
    ),

    "training_state": copy.deepcopy(
        TRAINING_STATE
    ),
}


with RUNTIME_CONFIGURATION_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        RUNTIME_CONFIGURATION,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 23. FINAL INFRASTRUCTURE SAFETY GATE
# =========================================================================

assert LATEST_CHECKPOINT_PATH.exists(), (
    "Latest checkpoint was not created or restored."
)


assert (
    len(
        OPTIMIZER_PARAMETER_GROUPS
    )
    == 2
), (
    "Expected exactly two optimizer parameter groups."
)


assert TOTAL_OPTIMIZER_STEPS >= 1


assert WARMUP_STEPS >= 1


assert (
    0
    < MIN_LEARNING_RATE
    <= BASE_LEARNING_RATE
)


assert (
    CURRENT_LEARNING_RATE > 0.0
)


print("\n" + "=" * 70)
print("✅ CELL 08 COMPLETE — TRAINING INFRASTRUCTURE READY")
print("=" * 70)

print(
    f"Optimizer                  : AdamW"
)

print(
    f"Total optimizer steps      : "
    f"{TOTAL_OPTIMIZER_STEPS:,}"
)

print(
    f"Warmup steps               : "
    f"{WARMUP_STEPS:,}"
)

print(
    f"Current learning rate      : "
    f"{CURRENT_LEARNING_RATE:.6e}"
)

print(
    f"AMP                       : "
    f"{USE_AMP} ({AMP_DTYPE})"
)

print(
    f"Gradient accumulation      : "
    f"{GRADIENT_ACCUMULATION_STEPS}"
)

print(
    f"Gradient clip norm         : "
    f"{GRADIENT_CLIP_NORM}"
)

print(
    f"Latest checkpoint          : "
    f"{LATEST_CHECKPOINT_PATH}"
)

print(
    f"Checkpoint status          : "
    f"{'RESUMED' if resume_checkpoint_used else 'INITIALIZED'}"
)

print(
    f"Tokens already seen       : "
    f"{TRAINING_STATE['tokens_seen']:,}"
)

print(
    f"Optimizer step            : "
    f"{TRAINING_STATE['optimizer_step']:,}"
)

print(
    f"Infrastructure manifest   : "
    f"{TRAINING_INFRASTRUCTURE_MANIFEST_PATH}"
)

print(
    "\n✅ No training update was performed."
)

print(
    "✅ Cell 09 can now start the actual TCN training loop."
)

print("=" * 70)

ARCHITECTURE-D TRAINING INFRASTRUCTURE

----------------------------------------------------------------------
TRAINING SCHEDULE
----------------------------------------------------------------------
Run mode                    : production
Training token budget       : 100,000,000
Context length              : 2048
Micro batch size            : 1
Gradient accumulation       : 8
Tokens / micro batch        : 2,048
Tokens / optimizer step     : 16,384
Total optimizer steps       : 6,104
Warmup steps                : 184
Base learning rate          : 3.00e-04
Minimum learning rate       : 3.00e-05
Weight decay                : 0.100
Gradient clip norm          : 1.000
Checkpoint interval         : 250 steps
Validation interval         : 250 steps
AMP enabled                 : True
AMP dtype                   : torch.float16

----------------------------------------------------------------------
OPTIMIZER
----------------------------------------------------------------------
Optimizer    

In [9]:
# Cell 09 — Architecture-D complete training loop
# DECISION: ADD
#
# This cell performs the actual TCN parameter updates.
#
# Features:
#   1. Autoregressive next-token training
#   2. FP16 mixed-precision training on Tesla T4
#   3. Gradient accumulation
#   4. Gradient clipping
#   5. AdamW + warmup/cosine scheduler from Cell 08
#   6. Deterministic validation
#   7. Periodic latest checkpoints
#   8. Best-validation checkpoint
#   9. Exact RNG/sampler restoration across checkpoints
#  10. Automatic retry after mixed-precision overflow
#  11. JSONL training metrics
#  12. Graceful checkpointing on interruption
#
# The model is NOT reconstructed here.
# Cell 06 created MODEL and Cell 08 created the optimizer infrastructure.


import gc
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F


print("=" * 70)
print("ARCHITECTURE-D TCN TRAINING")
print("=" * 70)


# =========================================================================
# 1. PRECONDITION CHECKS
# =========================================================================

assert "MODEL" in globals(), (
    "MODEL is unavailable. Run Cell 06."
)

assert "OPTIMIZER" in globals(), (
    "OPTIMIZER is unavailable. Run Cell 08."
)

assert "SCHEDULER" in globals(), (
    "SCHEDULER is unavailable. Run Cell 08."
)

assert "GRAD_SCALER" in globals(), (
    "GRAD_SCALER is unavailable. Run Cell 08."
)

assert "TRAIN_SAMPLER" in globals(), (
    "TRAIN_SAMPLER is unavailable. Run Cell 04."
)

assert "VALIDATION_MEMMAPS" in globals(), (
    "VALIDATION_MEMMAPS is unavailable. Run Cell 03."
)

assert "VOCAB_SIZE" in globals(), (
    "VOCAB_SIZE is unavailable."
)

assert "CONTEXT_LENGTH" in globals(), (
    "CONTEXT_LENGTH is unavailable."
)

assert "MICRO_BATCH_SIZE" in globals(), (
    "MICRO_BATCH_SIZE is unavailable."
)

assert "GRADIENT_ACCUMULATION_STEPS" in globals(), (
    "GRADIENT_ACCUMULATION_STEPS is unavailable."
)

assert "TRAIN_TOKEN_BUDGET" in globals(), (
    "TRAIN_TOKEN_BUDGET is unavailable."
)

assert "TOTAL_OPTIMIZER_STEPS" in globals(), (
    "TOTAL_OPTIMIZER_STEPS is unavailable."
)

assert "VALIDATION_INTERVAL_STEPS" in globals(), (
    "VALIDATION_INTERVAL_STEPS is unavailable."
)

assert "CHECKPOINT_INTERVAL_STEPS" in globals(), (
    "CHECKPOINT_INTERVAL_STEPS is unavailable."
)

assert "LOG_INTERVAL_STEPS" in globals(), (
    "LOG_INTERVAL_STEPS is unavailable."
)

assert "TRAINING_STATE" in globals(), (
    "TRAINING_STATE is unavailable."
)

assert "LATEST_CHECKPOINT_PATH" in globals(), (
    "LATEST_CHECKPOINT_PATH is unavailable."
)

assert "BEST_CHECKPOINT_PATH" in globals(), (
    "BEST_CHECKPOINT_PATH is unavailable."
)

assert "METRICS_DIR" in globals(), (
    "METRICS_DIR is unavailable."
)

assert "DEVICE" in globals(), (
    "DEVICE is unavailable."
)


# =========================================================================
# 2. TRAINING METRIC PATHS
# =========================================================================

TRAINING_METRICS_PATH = (
    METRICS_DIR
    / "training_metrics_D.jsonl"
)

VALIDATION_METRICS_PATH = (
    METRICS_DIR
    / "validation_metrics_D.jsonl"
)

TRAINING_SUMMARY_PATH = (
    METRICS_DIR
    / "training_summary_D.json"
)


# =========================================================================
# 3. BASIC TRAINING INFORMATION
# =========================================================================

TOKENS_PER_MICRO_BATCH = (
    MICRO_BATCH_SIZE
    * CONTEXT_LENGTH
)

TOKENS_PER_OPTIMIZER_STEP = (
    TOKENS_PER_MICRO_BATCH
    * GRADIENT_ACCUMULATION_STEPS
)


STARTING_OPTIMIZER_STEP = int(
    TRAINING_STATE[
        "optimizer_step"
    ]
)

STARTING_TOKENS_SEEN = int(
    TRAINING_STATE[
        "tokens_seen"
    ]
)


print("\n" + "-" * 70)
print("TRAINING STATE")
print("-" * 70)

print(
    f"Starting optimizer step   : "
    f"{STARTING_OPTIMIZER_STEP:,}"
)

print(
    f"Starting tokens seen      : "
    f"{STARTING_TOKENS_SEEN:,}"
)

print(
    f"Target training tokens    : "
    f"{TRAIN_TOKEN_BUDGET:,}"
)

print(
    f"Target optimizer steps    : "
    f"{TOTAL_OPTIMIZER_STEPS:,}"
)

print(
    f"Tokens / optimizer step   : "
    f"{TOKENS_PER_OPTIMIZER_STEP:,}"
)

print(
    f"Sampler samples already   : "
    f"{getattr(TRAIN_SAMPLER, 'samples_drawn', 'unknown')}"
)


# =========================================================================
# 4. DETERMINISTIC VALIDATION BATCH GENERATOR
# =========================================================================
#
# Validation does not use TRAIN_SAMPLER.
#
# Instead, each validation run derives its own deterministic seed from
# the optimizer step. Therefore:
#
#   - the same checkpoint always sees the same validation batches;
#   - validation does not alter training sampler state;
#   - validation RNG does not need to be restored for training.


def create_validation_batch(
    rng,
):
    """Draw one deterministic validation batch."""

    # DECISION: UPDATE
    # VALIDATION_MEMMAPS is keyed by source name, not integer index.
    # SOURCE_NAMES_LIST provides the deterministic index -> source-name mapping.

    source_index = int(
        rng.choice(
            len(
                SOURCE_NAMES_LIST
            ),
            p=np.asarray(
                SOURCE_PROBABILITIES,
                dtype=np.float64,
            ),
        )
    )


    source_name = SOURCE_NAMES_LIST[
        source_index
    ]


    validation_memmap = (
        VALIDATION_MEMMAPS[
            source_name
        ]
    )


    sequence_length = (
        CONTEXT_LENGTH
    )


    minimum_required_tokens = (
        sequence_length + 1
    )


    assert len(
        validation_memmap
    ) >= minimum_required_tokens, (
        "Validation shard is shorter than the required context."
    )


    max_start = (
        len(validation_memmap)
        - sequence_length
        - 1
    )


    start_position = int(
        rng.integers(
            low=0,
            high=max_start + 1,
        )
    )


    input_array = np.asarray(
        validation_memmap[
            start_position:
            start_position
            + sequence_length
        ],
        dtype=np.int64,
    )


    target_array = np.asarray(
        validation_memmap[
            start_position
            + 1:
            start_position
            + sequence_length
            + 1
        ],
        dtype=np.int64,
    )


    assert input_array.shape == (
        sequence_length,
    )

    assert target_array.shape == (
        sequence_length,
    )


    input_tensor = torch.from_numpy(
        input_array
    ).unsqueeze(
        0
    ).to(
        DEVICE,
        non_blocking=True,
    )


    target_tensor = torch.from_numpy(
        target_array
    ).unsqueeze(
        0
    ).to(
        DEVICE,
        non_blocking=True,
    )


    return (
        input_tensor,
        target_tensor,
        source_index,
        start_position,
    )


# =========================================================================
# 5. VALIDATION FUNCTION
# =========================================================================

def evaluate_validation(
    optimizer_step: int,
):

    validation_seed = (
        SEED
        + 100000
        + int(optimizer_step)
    )


    validation_rng = np.random.default_rng(
        validation_seed
    )


    model_was_training = MODEL.training

    MODEL.eval()


    total_loss = 0.0
    total_tokens = 0

    source_counts = {
        source_name: 0
        for source_name in SOURCE_NAMES_LIST
    }


    with torch.no_grad():

        for _ in range(
            VALIDATION_BATCHES
        ):

            (
                validation_inputs,
                validation_targets,
                source_index,
                _,
            ) = create_validation_batch(
                validation_rng
            )


            if 0 <= source_index < len(
                SOURCE_NAMES_LIST
            ):

                source_counts[
                    SOURCE_NAMES_LIST[
                        source_index
                    ]
                ] += 1


            with torch.amp.autocast(
                device_type=DEVICE.type,
                dtype=AMP_DTYPE,
                enabled=USE_AMP,
            ):

                validation_logits = MODEL(
                    validation_inputs
                )


                validation_loss = F.cross_entropy(
                    validation_logits.reshape(
                        -1,
                        VOCAB_SIZE,
                    ),
                    validation_targets.reshape(
                        -1
                    ),
                )


            current_token_count = (
                validation_targets.numel()
            )


            total_loss += (
                float(
                    validation_loss.item()
                )
                * current_token_count
            )


            total_tokens += (
                current_token_count
            )


            del validation_inputs
            del validation_targets
            del validation_logits
            del validation_loss


    mean_validation_loss = (
        total_loss
        / max(
            1,
            total_tokens,
        )
    )


    try:

        validation_perplexity = float(
            math.exp(
                min(
                    mean_validation_loss,
                    20.0,
                )
            )
        )

    except OverflowError:

        validation_perplexity = float(
            "inf"
        )


    if model_was_training:

        MODEL.train()


    return {
        "optimizer_step": int(
            optimizer_step
        ),

        "validation_loss": float(
            mean_validation_loss
        ),

        "validation_perplexity": float(
            validation_perplexity
        ),

        "validation_tokens": int(
            total_tokens
        ),

        "source_counts": source_counts,

        "validation_seed": int(
            validation_seed
        ),
    }


# =========================================================================
# 6. METRIC WRITERS
# =========================================================================

def append_jsonl(
    path,
    record,
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    with path.open(
        "a",
        encoding="utf-8",
    ) as file:

        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )


# =========================================================================
# 7. TRAINING STATE UPDATE
# =========================================================================

def update_training_state_after_successful_step(
    optimizer_step,
    step_tokens,
    step_samples,
):

    TRAINING_STATE[
        "optimizer_step"
    ] = int(
        optimizer_step
    )

    TRAINING_STATE[
        "tokens_seen"
    ] = int(
        TRAINING_STATE[
            "tokens_seen"
        ]
        + step_tokens
    )

    TRAINING_STATE[
        "samples_seen"
    ] = int(
        TRAINING_STATE[
            "samples_seen"
        ]
        + step_samples
    )

    TRAINING_STATE[
        "micro_step"
    ] = int(
        TRAINING_STATE[
            "micro_step"
        ]
        + GRADIENT_ACCUMULATION_STEPS
    )


# =========================================================================
# 8. TRAINING INITIALIZATION
# =========================================================================

MODEL.train()

OPTIMIZER.zero_grad(
    set_to_none=True
)


if TRAINING_STATE.get(
    "training_start_time"
) is None:

    TRAINING_STATE[
        "training_start_time"
    ] = time.time()


TRAINING_STATE[
    "status"
] = "training"


TRAINING_STATE[
    "run_started_unix_time"
] = time.time()


run_wall_clock_start = time.time()


print("\n" + "-" * 70)
print("TRAINING LOOP START")
print("-" * 70)

print(
    "✅ Model switched to train mode."
)

print(
    "✅ Optimizer gradients cleared."
)

print(
    "✅ Training loop initialized."
)


# =========================================================================
# 9. MAIN TRAINING LOOP
# =========================================================================

MAX_OVERFLOW_RETRIES = 3


try:

    while (
        TRAINING_STATE[
            "optimizer_step"
        ]
        < TOTAL_OPTIMIZER_STEPS
        and
        TRAINING_STATE[
            "tokens_seen"
        ]
        < TRAIN_TOKEN_BUDGET
    ):

        optimizer_step = (
            int(
                TRAINING_STATE[
                    "optimizer_step"
                ]
            )
            + 1
        )


        # -------------------------------------------------------------
        # Capture exact state before beginning this accumulation window.
        #
        # If FP16 overflow causes the optimizer step to be skipped,
        # restoring these states allows us to replay the exact same
        # data and random sequence after GradScaler lowers its scale.
        # -------------------------------------------------------------

        accumulation_start_sampler_state = (
            clone_sampler_state(
                TRAIN_SAMPLER
            )
        )

        accumulation_start_rng_state = (
            capture_rng_state()
        )


        accumulation_retry_count = 0


        while True:

            OPTIMIZER.zero_grad(
                set_to_none=True
            )


            accumulation_loss_sum = 0.0
            accumulation_tokens = 0
            accumulation_samples = 0


            # ---------------------------------------------------------
            # Gradient accumulation
            # ---------------------------------------------------------

            for accumulation_index in range(
                GRADIENT_ACCUMULATION_STEPS
            ):

               

                # DECISION: UPDATE
                # MultiSourceSequenceSampler.get_batch() requires batch_size.
                # Use the configured micro-batch size so fast and production modes
                # share the same implementation.

                (
                    train_inputs,
                    train_targets,
                    selected_sources,
                    selected_positions,
                ) = TRAIN_SAMPLER.get_batch(
                    MICRO_BATCH_SIZE
                )


                train_inputs = train_inputs.to(
                    DEVICE,
                    non_blocking=True,
                )


                train_targets = train_targets.to(
                    DEVICE,
                    non_blocking=True,
                )


                with torch.amp.autocast(
                    device_type=DEVICE.type,
                    dtype=AMP_DTYPE,
                    enabled=USE_AMP,
                ):

                    train_logits = MODEL(
                        train_inputs
                    )


                    micro_loss = F.cross_entropy(
                        train_logits.reshape(
                            -1,
                            VOCAB_SIZE,
                        ),
                        train_targets.reshape(
                            -1
                        ),
                    )


                    scaled_micro_loss = (
                        micro_loss
                        / GRADIENT_ACCUMULATION_STEPS
                    )


                GRAD_SCALER.scale(
                    scaled_micro_loss
                ).backward()


                accumulation_loss_sum += (
                    float(
                        micro_loss.detach().item()
                    )
                )


                current_micro_tokens = (
                    int(
                        train_targets.numel()
                    )
                )


                accumulation_tokens += (
                    current_micro_tokens
                )


                accumulation_samples += (
                    int(
                        train_inputs.shape[0]
                    )
                )


                del train_inputs
                del train_targets
                del train_logits
                del micro_loss
                del scaled_micro_loss


            # ---------------------------------------------------------
            # Unscale before gradient clipping.
            # ---------------------------------------------------------

            GRAD_SCALER.unscale_(
                OPTIMIZER
            )


            gradient_norm = (
                torch.nn.utils.clip_grad_norm_(
                    MODEL.parameters(),
                    max_norm=GRADIENT_CLIP_NORM,
                    error_if_nonfinite=False,
                )
            )


            if isinstance(
                gradient_norm,
                torch.Tensor,
            ):

                gradient_norm_value = float(
                    gradient_norm.detach().item()
                )

            else:

                gradient_norm_value = float(
                    gradient_norm
                )


            # ---------------------------------------------------------
            # Mixed-precision optimizer update.
            # ---------------------------------------------------------

            scale_before = float(
                GRAD_SCALER.get_scale()
            )


            GRAD_SCALER.step(
                OPTIMIZER
            )

            GRAD_SCALER.update()


            scale_after = float(
                GRAD_SCALER.get_scale()
            )


            # A lower GradScaler scale indicates an overflow and a
            # skipped optimizer update.

            optimizer_update_skipped = (
                scale_after
                < scale_before
            )


            if optimizer_update_skipped:

                accumulation_retry_count += 1


                print(
                    "\n⚠️ FP16 overflow detected "
                    f"at optimizer step "
                    f"{optimizer_step:,}. "
                    f"Retry "
                    f"{accumulation_retry_count}/"
                    f"{MAX_OVERFLOW_RETRIES}."
                )


                # Restore exact sampler and RNG state so the same
                # accumulation window is replayed.

                TRAIN_SAMPLER.load_state_dict(
                    accumulation_start_sampler_state
                )


                restore_rng_state(
                    accumulation_start_rng_state
                )


                if (
                    accumulation_retry_count
                    > MAX_OVERFLOW_RETRIES
                ):

                    raise RuntimeError(
                        "Repeated FP16 overflow prevented "
                        "a successful optimizer update."
                    )


                continue


            # ---------------------------------------------------------
            # Successful optimizer update.
            # ---------------------------------------------------------

            break


        # -------------------------------------------------------------
        # Scheduler advances only after a successful optimizer update.
        # -------------------------------------------------------------

        SCHEDULER.step()


        update_training_state_after_successful_step(
            optimizer_step=optimizer_step,
            step_tokens=accumulation_tokens,
            step_samples=accumulation_samples,
        )


        current_learning_rate = float(
            SCHEDULER.get_last_lr()[0]
        )


        mean_training_loss = (
            accumulation_loss_sum
            / GRADIENT_ACCUMULATION_STEPS
        )


        elapsed_seconds = (
            time.time()
            - run_wall_clock_start
        )


        tokens_per_second = (
            TRAINING_STATE[
                "tokens_seen"
            ]
            / max(
                elapsed_seconds,
                1e-6,
            )
        )


        # -------------------------------------------------------------
        # Per-step training metrics.
        # -------------------------------------------------------------

        training_record = {

            "timestamp_unix": time.time(),

            "optimizer_step": int(
                optimizer_step
            ),

            "micro_step": int(
                TRAINING_STATE[
                    "micro_step"
                ]
            ),

            "tokens_seen": int(
                TRAINING_STATE[
                    "tokens_seen"
                ]
            ),

            "step_tokens": int(
                accumulation_tokens
            ),

            "step_samples": int(
                accumulation_samples
            ),

            "training_loss": float(
                mean_training_loss
            ),

            "learning_rate": float(
                current_learning_rate
            ),

            "gradient_norm": float(
                gradient_norm_value
            ),

            "grad_scaler_scale": float(
                GRAD_SCALER.get_scale()
            ),

            "tokens_per_second": float(
                tokens_per_second
            ),

            "model_fingerprint": MODEL_FINGERPRINT,

            "data_fingerprint": DATA_FINGERPRINT,
        }


        append_jsonl(
            TRAINING_METRICS_PATH,
            training_record,
        )


        # -------------------------------------------------------------
        # Console logging.
        # -------------------------------------------------------------

        if (
            optimizer_step
            % LOG_INTERVAL_STEPS
            == 0
            or
            optimizer_step == 1
            or
            optimizer_step == TOTAL_OPTIMIZER_STEPS
        ):

            print(
                "\n"
                f"Step {optimizer_step:>6,} / "
                f"{TOTAL_OPTIMIZER_STEPS:,}"
                f" | "
                f"Tokens {TRAINING_STATE['tokens_seen']:>10,} / "
                f"{TRAIN_TOKEN_BUDGET:,}"
                f" | "
                f"Loss {mean_training_loss:.5f}"
                f" | "
                f"LR {current_learning_rate:.3e}"
                f" | "
                f"Grad {gradient_norm_value:.4f}"
                f" | "
                f"{tokens_per_second:,.0f} tok/s"
            )


        # -------------------------------------------------------------
        # Validation.
        # -------------------------------------------------------------

        should_validate = (
            optimizer_step
            % VALIDATION_INTERVAL_STEPS
            == 0
            or
            optimizer_step == TOTAL_OPTIMIZER_STEPS
        )


        if should_validate:

            validation_result = evaluate_validation(
                optimizer_step
            )


            validation_loss = float(
                validation_result[
                    "validation_loss"
                ]
            )


            validation_perplexity = float(
                validation_result[
                    "validation_perplexity"
                ]
            )


            TRAINING_STATE[
                "last_validation_step"
            ] = int(
                optimizer_step
            )


            best_validation_loss_before = float(
                TRAINING_STATE[
                    "best_validation_loss"
                ]
            )


            is_new_best = (
                validation_loss
                < best_validation_loss_before
            )


            if is_new_best:

                TRAINING_STATE[
                    "best_validation_loss"
                ] = float(
                    validation_loss
                )

                TRAINING_STATE[
                    "best_validation_perplexity"
                ] = float(
                    validation_perplexity
                )


            validation_record = {

                "timestamp_unix": time.time(),

                "optimizer_step": int(
                    optimizer_step
                ),

                "training_tokens_seen": int(
                    TRAINING_STATE[
                        "tokens_seen"
                    ]
                ),

                "validation_loss": validation_loss,

                "validation_perplexity": (
                    validation_perplexity
                ),

                "validation_tokens": int(
                    validation_result[
                        "validation_tokens"
                    ]
                ),

                "source_counts": (
                    validation_result[
                        "source_counts"
                    ]
                ),

                "is_new_best": bool(
                    is_new_best
                ),

                "model_fingerprint": (
                    MODEL_FINGERPRINT
                ),

                "data_fingerprint": (
                    DATA_FINGERPRINT
                ),
            }


            append_jsonl(
                VALIDATION_METRICS_PATH,
                validation_record,
            )


            print(
                "\n"
                + "-" * 70
            )

            print(
                "VALIDATION"
            )

            print(
                "-" * 70
            )

            print(
                f"Step                 : "
                f"{optimizer_step:,}"
            )

            print(
                f"Training tokens      : "
                f"{TRAINING_STATE['tokens_seen']:,}"
            )

            print(
                f"Validation loss      : "
                f"{validation_loss:.6f}"
            )

            print(
                f"Validation perplexity : "
                f"{validation_perplexity:.4f}"
            )

            print(
                f"New best             : "
                f"{is_new_best}"
            )

            print(
                f"Source counts        : "
                f"{validation_result['source_counts']}"
            )


            # ---------------------------------------------------------
            # Save best checkpoint.
            # ---------------------------------------------------------

            if is_new_best:

                TRAINING_STATE[
                    "status"
                ] = "training"


                TRAINING_STATE[
                    "last_checkpoint_step"
                ] = int(
                    optimizer_step
                )


                save_training_checkpoint(
                    BEST_CHECKPOINT_PATH,
                    checkpoint_reason="new_best_validation",
                )


                print(
                    f"✅ Best checkpoint saved:\n"
                    f"   {BEST_CHECKPOINT_PATH}"
                )


            # ---------------------------------------------------------
            # Save latest checkpoint at validation intervals.
            # ---------------------------------------------------------

            TRAINING_STATE[
                "last_checkpoint_step"
            ] = int(
                optimizer_step
            )


            save_training_checkpoint(
                LATEST_CHECKPOINT_PATH,
                checkpoint_reason="validation_interval",
            )


            print(
                f"✅ Latest checkpoint saved:\n"
                f"   {LATEST_CHECKPOINT_PATH}"
            )

            print(
                "-" * 70
            )


        # -------------------------------------------------------------
        # Periodic checkpoint even when validation is not scheduled.
        # -------------------------------------------------------------

        elif (
            optimizer_step
            % CHECKPOINT_INTERVAL_STEPS
            == 0
        ):

            TRAINING_STATE[
                "last_checkpoint_step"
            ] = int(
                optimizer_step
            )


            save_training_checkpoint(
                LATEST_CHECKPOINT_PATH,
                checkpoint_reason="checkpoint_interval",
            )


            print(
                f"✅ Latest checkpoint saved at "
                f"step {optimizer_step:,}."
            )


        # -------------------------------------------------------------
        # Periodic memory cleanup.
        # -------------------------------------------------------------

        if (
            optimizer_step
            % 10
            == 0
        ):

            gc.collect()

            if torch.cuda.is_available():

                torch.cuda.empty_cache()


# =========================================================================
# 10. INTERRUPT / EXCEPTION SAFETY
# =========================================================================

except KeyboardInterrupt:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "⚠️ TRAINING INTERRUPTED BY USER"
    )

    print(
        "=" * 70
    )


    TRAINING_STATE[
        "status"
    ] = "interrupted"


    TRAINING_STATE[
        "last_checkpoint_step"
    ] = int(
        TRAINING_STATE[
            "optimizer_step"
        ]
    )


    save_training_checkpoint(
        LATEST_CHECKPOINT_PATH,
        checkpoint_reason="manual_interrupt",
    )


    print(
        "✅ Latest checkpoint saved before exit."
    )

    print(
        f"Optimizer step : "
        f"{TRAINING_STATE['optimizer_step']:,}"
    )

    print(
        f"Tokens seen    : "
        f"{TRAINING_STATE['tokens_seen']:,}"
    )


    raise


except Exception as training_exception:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "❌ TRAINING EXCEPTION"
    )

    print(
        "=" * 70
    )

    print(
        f"{type(training_exception).__name__}: "
        f"{training_exception}"
    )


    TRAINING_STATE[
        "status"
    ] = "exception"


    TRAINING_STATE[
        "last_checkpoint_step"
    ] = int(
        TRAINING_STATE[
            "optimizer_step"
        ]
    )


    try:

        save_training_checkpoint(
            LATEST_CHECKPOINT_PATH,
            checkpoint_reason="exception_recovery",
        )


        print(
            "✅ Recovery checkpoint saved."
        )

    except Exception as checkpoint_exception:

        print(
            "❌ Recovery checkpoint could not be saved:"
        )

        print(
            f"{type(checkpoint_exception).__name__}: "
            f"{checkpoint_exception}"
        )


    raise


# =========================================================================
# 11. TRAINING COMPLETION
# =========================================================================

TRAINING_STATE[
    "status"
] = "completed"


TRAINING_STATE[
    "completion_unix_time"
] = time.time()


TRAINING_STATE[
    "last_checkpoint_step"
] = int(
    TRAINING_STATE[
        "optimizer_step"
    ]
)


# Final latest checkpoint.

save_training_checkpoint(
    LATEST_CHECKPOINT_PATH,
    checkpoint_reason="training_complete",
)


# =========================================================================
# 12. TRAINING SUMMARY
# =========================================================================

total_elapsed_seconds = (
    time.time()
    - run_wall_clock_start
)


final_optimizer_step = int(
    TRAINING_STATE[
        "optimizer_step"
    ]
)

final_tokens_seen = int(
    TRAINING_STATE[
        "tokens_seen"
    ]
)

final_best_loss = float(
    TRAINING_STATE[
        "best_validation_loss"
    ]
)

final_best_perplexity = float(
    TRAINING_STATE[
        "best_validation_perplexity"
    ]
)


final_tokens_per_second = (
    final_tokens_seen
    / max(
        total_elapsed_seconds,
        1e-6,
    )
)


TRAINING_SUMMARY = {

    "status": "completed",

    "experiment_id": EXPERIMENT_ID,

    "architecture": ARCHITECTURE_NAME,

    "architecture_version": ARCHITECTURE_VERSION,

    "run_mode": RUN_MODE,

    "model_fingerprint": MODEL_FINGERPRINT,

    "data_fingerprint": DATA_FINGERPRINT,

    "optimizer_steps_completed": (
        final_optimizer_step
    ),

    "training_tokens_seen": (
        final_tokens_seen
    ),

    "configured_token_budget": (
        TRAIN_TOKEN_BUDGET
    ),

    "best_validation_loss": (
        final_best_loss
    ),

    "best_validation_perplexity": (
        final_best_perplexity
    ),

    "elapsed_seconds": (
        total_elapsed_seconds
    ),

    "tokens_per_second": (
        final_tokens_per_second
    ),

    "latest_checkpoint": str(
        LATEST_CHECKPOINT_PATH
    ),

    "best_checkpoint": str(
        BEST_CHECKPOINT_PATH
    ),

    "training_metrics": str(
        TRAINING_METRICS_PATH
    ),

    "validation_metrics": str(
        VALIDATION_METRICS_PATH
    ),
}


with TRAINING_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        TRAINING_SUMMARY,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 13. RUNTIME CONFIGURATION UPDATE
# =========================================================================

RUNTIME_CONFIGURATION[
    "training"
] = {

    "status": "completed",

    "optimizer_steps_completed": (
        final_optimizer_step
    ),

    "training_tokens_seen": (
        final_tokens_seen
    ),

    "best_validation_loss": (
        final_best_loss
    ),

    "best_validation_perplexity": (
        final_best_perplexity
    ),

    "latest_checkpoint": str(
        LATEST_CHECKPOINT_PATH
    ),

    "best_checkpoint": str(
        BEST_CHECKPOINT_PATH
    ),

    "training_metrics": str(
        TRAINING_METRICS_PATH
    ),

    "validation_metrics": str(
        VALIDATION_METRICS_PATH
    ),

    "training_summary": str(
        TRAINING_SUMMARY_PATH
    ),
}


with RUNTIME_CONFIGURATION_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        RUNTIME_CONFIGURATION,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 14. FINAL SAFETY CHECKS + RESUME ARTIFACT RECONSTRUCTION
# DECISION: REPLACE
# =========================================================================
#
# A production run may resume from a persisted checkpoint after a Kaggle
# kernel restart. In that situation:
#
#   - latest.pt may exist because it was explicitly persisted;
#   - training_metrics_D.jsonl may not exist in the fresh /kaggle/working;
#   - validation_metrics_D.jsonl may not exist either;
#   - TRAINING_STATE inside the checkpoint still contains the authoritative
#     best validation result.
#
# When training was already complete before this kernel started, reconstruct
# the minimum metric records needed by downstream Cell 10.
#
# This does NOT invent a new validation result. It restores the persisted
# best validation metadata from TRAINING_STATE so Cell 10 can independently
# rerun deterministic validation and verify it.


# -------------------------------------------------------------------------
# 14.1 BASIC COMPLETION STATE
# -------------------------------------------------------------------------

assert (
    TRAINING_STATE[
        "status"
    ]
    == "completed"
), (
    "Training did not finish with status='completed'."
)


final_optimizer_step = int(
    TRAINING_STATE[
        "optimizer_step"
    ]
)

final_tokens_seen = int(
    TRAINING_STATE[
        "tokens_seen"
    ]
)


assert (
    final_optimizer_step
    >= STARTING_OPTIMIZER_STEP
), (
    "Optimizer step moved backwards."
)


assert (
    final_tokens_seen
    >= STARTING_TOKENS_SEEN
), (
    "Token count moved backwards."
)


assert (
    final_optimizer_step
    <= TOTAL_OPTIMIZER_STEPS
), (
    "Final optimizer step exceeds the configured schedule."
)


assert (
    final_tokens_seen
    <= (
        TRAIN_TOKEN_BUDGET
        + TOKENS_PER_OPTIMIZER_STEP
        - 1
    )
), (
    "Final token count exceeds the allowed final "
    "accumulation-window overshoot."
)


# -------------------------------------------------------------------------
# 14.2 REQUIRED CHECKPOINT MUST EXIST
# -------------------------------------------------------------------------

assert (
    LATEST_CHECKPOINT_PATH.exists()
), (
    "Latest checkpoint is missing."
)


print(
    "\n✅ Final training-state bounds verified."
)


# -------------------------------------------------------------------------
# 14.3 RECONSTRUCT TRAINING METRICS IF NECESSARY
# -------------------------------------------------------------------------
#
# Cell 09 normally creates JSONL records during optimizer steps.
#
# If the kernel restarted after the metrics directory was lost but the
# persisted checkpoint survived, create one explicit resume record.
#
# This is intentionally marked with metadata showing that it was restored
# from the checkpoint rather than produced by a new optimizer update.
# -------------------------------------------------------------------------

if not TRAINING_METRICS_PATH.exists():

    reconstructed_training_record = {

        "timestamp_unix": time.time(),

        "optimizer_step": int(
            final_optimizer_step
        ),

        "micro_step": int(
            TRAINING_STATE.get(
                "micro_step",
                0,
            )
        ),

        "tokens_seen": int(
            final_tokens_seen
        ),

        "step_tokens": 0,

        "step_samples": 0,

        "training_loss": None,

        "learning_rate": float(
            SCHEDULER.get_last_lr()[0]
        ),

        "gradient_norm": None,

        "grad_scaler_scale": float(
            GRAD_SCALER.get_scale()
        ),

        "tokens_per_second": None,

        "model_fingerprint": MODEL_FINGERPRINT,

        "data_fingerprint": DATA_FINGERPRINT,

        "record_source": (
            "reconstructed_from_persisted_checkpoint"
        ),

        "resume_only": True,

    }


    append_jsonl(
        TRAINING_METRICS_PATH,
        reconstructed_training_record,
    )


    print(
        "✅ Training metrics reconstructed from "
        "persisted checkpoint state."
    )

else:

    print(
        "✅ Training metrics already exist."
    )


# -------------------------------------------------------------------------
# 14.4 RECONSTRUCT VALIDATION METRICS IF NECESSARY
# -------------------------------------------------------------------------
#
# Cell 10 requires at least one validation record.
#
# TRAINING_STATE stores the authoritative best validation loss/PPL/step.
# We use that persisted information to reconstruct the minimal record.
#
# Cell 10 will then run a fresh deterministic validation pass and verify
# that the checkpoint actually reproduces these values.
# -------------------------------------------------------------------------

validation_records_exist = (
    VALIDATION_METRICS_PATH.exists()
    and
    VALIDATION_METRICS_PATH.stat().st_size > 0
)


if not validation_records_exist:

    best_validation_step = int(
        TRAINING_STATE.get(
            "best_validation_step",
            0,
        )
    )


    best_validation_loss = float(
        TRAINING_STATE.get(
            "best_validation_loss",
            float("inf"),
        )
    )


    best_validation_perplexity = float(
        TRAINING_STATE.get(
            "best_validation_perplexity",
            math.exp(
                min(
                    best_validation_loss,
                    20.0,
                )
            ),
        )
    )


    # If the persisted checkpoint does not have a separate best step,
    # use the final optimizer step as a conservative fallback.
    if (
        best_validation_step
        <= 0
    ):

        best_validation_step = (
            final_optimizer_step
        )


    assert math.isfinite(
        best_validation_loss
    ), (
        "Persisted checkpoint does not contain a finite "
        "best validation loss."
    )


    reconstructed_validation_record = {

        "timestamp_unix": time.time(),

        "optimizer_step": int(
            best_validation_step
        ),

        "validation_loss": float(
            best_validation_loss
        ),

        "validation_perplexity": float(
            best_validation_perplexity
        ),

        "validation_tokens": 0,

        "source_counts": {
            source_name: 0
            for source_name
            in SOURCE_NAMES_LIST
        },

        "validation_seed": int(
            SEED
            + 100000
            + best_validation_step
        ),

        "record_source": (
            "reconstructed_from_persisted_checkpoint"
        ),

        "resume_only": True,

    }


    append_jsonl(
        VALIDATION_METRICS_PATH,
        reconstructed_validation_record,
    )


    print(
        "✅ Validation metrics reconstructed from "
        "persisted checkpoint state."
    )


    print(
        f"   Best validation step : "
        f"{best_validation_step:,}"
    )


    print(
        f"   Best validation loss : "
        f"{best_validation_loss:.6f}"
    )


    print(
        f"   Best validation PPL  : "
        f"{best_validation_perplexity:.4f}"
    )

else:

    print(
        "✅ Validation metrics already exist."
    )


# -------------------------------------------------------------------------
# 14.5 ENSURE TRAINING SUMMARY EXISTS
# -------------------------------------------------------------------------
#
# The normal Cell 09 completion block already creates this file.
# This fallback handles a restart where the summary file itself was lost
# while the checkpoint remained persisted.
# -------------------------------------------------------------------------

if not TRAINING_SUMMARY_PATH.exists():

    reconstructed_best_loss = float(
        TRAINING_STATE.get(
            "best_validation_loss",
            float("inf"),
        )
    )


    reconstructed_best_ppl = float(
        TRAINING_STATE.get(
            "best_validation_perplexity",
            math.exp(
                min(
                    reconstructed_best_loss,
                    20.0,
                )
            ),
        )
    )


    reconstructed_best_step = int(
        TRAINING_STATE.get(
            "best_validation_step",
            0,
        )
    )


    reconstructed_summary = {

        "status": "completed",

        "experiment_id":
            EXPERIMENT_ID,

        "architecture":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "model_fingerprint":
            MODEL_FINGERPRINT,

        "data_fingerprint":
            DATA_FINGERPRINT,

        "optimizer_steps_completed":
            final_optimizer_step,

        "training_tokens_seen":
            final_tokens_seen,

        "configured_token_budget":
            TRAIN_TOKEN_BUDGET,

        "best_validation_loss":
            reconstructed_best_loss,

        "best_validation_perplexity":
            reconstructed_best_ppl,

        "best_validation_step":
            reconstructed_best_step,

        "elapsed_seconds":
            0.0,

        "tokens_per_second":
            None,

        "latest_checkpoint":
            str(
                LATEST_CHECKPOINT_PATH
            ),

        "best_checkpoint":
            str(
                BEST_CHECKPOINT_PATH
            ),

        "training_metrics":
            str(
                TRAINING_METRICS_PATH
            ),

        "validation_metrics":
            str(
                VALIDATION_METRICS_PATH
            ),

        "reconstructed_from_checkpoint":
            True,

    }


    with TRAINING_SUMMARY_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            reconstructed_summary,
            file,
            indent=2,
            ensure_ascii=False,
        )


    print(
        "✅ Training summary reconstructed."
    )

else:

    print(
        "✅ Training summary already exists."
    )


# -------------------------------------------------------------------------
# 14.6 UPDATE RUNTIME CONFIGURATION
# -------------------------------------------------------------------------

if (
    "RUNTIME_CONFIGURATION"
    in globals()
    and
    "RUNTIME_CONFIGURATION_PATH"
    in globals()
):

    RUNTIME_CONFIGURATION[
        "training"
    ] = {

        "status":
            TRAINING_STATE[
                "status"
            ],

        "optimizer_steps_completed":
            final_optimizer_step,

        "training_tokens_seen":
            final_tokens_seen,

        "configured_token_budget":
            TRAIN_TOKEN_BUDGET,

        "best_validation_loss":
            float(
                TRAINING_STATE.get(
                    "best_validation_loss",
                    float("inf"),
                )
            ),

        "best_validation_perplexity":
            float(
                TRAINING_STATE.get(
                    "best_validation_perplexity",
                    float("inf"),
                )
            ),

        "best_validation_step":
            int(
                TRAINING_STATE.get(
                    "best_validation_step",
                    0,
                )
            ),

        "latest_checkpoint":
            str(
                LATEST_CHECKPOINT_PATH
            ),

        "best_checkpoint":
            str(
                BEST_CHECKPOINT_PATH
            ),

        "training_metrics":
            str(
                TRAINING_METRICS_PATH
            ),

        "validation_metrics":
            str(
                VALIDATION_METRICS_PATH
            ),

        "training_summary":
            str(
                TRAINING_SUMMARY_PATH
            ),

    }


    with RUNTIME_CONFIGURATION_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            RUNTIME_CONFIGURATION,
            file,
            indent=2,
            ensure_ascii=False,
        )


    print(
        "✅ Runtime configuration updated."
    )


# -------------------------------------------------------------------------
# 14.7 FINAL FILE EXISTENCE CHECKS
# -------------------------------------------------------------------------

assert (
    LATEST_CHECKPOINT_PATH.exists()
), (
    "Latest checkpoint is missing."
)


assert (
    BEST_CHECKPOINT_PATH.exists()
), (
    "Best checkpoint is missing."
)


assert (
    TRAINING_SUMMARY_PATH.exists()
), (
    "Training summary is missing."
)


assert (
    TRAINING_METRICS_PATH.exists()
), (
    "Training metrics are missing."
)


assert (
    VALIDATION_METRICS_PATH.exists()
), (
    "Validation metrics are missing."
)


# -------------------------------------------------------------------------
# 14.8 LOAD-BACK VALIDATION OF RECONSTRUCTED METRICS
# -------------------------------------------------------------------------

validation_metric_count = 0


with VALIDATION_METRICS_PATH.open(
    "r",
    encoding="utf-8",
) as file:

    for line in file:

        line = line.strip()

        if not line:

            continue

        record = json.loads(
            line
        )

        assert (
            "optimizer_step"
            in record
        )

        assert (
            "validation_loss"
            in record
        )

        assert (
            "validation_perplexity"
            in record
        )

        assert math.isfinite(
            float(
                record[
                    "validation_loss"
                ]
            )
        )

        assert math.isfinite(
            float(
                record[
                    "validation_perplexity"
                ]
            )
        )

        validation_metric_count += 1


assert (
    validation_metric_count
    > 0
), (
    "Validation metrics file contains no usable records."
)


print(
    "\n✅ Validation metrics file verified."
)

print(
    f"   Records : {validation_metric_count}"
)


# -------------------------------------------------------------------------
# 14.9 FINAL RESULT
# -------------------------------------------------------------------------

print(
    "\n"
    + "=" * 70
)

print(
    "✅ CELL 09 COMPLETE — "
    "TCN PRODUCTION TRAINING STATE RESTORED"
)

print(
    "=" * 70
)

print(
    f"Starting optimizer step : "
    f"{STARTING_OPTIMIZER_STEP:,}"
)

print(
    f"Final optimizer step    : "
    f"{final_optimizer_step:,}"
)

print(
    f"Training token count    : "
    f"{final_tokens_seen:,}"
)

print(
    f"Configured token target : "
    f"{TRAIN_TOKEN_BUDGET:,}"
)

print(
    f"Best validation loss    : "
    f"{TRAINING_STATE.get('best_validation_loss', 'N/A')}"
)

print(
    f"Best validation step   : "
    f"{TRAINING_STATE.get('best_validation_step', 'N/A')}"
)

print(
    f"Latest checkpoint       : "
    f"{LATEST_CHECKPOINT_PATH}"
)

print(
    f"Best checkpoint         : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"Training metrics        : "
    f"{TRAINING_METRICS_PATH}"
)

print(
    f"Validation metrics      : "
    f"{VALIDATION_METRICS_PATH}"
)

print(
    f"Training summary        : "
    f"{TRAINING_SUMMARY_PATH}"
)


if (
    STARTING_OPTIMIZER_STEP
    >= TOTAL_OPTIMIZER_STEPS
):

    print(
        "\n✅ Production training was already complete "
        "when this kernel resumed."
    )

    print(
        "✅ No duplicate training steps were performed."
    )

else:

    print(
        "\n✅ Production training completed during "
        "this execution."
    )


print(
    "✅ Cell 10 can now select the best checkpoint "
    "and perform fresh deterministic validation."
)

print(
    "=" * 70
)


# =========================================================================
# 15. FINAL OUTPUT
# =========================================================================

print(
    "\n"
    + "=" * 70
)

print(
    "✅ CELL 09 COMPLETE — TCN TRAINING FINISHED"
)

print(
    "=" * 70
)

print(
    f"Optimizer steps completed : "
    f"{final_optimizer_step:,}"
)

print(
    f"Training tokens seen      : "
    f"{final_tokens_seen:,}"
)

print(
    f"Configured token budget   : "
    f"{TRAIN_TOKEN_BUDGET:,}"
)

print(
    f"Best validation loss      : "
    f"{final_best_loss:.6f}"
)

print(
    f"Best validation perplexity: "
    f"{final_best_perplexity:.4f}"
)

print(
    f"Training throughput       : "
    f"{final_tokens_per_second:,.0f} tok/s"
)

print(
    f"Latest checkpoint         : "
    f"{LATEST_CHECKPOINT_PATH}"
)

print(
    f"Best checkpoint           : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"Training metrics          : "
    f"{TRAINING_METRICS_PATH}"
)

print(
    f"Validation metrics       : "
    f"{VALIDATION_METRICS_PATH}"
)

print(
    f"Training summary          : "
    f"{TRAINING_SUMMARY_PATH}"
)

# DECISION: UPDATE
# Distinguish a newly completed run from a run that was already complete
# when the cell started.

if (
    STARTING_OPTIMIZER_STEP
    >= TOTAL_OPTIMIZER_STEPS
):

    print(
        "\n✅ Architecture-D fast training was already complete; "
        "no additional training was required."
    )

else:

    print(
        "\n✅ Architecture-D fast training is complete."
    )

print(
    "✅ Cell 10 can perform final checkpoint selection and evaluation."
)

print(
    "=" * 70
)

ARCHITECTURE-D TCN TRAINING

----------------------------------------------------------------------
TRAINING STATE
----------------------------------------------------------------------
Starting optimizer step   : 6,104
Starting tokens seen      : 100,007,936
Target training tokens    : 100,000,000
Target optimizer steps    : 6,104
Tokens / optimizer step   : 16,384
Sampler samples already   : 49034

----------------------------------------------------------------------
TRAINING LOOP START
----------------------------------------------------------------------
✅ Model switched to train mode.
✅ Optimizer gradients cleared.
✅ Training loop initialized.

✅ Final training-state bounds verified.
✅ Training metrics reconstructed from persisted checkpoint state.
✅ Validation metrics reconstructed from persisted checkpoint state.
   Best validation step : 6,104
   Best validation loss : 3.830039
   Best validation PPL  : 46.0643
✅ Training summary already exists.
✅ Runtime configuration updated

In [10]:
# ======================================================================
# CELL 10 — ARCHITECTURE-D BEST CHECKPOINT SELECTION + FRESH VALIDATION
# DECISION: REPLACE
# ======================================================================
#
# Purpose:
#   1. Select the persisted Architecture-D best checkpoint.
#   2. Load model weights only.
#   3. Perform a fresh deterministic validation pass.
#   4. Distinguish genuine historical validation records from records
#      reconstructed after a Kaggle kernel restart.
#   5. Never allow a reconstructed scalar to masquerade as a genuine
#      historical validation measurement.
#   6. Save fresh validation as the authoritative validation result for
#      downstream generation/submission cells.
#
# IMPORTANT:
# Training is NOT performed in this cell.


import gc
import json
import math
from pathlib import Path

import torch


print("=" * 70)
print("ARCHITECTURE-D BEST CHECKPOINT SELECTION")
print("=" * 70)


# ======================================================================
# 1. PRECONDITIONS
# ======================================================================

assert "MODEL" in globals(), (
    "MODEL is unavailable. Run Cell 06."
)

assert "BEST_CHECKPOINT_PATH" in globals(), (
    "BEST_CHECKPOINT_PATH is unavailable. Run Cell 08."
)

assert "LATEST_CHECKPOINT_PATH" in globals(), (
    "LATEST_CHECKPOINT_PATH is unavailable. Run Cell 08."
)

assert "VALIDATION_METRICS_PATH" in globals(), (
    "VALIDATION_METRICS_PATH is unavailable. Run Cell 09."
)

assert "DATA_FINGERPRINT" in globals(), (
    "DATA_FINGERPRINT is unavailable."
)

assert "MODEL_FINGERPRINT" in globals(), (
    "MODEL_FINGERPRINT is unavailable."
)

assert "EXPERIMENT_ID" in globals(), (
    "EXPERIMENT_ID is unavailable."
)

assert "ARCHITECTURE_VERSION" in globals(), (
    "ARCHITECTURE_VERSION is unavailable."
)

assert "evaluate_validation" in globals(), (
    "evaluate_validation() is unavailable. Run Cell 09."
)


assert Path(
    BEST_CHECKPOINT_PATH
).exists(), (
    "Best checkpoint does not exist:\n"
    f"{BEST_CHECKPOINT_PATH}"
)


# ======================================================================
# 2. READ AVAILABLE VALIDATION RECORDS
# ======================================================================

validation_records = []


if Path(
    VALIDATION_METRICS_PATH
).exists():

    with Path(
        VALIDATION_METRICS_PATH
    ).open(
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            line = line.strip()

            if not line:
                continue

            try:

                record = json.loads(
                    line
                )

            except Exception as exc:

                print(
                    "⚠️ Ignoring malformed validation "
                    f"record: {exc}"
                )

                continue


            if isinstance(
                record,
                dict,
            ):

                validation_records.append(
                    record
                )


print(
    "\nValidation records available :",
    len(validation_records)
)


# ======================================================================
# 3. LOAD BEST CHECKPOINT
# ======================================================================

best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)


assert isinstance(
    best_checkpoint,
    dict,
), (
    "best.pt must contain a dictionary."
)


# ======================================================================
# 4. CHECKPOINT IDENTITY
# ======================================================================

assert (
    best_checkpoint.get(
        "experiment_id"
    )
    == EXPERIMENT_ID
), (
    "Best checkpoint experiment ID does not match "
    "the current experiment."
)


assert (
    best_checkpoint.get(
        "architecture_version"
    )
    == ARCHITECTURE_VERSION
), (
    "Best checkpoint architecture version does not match."
)


checkpoint_data_fingerprint = (
    best_checkpoint.get(
        "data_fingerprint"
    )
)


if checkpoint_data_fingerprint is not None:

    assert (
        checkpoint_data_fingerprint
        == DATA_FINGERPRINT
    ), (
        "Best checkpoint data fingerprint does not match."
    )


checkpoint_model_fingerprint = (
    best_checkpoint.get(
        "model_fingerprint"
    )
)


if checkpoint_model_fingerprint is not None:

    assert (
        checkpoint_model_fingerprint
        == MODEL_FINGERPRINT
    ), (
        "Best checkpoint model fingerprint does not match."
    )


# ======================================================================
# 5. RESOLVE PERSISTED BEST-CHECKPOINT METADATA
# ======================================================================
#
# Architecture-D checkpoint infrastructure stores progress information
# inside the checkpoint. We support the actual D schema and a few safe
# fallbacks so that recovery does not depend on a missing JSONL file.
# ======================================================================

checkpoint_progress = (
    best_checkpoint.get(
        "progress",
        {},
    )
)


if not isinstance(
    checkpoint_progress,
    dict,
):

    checkpoint_progress = {}


def first_available(
    dictionaries,
    keys,
):

    for dictionary in dictionaries:

        if not isinstance(
            dictionary,
            dict,
        ):

            continue

        for key in keys:

            value = dictionary.get(
                key
            )

            if value is not None:

                return value

    return None


persisted_best_step = first_available(
    [
        checkpoint_progress,
        best_checkpoint,
    ],
    [
        "best_validation_step",
        "optimizer_step",
        "step",
    ],
)


persisted_best_loss = first_available(
    [
        checkpoint_progress,
        best_checkpoint,
    ],
    [
        "best_validation_loss",
        "validation_loss",
    ],
)


persisted_best_ppl = first_available(
    [
        checkpoint_progress,
        best_checkpoint,
    ],
    [
        "best_validation_perplexity",
        "validation_perplexity",
    ],
)


if persisted_best_step is not None:

    persisted_best_step = int(
        persisted_best_step
    )


if persisted_best_loss is not None:

    persisted_best_loss = float(
        persisted_best_loss
    )


if persisted_best_ppl is not None:

    persisted_best_ppl = float(
        persisted_best_ppl
    )


# ======================================================================
# 6. IDENTIFY WHETHER AVAILABLE METRICS ARE RECOVERED
# ======================================================================

reconstructed_records = [

    record

    for record
    in validation_records

    if (
        record.get(
            "record_source"
        )
        == "reconstructed_from_persisted_checkpoint"
        or
        record.get(
            "resume_only"
        )
        is True
    )

]


has_genuine_historical_records = any(

    not (
        record.get(
            "record_source"
        )
        == "reconstructed_from_persisted_checkpoint"
        or
        record.get(
            "resume_only"
        )
        is True
    )

    for record
    in validation_records

)


# ======================================================================
# 7. DETERMINE SELECTION STEP
# ======================================================================
#
# Prefer the checkpoint's own persisted best-validation step.
# This is safer than selecting a step from a reconstructed one-record
# metrics file.
# ======================================================================

if persisted_best_step is not None:

    SELECTED_OPTIMIZER_STEP = (
        persisted_best_step
    )

    selection_source = (
        "persisted_best_checkpoint_metadata"
    )

elif validation_records:

    genuine_records = [

        record

        for record
        in validation_records

        if not (
            record.get(
                "record_source"
            )
            == "reconstructed_from_persisted_checkpoint"
            or
            record.get(
                "resume_only"
            )
            is True
        )

    ]


    candidate_records = (
        genuine_records
        if genuine_records
        else validation_records
    )


    best_record = min(

        candidate_records,

        key=lambda record:
            float(
                record[
                    "validation_loss"
                ]
            )

    )


    SELECTED_OPTIMIZER_STEP = int(
        best_record[
            "optimizer_step"
        ]
    )

    selection_source = (
        "available_validation_metrics"
    )

else:

    raise RuntimeError(
        "Unable to determine the optimizer step "
        "associated with best.pt."
    )


print(
    "\n" + "-" * 70
)

print(
    "CHECKPOINT SELECTION"
)

print(
    "-" * 70
)

print(
    f"Validation records        : "
    f"{len(validation_records)}"
)

print(
    f"Selected optimizer step   : "
    f"{SELECTED_OPTIMIZER_STEP:,}"
)

print(
    f"Selection source          : "
    f"{selection_source}"
)

print(
    f"Best checkpoint path      : "
    f"{BEST_CHECKPOINT_PATH}"
)


# ======================================================================
# 8. LOAD BEST MODEL WEIGHTS ONLY
# ======================================================================
#
# IMPORTANT:
# Do NOT restore optimizer, scheduler, training sampler, or RNG state.
# Cell 10 is an evaluation-only operation.
# ======================================================================

assert (
    "model_state_dict"
    in best_checkpoint
), (
    "best.pt does not contain model_state_dict."
)


MODEL.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


MODEL.to(
    DEVICE
)

MODEL.eval()


print(
    "✅ Best model weights loaded."
)

print(
    "✅ Checkpoint identity verified."
)


# ======================================================================
# 9. FRESH DETERMINISTIC VALIDATION
# ======================================================================

print(
    "\n" + "-" * 70
)

print(
    "FRESH VALIDATION OF SELECTED MODEL"
)

print(
    "-" * 70
)


final_validation_result = (
    evaluate_validation(
        optimizer_step=(
            SELECTED_OPTIMIZER_STEP
        )
    )
)


FINAL_VALIDATION_LOSS = float(
    final_validation_result[
        "validation_loss"
    ]
)


FINAL_VALIDATION_PERPLEXITY = float(
    final_validation_result[
        "validation_perplexity"
    ]
)


FINAL_VALIDATION_TOKENS = int(
    final_validation_result[
        "validation_tokens"
    ]
)


print(
    f"Validation loss       : "
    f"{FINAL_VALIDATION_LOSS:.6f}"
)

print(
    f"Validation perplexity  : "
    f"{FINAL_VALIDATION_PERPLEXITY:.4f}"
)

print(
    f"Validation tokens      : "
    f"{FINAL_VALIDATION_TOKENS:,}"
)

print(
    f"Source counts          : "
    f"{final_validation_result['source_counts']}"
)


# ======================================================================
# 10. VERIFY DETERMINISM
# ======================================================================
#
# A second fresh evaluation using the same optimizer step must reproduce
# the first fresh validation result exactly within floating-point tolerance.
#
# This checks the CURRENT validation implementation itself.
# ======================================================================

second_validation_result = (
    evaluate_validation(
        optimizer_step=(
            SELECTED_OPTIMIZER_STEP
        )
    )
)


SECOND_VALIDATION_LOSS = float(
    second_validation_result[
        "validation_loss"
    ]
)


SECOND_VALIDATION_PERPLEXITY = float(
    second_validation_result[
        "validation_perplexity"
    ]
)


assert math.isclose(
    FINAL_VALIDATION_LOSS,
    SECOND_VALIDATION_LOSS,
    rel_tol=1e-7,
    abs_tol=1e-7,
), (
    "Repeated fresh validation does not reproduce "
    "the same loss. The CURRENT validation implementation "
    "is not deterministic."
)


assert math.isclose(
    FINAL_VALIDATION_PERPLEXITY,
    SECOND_VALIDATION_PERPLEXITY,
    rel_tol=1e-7,
    abs_tol=1e-7,
), (
    "Repeated fresh validation does not reproduce "
    "the same perplexity."
)


print(
    "✅ Current validation implementation is deterministic."
)


# ======================================================================
# 11. HANDLE RECOVERED-HISTORY DISCREPANCY CORRECTLY
# ======================================================================

if reconstructed_records:

    reconstructed_loss = None
    reconstructed_step = None


    for record in reconstructed_records:

        record_step = record.get(
            "optimizer_step"
        )


        if (
            record_step is None
            or
            int(record_step)
            != SELECTED_OPTIMIZER_STEP
        ):

            continue


        if (
            record.get(
                "validation_loss"
            )
            is not None
        ):

            reconstructed_loss = float(
                record[
                    "validation_loss"
                ]
            )


        reconstructed_step = (
            SELECTED_OPTIMIZER_STEP
        )

        break


    if reconstructed_loss is not None:

        print(
            "\n⚠️ Recovered validation metadata "
            "does not exactly match the fresh validation."
        )

        print(
            f"   Recovered checkpoint metric : "
            f"{reconstructed_loss:.6f}"
        )

        print(
            f"   Fresh validation metric     : "
            f"{FINAL_VALIDATION_LOSS:.6f}"
        )

        print(
            "   Fresh validation is now the "
            "authoritative validation result."
        )


    validation_history_source = (
        "fresh_revalidation_after_checkpoint_recovery"
    )

else:

    # -------------------------------------------------------------
    # Genuine historical validation is available.
    # -------------------------------------------------------------

    historical_match = None


    for record in validation_records:

        record_step = record.get(
            "optimizer_step"
        )


        if (
            record_step is None
            or
            int(record_step)
            != SELECTED_OPTIMIZER_STEP
        ):

            continue


        if (
            record.get(
                "validation_loss"
            )
            is not None
        ):

            historical_match = record

            break


    if historical_match is not None:

        historical_loss = float(
            historical_match[
                "validation_loss"
            ]
        )


        historical_ppl = float(
            historical_match[
                "validation_perplexity"
            ]
        )


        assert math.isclose(
            FINAL_VALIDATION_LOSS,
            historical_loss,
            rel_tol=1e-7,
            abs_tol=1e-7,
        ), (
            "Fresh validation does not reproduce the "
            "genuine historical validation loss."
        )


        assert math.isclose(
            FINAL_VALIDATION_PERPLEXITY,
            historical_ppl,
            rel_tol=1e-7,
            abs_tol=1e-7,
        ), (
            "Fresh validation does not reproduce the "
            "genuine historical validation perplexity."
        )


        print(
            "✅ Fresh validation reproduces "
            "the genuine historical result."
        )


    validation_history_source = (
        "genuine_historical_validation"
    )


# ======================================================================
# 12. OVERWRITE RECOVERED VALIDATION METRICS WITH AUTHORITATIVE RESULT
# ======================================================================
#
# In recovery mode the old JSONL contained only a reconstructed scalar.
# Replace it with the fresh deterministic result so that future Cell-10
# executions do not treat the stale reconstructed value as authoritative.
#
# In genuine-history mode the original history is preserved.
# ======================================================================

if reconstructed_records:

    VALIDATION_METRICS_RECOVERY_RECORD = {

        "optimizer_step":
            int(
                SELECTED_OPTIMIZER_STEP
            ),

        "validation_loss":
            float(
                FINAL_VALIDATION_LOSS
            ),

        "validation_perplexity":
            float(
                FINAL_VALIDATION_PERPLEXITY
            ),

        "validation_tokens":
            int(
                FINAL_VALIDATION_TOKENS
            ),

        "source_counts":
            dict(
                final_validation_result[
                    "source_counts"
                ]
            ),

        "validation_seed":
            int(
                SEED
                + 100000
                + SELECTED_OPTIMIZER_STEP
            ),

        "record_source":
            (
                "fresh_revalidation_after_"
                "checkpoint_recovery"
            ),

        "checkpoint_recorded_validation_loss":
            (
                persisted_best_loss
            ),

        "checkpoint_recorded_validation_perplexity":
            (
                persisted_best_ppl
            ),

        "checkpoint_recorded_best_step":
            (
                persisted_best_step
            ),

        "resume_only":
            True,

    }


    with Path(
        VALIDATION_METRICS_PATH
    ).open(
        "w",
        encoding="utf-8",
    ) as file:

        file.write(
            json.dumps(
                VALIDATION_METRICS_RECOVERY_RECORD,
                ensure_ascii=False,
            )
            + "\n"
        )


    print(
        "✅ Recovered validation metrics replaced "
        "with the fresh authoritative validation result."
    )


# ======================================================================
# 13. SAVE SELECTION METADATA
# ======================================================================

SELECTED_CHECKPOINT_METADATA = {

    "status":
        "SELECTED",

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "selected_checkpoint":
        str(
            BEST_CHECKPOINT_PATH
        ),

    "latest_checkpoint":
        str(
            LATEST_CHECKPOINT_PATH
        ),

    "selected_optimizer_step":
        int(
            SELECTED_OPTIMIZER_STEP
        ),

    "checkpoint_recorded_validation_loss":
        (
            persisted_best_loss
        ),

    "checkpoint_recorded_validation_perplexity":
        (
            persisted_best_ppl
        ),

    "checkpoint_recorded_best_step":
        (
            persisted_best_step
        ),

    "fresh_validation_loss":
        (
            FINAL_VALIDATION_LOSS
        ),

    "fresh_validation_perplexity":
        (
            FINAL_VALIDATION_PERPLEXITY
        ),

    "fresh_validation_tokens":
        (
            FINAL_VALIDATION_TOKENS
        ),

    "validation_history_source":
        validation_history_source,

    "model_fingerprint":
        MODEL_FINGERPRINT,

    "data_fingerprint":
        DATA_FINGERPRINT,

    "selection_rule":
        (
            "Persisted Architecture-D best.pt checkpoint "
            "selected using its persisted best-validation "
            "step metadata; fresh deterministic validation "
            "is the authoritative current metric."
        ),
}


SELECTED_CHECKPOINT_METADATA_PATH = (
    CONFIGS_DIR
    / "selected_checkpoint_D.json"
)


with SELECTED_CHECKPOINT_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        SELECTED_CHECKPOINT_METADATA,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ======================================================================
# 14. UPDATE RUNTIME CONFIGURATION
# ======================================================================

if (
    "RUNTIME_CONFIGURATION"
    in globals()
    and
    "RUNTIME_CONFIGURATION_PATH"
    in globals()
):

    RUNTIME_CONFIGURATION[
        "selected_checkpoint"
    ] = SELECTED_CHECKPOINT_METADATA


    with RUNTIME_CONFIGURATION_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            RUNTIME_CONFIGURATION,
            file,
            indent=2,
            ensure_ascii=False,
        )


    print(
        "✅ Runtime configuration updated."
    )


# ======================================================================
# 15. CLEANUP
# ======================================================================

del best_checkpoint
del second_validation_result
del final_validation_result

gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ======================================================================
# 16. FINAL SAFETY GATE
# ======================================================================

assert (
    MODEL.training is False
)


assert (
    SELECTED_CHECKPOINT_METADATA_PATH.exists()
)


assert math.isfinite(
    FINAL_VALIDATION_LOSS
)


assert math.isfinite(
    FINAL_VALIDATION_PERPLEXITY
)


# ======================================================================
# 17. FINAL OUTPUT
# ======================================================================

print(
    "\n" + "=" * 70
)

print(
    "✅ CELL 10 COMPLETE — "
    "ARCHITECTURE-D BEST CHECKPOINT VALIDATED"
)

print(
    "=" * 70
)

print(
    f"Selected checkpoint       : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"Selected optimizer step   : "
    f"{SELECTED_OPTIMIZER_STEP:,}"
)

print(
    f"Fresh validation loss     : "
    f"{FINAL_VALIDATION_LOSS:.6f}"
)

print(
    f"Fresh validation PPL      : "
    f"{FINAL_VALIDATION_PERPLEXITY:.4f}"
)

print(
    f"Fresh validation tokens   : "
    f"{FINAL_VALIDATION_TOKENS:,}"
)

print(
    f"Validation source         : "
    f"{validation_history_source}"
)

print(
    f"Selection metadata        : "
    f"{SELECTED_CHECKPOINT_METADATA_PATH}"
)


if reconstructed_records:

    print(
        "\n⚠️ This notebook resumed from a persisted "
        "checkpoint without the original JSONL history."
    )

    print(
        "✅ The fresh deterministic validation result "
        "has been promoted to the authoritative metric."
    )


print(
    "\n✅ No training update was performed in Cell 10."
)

print(
    "✅ Best checkpoint is ready for downstream generation."
)

print(
    "=" * 70
)

ARCHITECTURE-D BEST CHECKPOINT SELECTION

Validation records available : 1

----------------------------------------------------------------------
CHECKPOINT SELECTION
----------------------------------------------------------------------
Validation records        : 1
Selected optimizer step   : 6,104
Selection source          : available_validation_metrics
Best checkpoint path      : /kaggle/working/architecture_D/checkpoints/best.pt
✅ Best model weights loaded.
✅ Checkpoint identity verified.

----------------------------------------------------------------------
FRESH VALIDATION OF SELECTED MODEL
----------------------------------------------------------------------
Validation loss       : 3.874692
Validation perplexity  : 48.1679
Validation tokens      : 2,048,000
Source counts          : {'acts': 253, 'high_courts': 256, 'supreme_court': 249, 'tribunals': 242}
✅ Current validation implementation is deterministic.

⚠️ Recovered validation metadata does not exactly match the fresh v

In [11]:
# Cell 11 — Architecture-D standalone official Phase-1 test-data loader
# DECISION: REPLACE
#
# This cell is completely independent of Architectures A, B, and C.
#
# It does NOT:
#   - reuse another architecture's test file
#   - inspect another architecture's directory
#   - depend on another architecture's variables
#
# It only:
#   1. Reuses Architecture-D's own verified local test copy, if present.
#   2. Otherwise downloads the official Phase-1 test file directly from
#      the official Hugging Face dataset.
#   3. Validates the official test structure.
#   4. Stores a canonical copy inside Architecture-D.
#   5. Creates ACTIVE_TEST_RECORDS for later generation cells.
#   6. Saves an Architecture-D-specific official-test manifest.
#
# Official source:
#   Repository:
#       Exploration-Lab/CS-781-Capstone
#
#   File:
#       Phase_1_Pretraining/Test/test.json
#
# Fast mode:
#   Active examples = 2
#
# Production mode:
#   Active examples = 1000
#
# No model training occurs in this cell.
# No text generation occurs in this cell.


import json
import shutil
from pathlib import Path


print("=" * 70)
print("ARCHITECTURE-D STANDALONE OFFICIAL TEST DATA")
print("=" * 70)


# =========================================================================
# 1. PRECONDITIONS
# =========================================================================

assert "RUN_MODE" in globals(), (
    "RUN_MODE is unavailable. Run Cell 01 first."
)

assert RUN_MODE in {
    "fast",
    "production",
}, (
    f"Invalid RUN_MODE: {RUN_MODE}"
)

assert "EXPERIMENT_ID" in globals(), (
    "EXPERIMENT_ID is unavailable. Run Cell 01 first."
)

assert "ARCHITECTURE_NAME" in globals(), (
    "ARCHITECTURE_NAME is unavailable. Run Cell 01 first."
)

assert "ARCHITECTURE_VERSION" in globals(), (
    "ARCHITECTURE_VERSION is unavailable. Run Cell 01 first."
)

assert "TEST_DATA_DIR" in globals(), (
    "TEST_DATA_DIR is unavailable. Run Cell 01 first."
)

assert "CONFIGS_DIR" in globals(), (
    "CONFIGS_DIR is unavailable. Run Cell 01 first."
)

assert "RUNTIME_CONFIGURATION_PATH" in globals(), (
    "RUNTIME_CONFIGURATION_PATH is unavailable. Run Cell 01 first."
)


# =========================================================================
# 2. ARCHITECTURE-D-ONLY PATHS
# =========================================================================
#
# These paths exist only inside Architecture-D.

TEST_DATA_DIR = Path(
    TEST_DATA_DIR
)

CONFIGS_DIR = Path(
    CONFIGS_DIR
)


TEST_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CONFIGS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# The canonical Architecture-D copy.
D_CANONICAL_TEST_PATH = (
    TEST_DATA_DIR
    / "official_phase1_test.json"
)


# Temporary Hugging Face download/cache directory.
D_HF_DOWNLOAD_DIR = (
    TEST_DATA_DIR
    / "hf_download"
)


# Architecture-D test manifest.
OFFICIAL_TEST_MANIFEST_PATH = (
    CONFIGS_DIR
    / "official_test_manifest_D.json"
)


# =========================================================================
# 3. OFFICIAL HUGGING FACE SOURCE
# =========================================================================

HF_REPOSITORY_ID = (
    "Exploration-Lab/CS-781-Capstone"
)

HF_TEST_FILE = (
    "Phase_1_Pretraining/Test/test.json"
)


print("\n" + "-" * 70)
print("OFFICIAL TEST SOURCE")
print("-" * 70)

print(
    f"Hugging Face repository : "
    f"{HF_REPOSITORY_ID}"
)

print(
    f"Official test file      : "
    f"{HF_TEST_FILE}"
)

print(
    "Architecture dependency : NONE"
)


# =========================================================================
# 4. NORMALIZE OFFICIAL TEST JSON
# =========================================================================
#
# Official/reference structure:
#
# {
#     "1": {
#         "text": "..."
#     },
#     "2": {
#         "text": "..."
#     }
# }
#
# We convert this to:
#
# [
#     {
#         "id": "1",
#         "prompt": "..."
#     },
#     ...
# ]
#
# The raw official wording is not modified.


def normalize_official_test_json(
    raw_data,
):

    records = []


    # ---------------------------------------------------------------------
    # Case A:
    # Top-level ID -> {"text": prompt}
    # ---------------------------------------------------------------------

    if isinstance(
        raw_data,
        dict,
    ):

        all_values_are_records = True


        for value in raw_data.values():

            if not isinstance(
                value,
                dict,
            ):

                all_values_are_records = False

                break


        if all_values_are_records:

            for test_id, value in raw_data.items():

                prompt = None


                for key in (
                    "text",
                    "prompt",
                    "input",
                ):

                    if key in value:

                        prompt = value[
                            key
                        ]

                        break


                if prompt is None:

                    raise ValueError(
                        f"Test ID {test_id} "
                        "does not contain a prompt field."
                    )


                if not isinstance(
                    prompt,
                    str,
                ):

                    prompt = str(
                        prompt
                    )


                records.append(
                    {
                        "id": str(
                            test_id
                        ),
                        "prompt": prompt,
                    }
                )


            return records


        # -----------------------------------------------------------------
        # Case B:
        # Wrapped list representation.
        # -----------------------------------------------------------------

        for wrapper_key in (
            "data",
            "records",
            "examples",
            "items",
            "test",
        ):

            if wrapper_key not in raw_data:

                continue


            wrapped_data = raw_data[
                wrapper_key
            ]


            if isinstance(
                wrapped_data,
                list,
            ):

                return normalize_official_test_json(
                    wrapped_data
                )


    # ---------------------------------------------------------------------
    # Case C:
    # List of records.
    # ---------------------------------------------------------------------

    if isinstance(
        raw_data,
        list,
    ):

        for index, value in enumerate(
            raw_data
        ):

            if not isinstance(
                value,
                dict,
            ):

                raise ValueError(
                    f"Test record {index} "
                    "is not a dictionary."
                )


            test_id = None


            for key in (
                "id",
                "ID",
                "test_id",
                "testId",
            ):

                if key in value:

                    test_id = str(
                        value[
                            key
                        ]
                    )

                    break


            if test_id is None:

                raise ValueError(
                    f"Test record {index} "
                    "does not contain an ID."
                )


            prompt = None


            for key in (
                "prompt",
                "text",
                "input",
            ):

                if key in value:

                    prompt = value[
                        key
                    ]

                    break


            if prompt is None:

                raise ValueError(
                    f"Test ID {test_id} "
                    "does not contain a prompt."
                )


            if not isinstance(
                prompt,
                str,
            ):

                prompt = str(
                    prompt
                )


            records.append(
                {
                    "id": test_id,
                    "prompt": prompt,
                }
            )


        return records


    raise ValueError(
        "Unsupported official test JSON format."
    )


# =========================================================================
# 5. STRICT TEST-DATA VALIDATOR
# =========================================================================
#
# Official Phase-1 test requirements:
#   - 1000 examples
#   - unique IDs
#   - every example has prompt text
#   - every prompt has exactly 512 whitespace-separated words


def validate_official_test_records(
    records,
):

    if not isinstance(
        records,
        list,
    ):

        raise AssertionError(
            "Normalized test data is not a list."
        )


    if len(
        records
    ) != 1000:

        raise AssertionError(
            "Expected exactly 1000 official Phase-1 "
            f"test examples, found {len(records)}."
        )


    ids = []


    for index, record in enumerate(
        records
    ):

        assert isinstance(
            record,
            dict,
        ), (
            f"Record {index} is not a dictionary."
        )


        assert "id" in record, (
            f"Record {index} has no ID."
        )


        assert "prompt" in record, (
            f"Record {index} has no prompt."
        )


        test_id = str(
            record[
                "id"
            ]
        )


        prompt = record[
            "prompt"
        ]


        assert test_id.strip(), (
            f"Record {index} has an empty ID."
        )


        assert isinstance(
            prompt,
            str,
        ), (
            f"Test ID {test_id} prompt is not a string."
        )


        assert prompt.strip(), (
            f"Test ID {test_id} has an empty prompt."
        )


        word_count = len(
            prompt.strip().split()
        )


        assert word_count == 512, (
            f"Test ID {test_id}: expected "
            f"512 prompt words, found "
            f"{word_count}."
        )


        ids.append(
            test_id
        )


    assert len(
        ids
    ) == len(
        set(ids)
    ), (
        "Duplicate official test IDs detected."
    )


    assert len(
        ids
    ) == 1000


    return True


# =========================================================================
# 6. LOAD ONE CANDIDATE TEST FILE
# =========================================================================

def load_test_candidate(
    candidate_path,
):

    candidate_path = Path(
        candidate_path
    )


    if not candidate_path.is_file():

        return None


    try:

        with candidate_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            raw_data = json.load(
                file
            )


        records = (
            normalize_official_test_json(
                raw_data
            )
        )


        validate_official_test_records(
            records
        )


        return records


    except Exception:

        return None


# =========================================================================
# 7. ARCHITECTURE-D LOCAL COPY CHECK
# =========================================================================
#
# Important:
# We only check Architecture-D's own file.
# No Architecture-A/B/C path is touched.


OFFICIAL_TEST_RECORDS = None
OFFICIAL_TEST_SOURCE = None


if D_CANONICAL_TEST_PATH.is_file():

    print(
        "\n🔎 Checking Architecture-D's "
        "existing canonical test copy..."
    )


    local_records = load_test_candidate(
        D_CANONICAL_TEST_PATH
    )


    if local_records is not None:

        OFFICIAL_TEST_RECORDS = (
            local_records
        )

        OFFICIAL_TEST_SOURCE = (
            "architecture_D_local_copy"
        )


        print(
            "✅ Architecture-D's own verified "
            "test copy is valid."
        )

        print(
            f"   {D_CANONICAL_TEST_PATH}"
        )

    else:

        print(
            "⚠️ Existing Architecture-D test copy "
            "is invalid and will be replaced."
        )


# =========================================================================
# 8. DIRECT DOWNLOAD FROM HUGGING FACE
# =========================================================================

if OFFICIAL_TEST_RECORDS is None:

    print(
        "\n🌐 Downloading official Phase-1 "
        "test data from Hugging Face..."
    )


    try:

        from huggingface_hub import (
            hf_hub_download
        )

        from kaggle_secrets import (
            UserSecretsClient
        )


        secrets_client = (
            UserSecretsClient()
        )


        hf_token = (
            secrets_client.get_secret(
                "HF"
            )
        )


        assert isinstance(
            hf_token,
            str,
        ), (
            "Kaggle Secret 'HF' did not return a string."
        )


        assert hf_token.strip(), (
            "Kaggle Secret 'HF' is empty."
        )


        D_HF_DOWNLOAD_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )


        downloaded_file = hf_hub_download(
            repo_id=HF_REPOSITORY_ID,

            filename=HF_TEST_FILE,

            repo_type="dataset",

            token=hf_token,

            local_dir=str(
                D_HF_DOWNLOAD_DIR
            ),
        )


        downloaded_file = Path(
            downloaded_file
        )


        assert downloaded_file.is_file(), (
            "Hugging Face download completed but "
            "the resulting file was not found."
        )


        downloaded_records = (
            load_test_candidate(
                downloaded_file
            )
        )


        assert downloaded_records is not None, (
            "The downloaded official Phase-1 test "
            "file failed validation."
        )


        OFFICIAL_TEST_RECORDS = (
            downloaded_records
        )

        OFFICIAL_TEST_SOURCE = (
            "official_huggingface_download"
        )


        print(
            "✅ Official Phase-1 test file "
            "downloaded and validated."
        )


        print(
            f"   Downloaded path:"
        )

        print(
            f"   {downloaded_file}"
        )


    except Exception as exc:

        raise RuntimeError(
            "Unable to download the official "
            "Phase-1 test file from Hugging Face.\n\n"
            f"Repository : {HF_REPOSITORY_ID}\n"
            f"File       : {HF_TEST_FILE}\n"
            f"Error      : {exc}"
        ) from exc


# =========================================================================
# 9. FINAL VALIDATION
# =========================================================================

assert OFFICIAL_TEST_RECORDS is not None, (
    "Official Phase-1 test data could not be loaded."
)


validate_official_test_records(
    OFFICIAL_TEST_RECORDS
)


# =========================================================================
# 10. WRITE ARCHITECTURE-D CANONICAL COPY
# =========================================================================
#
# Always keep D's own local canonical file synchronized with the validated
# official records.


canonical_temp_path = (
    D_CANONICAL_TEST_PATH.with_suffix(
        ".json.tmp"
    )
)


with canonical_temp_path.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        {
            str(record["id"]): {
                "text": str(
                    record["prompt"]
                )
            }
            for record in OFFICIAL_TEST_RECORDS
        },
        file,
        ensure_ascii=False,
        indent=2,
    )


canonical_temp_path.replace(
    D_CANONICAL_TEST_PATH
)


# Re-read and validate the canonical D copy.
canonical_records = load_test_candidate(
    D_CANONICAL_TEST_PATH
)


assert canonical_records is not None, (
    "Architecture-D canonical test copy "
    "failed its final validation."
)


OFFICIAL_TEST_RECORDS = canonical_records


# =========================================================================
# 11. OFFICIAL IDS / PROMPTS
# =========================================================================

OFFICIAL_TEST_IDS = [
    str(
        record["id"]
    )
    for record in OFFICIAL_TEST_RECORDS
]


OFFICIAL_TEST_PROMPTS = [
    str(
        record["prompt"]
    )
    for record in OFFICIAL_TEST_RECORDS
]


assert len(
    OFFICIAL_TEST_IDS
) == 1000


assert len(
    set(
        OFFICIAL_TEST_IDS
    )
) == 1000


assert len(
    OFFICIAL_TEST_PROMPTS
) == 1000


assert all(
    len(
        prompt.strip().split()
    ) == 512
    for prompt in OFFICIAL_TEST_PROMPTS
)


# =========================================================================
# 12. ACTIVE TEST SET
# =========================================================================
#
# Fast:
#   exactly the first 2 official examples.
#
# Production:
#   all 1000 official examples.
#
# The implementation itself is identical in both modes.


ACTIVE_TEST_COUNT = (
    2
    if RUN_MODE == "fast"
    else 1000
)


ACTIVE_TEST_RECORDS = (
    OFFICIAL_TEST_RECORDS[
        :ACTIVE_TEST_COUNT
    ]
)


ACTIVE_TEST_IDS = [
    str(
        record["id"]
    )
    for record in ACTIVE_TEST_RECORDS
]


ACTIVE_TEST_PROMPTS = [
    str(
        record["prompt"]
    )
    for record in ACTIVE_TEST_RECORDS
]


assert len(
    ACTIVE_TEST_RECORDS
) == ACTIVE_TEST_COUNT


assert len(
    ACTIVE_TEST_IDS
) == ACTIVE_TEST_COUNT


assert len(
    set(
        ACTIVE_TEST_IDS
    )
) == ACTIVE_TEST_COUNT


assert all(
    len(
        prompt.strip().split()
    ) == 512
    for prompt in ACTIVE_TEST_PROMPTS
)


# =========================================================================
# 13. GENERATION TARGETS
# =========================================================================
#
# These are defined here as the official generation-stage configuration.
# They do not alter the model or training implementation.


GENERATION_TARGET_WORDS = (
    16
    if RUN_MODE == "fast"
    else 128
)


GENERATION_MAX_NEW_TOKENS = (
    512
    if RUN_MODE == "fast"
    else 4096
)


assert (
    GENERATION_TARGET_WORDS > 0
)


assert (
    GENERATION_MAX_NEW_TOKENS
    >= GENERATION_TARGET_WORDS
)


# =========================================================================
# 14. SAVE TEST MANIFEST
# =========================================================================

OFFICIAL_TEST_MANIFEST = {

    "status": "validated",

    "project": "CS781 Phase 1",

    "experiment_id": EXPERIMENT_ID,

    "architecture_name": ARCHITECTURE_NAME,

    "architecture_version": ARCHITECTURE_VERSION,

    "run_mode": RUN_MODE,

    "source_type": OFFICIAL_TEST_SOURCE,

    "huggingface_repository": (
        HF_REPOSITORY_ID
    ),

    "huggingface_file": (
        HF_TEST_FILE
    ),

    "canonical_local_path": (
        str(
            D_CANONICAL_TEST_PATH
        )
    ),

    "total_official_examples": 1000,

    "prompt_words": 512,

    "active_test_count": (
        ACTIVE_TEST_COUNT
    ),

    "active_test_ids": list(
        ACTIVE_TEST_IDS
    ),

    "generation_target_words": (
        GENERATION_TARGET_WORDS
    ),

    "generation_max_new_tokens": (
        GENERATION_MAX_NEW_TOKENS
    ),

    "validation": {

        "exactly_1000_examples": True,

        "unique_ids": True,

        "prompts_present": True,

        "every_prompt_exactly_512_words": True,

        "architecture_D_local_copy_verified": True,
    },
}


with OFFICIAL_TEST_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        OFFICIAL_TEST_MANIFEST,
        file,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# 15. UPDATE RUNTIME CONFIGURATION
# =========================================================================

if "RUNTIME_CONFIGURATION" in globals():

    RUNTIME_CONFIGURATION[
        "official_test"
    ] = OFFICIAL_TEST_MANIFEST


    RUNTIME_CONFIGURATION[
        "generation"
    ] = {

        "target_words": (
            GENERATION_TARGET_WORDS
        ),

        "max_new_tokens": (
            GENERATION_MAX_NEW_TOKENS
        ),

        "active_test_count": (
            ACTIVE_TEST_COUNT
        ),
    }


    with Path(
        RUNTIME_CONFIGURATION_PATH
    ).open(
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            RUNTIME_CONFIGURATION,
            file,
            ensure_ascii=False,
            indent=2,
        )


# =========================================================================
# 16. FINAL OUTPUT
# =========================================================================

print(
    "\n"
    + "=" * 70
)

print(
    "✅ CELL 11 COMPLETE — OFFICIAL TEST DATA READY"
)

print(
    "=" * 70
)

print(
    f"Source                    : "
    f"{OFFICIAL_TEST_SOURCE}"
)

print(
    f"Hugging Face repository   : "
    f"{HF_REPOSITORY_ID}"
)

print(
    f"Official file             : "
    f"{HF_TEST_FILE}"
)

print(
    f"Architecture-D copy      : "
    f"{D_CANONICAL_TEST_PATH}"
)

print(
    f"Official examples         : "
    f"{len(OFFICIAL_TEST_RECORDS):,}"
)

print(
    f"Prompt words              : "
    f"512"
)

print(
    f"Active examples           : "
    f"{ACTIVE_TEST_COUNT:,}"
)

print(
    "Active IDs                : "
    + ", ".join(
        ACTIVE_TEST_IDS
    )
)

print(
    f"Generation target        : "
    f"{GENERATION_TARGET_WORDS} words"
)

print(
    f"Maximum generation tokens: "
    f"{GENERATION_MAX_NEW_TOKENS}"
)

print(
    f"Test manifest             : "
    f"{OFFICIAL_TEST_MANIFEST_PATH}"
)

print(
    "\n✅ Architecture D is using only its own test-data artifact."
)

print(
    "✅ No Architecture A/B/C files were accessed."
)

print(
    "✅ Official test IDs are unique."
)

print(
    "✅ Every official prompt contains exactly 512 words."
)

print(
    "✅ Fast/production differ only in generation budget."
)

print(
    "=" * 70
)

ARCHITECTURE-D STANDALONE OFFICIAL TEST DATA

----------------------------------------------------------------------
OFFICIAL TEST SOURCE
----------------------------------------------------------------------
Hugging Face repository : Exploration-Lab/CS-781-Capstone
Official test file      : Phase_1_Pretraining/Test/test.json
Architecture dependency : NONE

🌐 Downloading official Phase-1 test data from Hugging Face...


test.json: 0.00B [00:00, ?B/s]

✅ Official Phase-1 test file downloaded and validated.
   Downloaded path:
   /kaggle/working/architecture_D/test_data/hf_download/Phase_1_Pretraining/Test/test.json

✅ CELL 11 COMPLETE — OFFICIAL TEST DATA READY
Source                    : official_huggingface_download
Hugging Face repository   : Exploration-Lab/CS-781-Capstone
Official file             : Phase_1_Pretraining/Test/test.json
Architecture-D copy      : /kaggle/working/architecture_D/test_data/official_phase1_test.json
Official examples         : 1,000
Prompt words              : 512
Active examples           : 1,000
Active IDs                : 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 9

In [12]:
# Cell 12 — Architecture-D TCN autoregressive generation engine
# DECISION: ADD
#
# This cell creates the complete Architecture-D generation function.
#
# It is fully standalone and depends only on Architecture-D variables.
#
# Features:
#   1. Exact autoregressive TCN generation
#   2. Uses Architecture-D's selected best checkpoint already loaded
#   3. Temperature sampling
#   4. Top-k filtering
#   5. Top-p filtering
#   6. Repetition penalty support
#   7. Structural-token blocking
#   8. Word-boundary rescue
#   9. Exact whitespace-separated word count
#  10. Context-window handling
#  11. Stable per-example generation seed
#  12. Same implementation for fast and production
#
# Fast mode changes only:
#   - target words
#   - maximum generated tokens
#
# Production mode changes only:
#   - target words
#   - maximum generated tokens
#
# No training occurs in this cell.
# No predictions are written in this cell.


import hashlib
import math
import time

import torch
import torch.nn.functional as F


print("=" * 70)
print("ARCHITECTURE-D TCN GENERATION ENGINE")
print("=" * 70)


# =========================================================================
# 1. PRECONDITIONS
# =========================================================================

assert "MODEL" in globals(), (
    "MODEL is unavailable. Run Cell 10 first."
)

assert "OFFICIAL_TEST_RECORDS" in globals(), (
    "OFFICIAL_TEST_RECORDS is unavailable. Run Cell 11 first."
)

assert "ACTIVE_TEST_RECORDS" in globals(), (
    "ACTIVE_TEST_RECORDS is unavailable. Run Cell 11 first."
)

assert "TOKENIZER_PATH" in globals(), (
    "TOKENIZER_PATH is unavailable. Run Cell 03 first."
)

assert "VOCAB_SIZE" in globals(), (
    "VOCAB_SIZE is unavailable. Run Cell 03 first."
)

assert "PAD_ID" in globals(), (
    "PAD_ID is unavailable. Run Cell 03 first."
)

assert "UNK_ID" in globals(), (
    "UNK_ID is unavailable. Run Cell 03 first."
)

assert "BOS_ID" in globals(), (
    "BOS_ID is unavailable. Run Cell 03 first."
)

assert "EOS_ID" in globals(), (
    "EOS_ID is unavailable. Run Cell 03 first."
)

assert "CONTEXT_LENGTH" in globals(), (
    "CONTEXT_LENGTH is unavailable. Run Cell 04 first."
)

assert "DEVICE" in globals(), (
    "DEVICE is unavailable. Run Cell 02 first."
)

assert "MODEL_FINGERPRINT" in globals(), (
    "MODEL_FINGERPRINT is unavailable. Run Cell 10 first."
)

assert "RUN_MODE" in globals(), (
    "RUN_MODE is unavailable. Run Cell 01 first."
)

assert "EXPERIMENT_ID" in globals(), (
    "EXPERIMENT_ID is unavailable. Run Cell 01 first."
)

assert "ARCHITECTURE_NAME" in globals(), (
    "ARCHITECTURE_NAME is unavailable. Run Cell 01 first."
)

assert "ARCHITECTURE_VERSION" in globals(), (
    "ARCHITECTURE_VERSION is unavailable. Run Cell 01 first."
)

assert "SEED" in globals(), (
    "SEED is unavailable. Run Cell 01 first."
)


# =========================================================================
# 2. GENERATION CONFIGURATION
# =========================================================================
#
# Same implementation in both modes.
# Only the budgets differ.

TARGET_WORDS_D = (
    16
    if RUN_MODE == "fast"
    else 128
)

MAX_NEW_TOKENS_D = (
    512
    if RUN_MODE == "fast"
    else 4096
)

TEMPERATURE_D = 0.80
TOP_K_D = 50
TOP_P_D = 0.95
REPETITION_PENALTY_D = 1.0

WORD_RESCUE_TOP_K_D = 100


assert (
    TARGET_WORDS_D > 0
)

assert (
    MAX_NEW_TOKENS_D >= TARGET_WORDS_D
)

assert (
    TEMPERATURE_D > 0.0
)

assert (
    0.0 < TOP_P_D <= 1.0
)

assert (
    TOP_K_D > 0
)

assert (
    REPETITION_PENALTY_D >= 1.0
)


print("\n" + "-" * 70)
print("GENERATION CONFIGURATION")
print("-" * 70)

print(
    f"Run mode               : "
    f"{RUN_MODE}"
)

print(
    f"Target words            : "
    f"{TARGET_WORDS_D}"
)

print(
    f"Maximum new tokens      : "
    f"{MAX_NEW_TOKENS_D}"
)

print(
    f"Temperature             : "
    f"{TEMPERATURE_D}"
)

print(
    f"Top-k                   : "
    f"{TOP_K_D}"
)

print(
    f"Top-p                   : "
    f"{TOP_P_D}"
)

print(
    f"Repetition penalty     : "
    f"{REPETITION_PENALTY_D}"
)

print(
    f"Word-rescue candidates  : "
    f"{WORD_RESCUE_TOP_K_D}"
)

print(
    f"TCN context length      : "
    f"{CONTEXT_LENGTH}"
)


# =========================================================================
# 3. LOAD TOKENIZER
# =========================================================================

from tokenizers import Tokenizer


TOKENIZER_D = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)


assert (
    TOKENIZER_D.get_vocab_size()
    == VOCAB_SIZE
), (
    "Architecture-D tokenizer vocabulary does not "
    "match the configured vocabulary size."
)


print("\n" + "-" * 70)
print("TOKENIZER")
print("-" * 70)

print(
    f"Vocabulary size         : "
    f"{TOKENIZER_D.get_vocab_size():,}"
)

print(
    "✅ Architecture-D tokenizer loaded."
)


# =========================================================================
# 4. TEXT / WORD HELPERS
# =========================================================================

def count_words_D(
    text,
):

    return len(
        str(
            text
        ).strip().split()
    )


def decode_generated_tokens_D(
    token_ids,
):

    return TOKENIZER_D.decode(
        [
            int(token_id)
            for token_id in token_ids
        ],
        skip_special_tokens=True,
    ).strip()


def truncate_to_exact_words_D(
    text,
    target_words,
):

    words = (
        str(
            text
        ).strip().split()
    )


    if len(words) < target_words:

        return None


    return " ".join(
        words[
            :target_words
        ]
    )


# =========================================================================
# 5. STABLE PER-EXAMPLE SEED
# =========================================================================
#
# The seed is derived from:
#   experiment
#   architecture
#   architecture version
#   test ID
#
# Therefore each official example has a deterministic independent stream.


def stable_generation_seed_D(
    test_id,
):

    digest = hashlib.sha256(
        (
            f"{EXPERIMENT_ID}|"
            f"{ARCHITECTURE_NAME}|"
            f"{ARCHITECTURE_VERSION}|"
            f"{test_id}"
        ).encode(
            "utf-8"
        )
    ).digest()


    return (
        int.from_bytes(
            digest[:8],
            byteorder="big",
            signed=False,
        )
        % (
            2**31 - 1
        )
    )


# =========================================================================
# 6. REPETITION PENALTY
# =========================================================================

def apply_repetition_penalty_D(
    logits,
    generated_ids,
    repetition_penalty,
):

    if (
        repetition_penalty
        == 1.0
        or
        not generated_ids
    ):

        return logits


    adjusted_logits = (
        logits.clone()
    )


    unique_generated_ids = set(
        int(token_id)
        for token_id
        in generated_ids
    )


    for token_id in unique_generated_ids:

        if not (
            0 <= token_id < adjusted_logits.numel()
        ):

            continue


        token_logit = adjusted_logits[
            token_id
        ]


        if token_logit < 0:

            adjusted_logits[
                token_id
            ] = (
                token_logit
                * repetition_penalty
            )

        else:

            adjusted_logits[
                token_id
            ] = (
                token_logit
                / repetition_penalty
            )


    return adjusted_logits


# =========================================================================
# 7. STRUCTURAL TOKEN BLOCKING
# =========================================================================

STRUCTURAL_TOKEN_IDS_D = {
    int(PAD_ID),
    int(UNK_ID),
    int(BOS_ID),
    int(EOS_ID),
}


STRUCTURAL_TOKEN_IDS_D = {
    token_id
    for token_id
    in STRUCTURAL_TOKEN_IDS_D
    if 0 <= token_id < VOCAB_SIZE
}


def block_structural_tokens_D(
    logits,
):

    logits = logits.clone()


    for token_id in STRUCTURAL_TOKEN_IDS_D:

        logits[
            token_id
        ] = float(
            "-inf"
        )


    return logits


# =========================================================================
# 8. TOP-K FILTER
# =========================================================================

def apply_top_k_D(
    logits,
    top_k,
):

    if (
        top_k <= 0
        or
        top_k >= logits.numel()
    ):

        return logits


    top_values = torch.topk(
        logits,
        k=top_k,
    ).values


    kth_value = top_values[
        -1
    ]


    return torch.where(
        logits < kth_value,

        torch.full_like(
            logits,
            float(
                "-inf"
            ),
        ),

        logits,
    )


# =========================================================================
# 9. TOP-P FILTER
# =========================================================================

def apply_top_p_D(
    logits,
    top_p,
):

    if (
        top_p >= 1.0
    ):

        return logits


    sorted_logits, sorted_indices = (
        torch.sort(
            logits,
            descending=True,
        )
    )


    sorted_probabilities = (
        torch.softmax(
            sorted_logits,
            dim=-1,
        )
    )


    cumulative_probabilities = (
        torch.cumsum(
            sorted_probabilities,
            dim=-1,
        )
    )


    remove_mask = (
        cumulative_probabilities
        > top_p
    )


    # Always retain at least the highest-probability candidate.
    remove_mask[
        0
    ] = False


    sorted_logits[
        remove_mask
    ] = float(
        "-inf"
    )


    filtered_logits = torch.full_like(
        logits,
        float(
            "-inf"
        ),
    )


    filtered_logits.scatter_(
        0,
        sorted_indices,
        sorted_logits,
    )


    return filtered_logits


# =========================================================================
# 10. STANDARD SAMPLER
# =========================================================================

def sample_next_token_D(
    logits,
    generated_ids,
    generator,
):

    candidate_logits = (
        logits.float()
    )


    candidate_logits = (
        apply_repetition_penalty_D(
            candidate_logits,
            generated_ids,
            REPETITION_PENALTY_D,
        )
    )


    candidate_logits = (
        block_structural_tokens_D(
            candidate_logits
        )
    )


    candidate_logits = (
        candidate_logits
        / max(
            TEMPERATURE_D,
            1e-6,
        )
    )


    candidate_logits = (
        apply_top_k_D(
            candidate_logits,
            TOP_K_D,
        )
    )


    candidate_logits = (
        apply_top_p_D(
            candidate_logits,
            TOP_P_D,
        )
    )


    probabilities = torch.softmax(
        candidate_logits,
        dim=-1,
    )


    if not torch.isfinite(
        probabilities
    ).all().item():

        raise RuntimeError(
            "Generation probability distribution "
            "contains NaN or Inf."
        )


    probability_sum = (
        probabilities.sum()
    )


    if not torch.isfinite(
        probability_sum
    ).item():

        raise RuntimeError(
            "Generation probability distribution "
            "has an invalid total probability."
        )


    probabilities = (
        probabilities
        / probability_sum
    )


    sampled = torch.multinomial(
        probabilities,
        num_samples=1,
        generator=generator,
    )


    return int(
        sampled.item()
    )


# =========================================================================
# 11. WORD-BOUNDARY RESCUE
# =========================================================================
#
# BPE tokenization means one token does not necessarily correspond to one
# whitespace-separated word.
#
# If normal sampling repeatedly produces tokens that do not advance the
# whitespace-word count, inspect the strongest vocabulary candidates and
# choose the highest-scoring candidate that does advance the word count.
#
# The rescue token still comes directly from the model's vocabulary.


def find_word_advancing_token_D(
    logits,
    generated_ids,
    current_word_count,
):

    candidate_logits = (
        logits.float()
        .clone()
    )


    candidate_logits = (
        apply_repetition_penalty_D(
            candidate_logits,
            generated_ids,
            REPETITION_PENALTY_D,
        )
    )


    candidate_logits = (
        block_structural_tokens_D(
            candidate_logits
        )
    )


    candidate_logits = (
        candidate_logits
        / max(
            TEMPERATURE_D,
            1e-6,
        )
    )


    candidate_count = min(
        WORD_RESCUE_TOP_K_D,
        candidate_logits.numel(),
    )


    candidate_values, candidate_indices = (
        torch.topk(
            candidate_logits,
            k=candidate_count,
        )
    )


    best_token = None
    best_score = float(
        "-inf"
    )


    for score, token_tensor in zip(
        candidate_values.tolist(),
        candidate_indices.tolist(),
    ):

        token_id = int(
            token_tensor
        )


        if token_id in (
            STRUCTURAL_TOKEN_IDS_D
        ):

            continue


        candidate_ids = (
            generated_ids
            + [
                token_id
            ]
        )


        candidate_text = (
            decode_generated_tokens_D(
                candidate_ids
            )
        )


        candidate_word_count = (
            count_words_D(
                candidate_text
            )
        )


        if (
            candidate_word_count
            > current_word_count
        ):

            if (
                score
                > best_score
            ):

                best_score = float(
                    score
                )

                best_token = (
                    token_id
                )


    return best_token


# =========================================================================
# 12. TCN GENERATION FUNCTION
# =========================================================================

@torch.inference_mode()
def generate_architecture_D_continuation(
    prompt,
    target_words=None,
    max_new_tokens=None,
    temperature=None,
    top_p=None,
    top_k=None,
    repetition_penalty=None,
    generation_seed=None,
    existing_generated_ids=None,
):

    # ---------------------------------------------------------------------
    # Resolve configuration.
    # ---------------------------------------------------------------------

    if target_words is None:

        target_words = (
            TARGET_WORDS_D
        )


    if max_new_tokens is None:

        max_new_tokens = (
            MAX_NEW_TOKENS_D
        )


    if temperature is None:

        temperature = (
            TEMPERATURE_D
        )


    if top_p is None:

        top_p = (
            TOP_P_D
        )


    if top_k is None:

        top_k = (
            TOP_K_D
        )


    if repetition_penalty is None:

        repetition_penalty = (
            REPETITION_PENALTY_D
        )


    target_words = int(
        target_words
    )

    max_new_tokens = int(
        max_new_tokens
    )

    temperature = float(
        temperature
    )

    top_p = float(
        top_p
    )

    top_k = int(
        top_k
    )

    repetition_penalty = float(
        repetition_penalty
    )


    assert target_words > 0

    assert max_new_tokens > 0

    assert (
        max_new_tokens
        >= target_words
    )

    assert temperature > 0.0

    assert 0.0 < top_p <= 1.0

    assert top_k > 0

    assert repetition_penalty >= 1.0


    prompt = str(
        prompt
    ).strip()


    assert prompt, (
        "Generation prompt is empty."
    )


    # ---------------------------------------------------------------------
    # Tokenize official prompt.
    # ---------------------------------------------------------------------

    prompt_ids = (
        TOKENIZER_D.encode(
            prompt
        ).ids
    )


    prompt_ids = [
        int(token_id)
        for token_id
        in prompt_ids
    ]


    assert prompt_ids, (
        "Prompt produced zero tokenizer IDs."
    )


    # ---------------------------------------------------------------------
    # Context handling.
    # ---------------------------------------------------------------------

    if len(
        prompt_ids
    ) > CONTEXT_LENGTH:

        if RUN_MODE == "fast":

            prompt_ids = (
                prompt_ids[
                    -CONTEXT_LENGTH:
                ]
            )

        else:

            raise ValueError(
                "Production official prompt exceeds "
                "Architecture-D context length.\n"
                f"Prompt tokens : "
                f"{len(prompt_ids)}\n"
                f"Context       : "
                f"{CONTEXT_LENGTH}"
            )


    # ---------------------------------------------------------------------
    # Existing generated IDs for automatic resume.
    # ---------------------------------------------------------------------

    if existing_generated_ids is None:

        generated_ids = []

    else:

        generated_ids = [
            int(token_id)
            for token_id
            in existing_generated_ids
        ]


    current_word_count = (
        count_words_D(
            decode_generated_tokens_D(
                generated_ids
            )
        )
    )


    if (
        current_word_count
        >= target_words
    ):

        completed_text = (
            truncate_to_exact_words_D(
                decode_generated_tokens_D(
                    generated_ids
                ),
                target_words,
            )
        )


        assert completed_text is not None


        assert (
            count_words_D(
                completed_text
            )
            == target_words
        )


        return (
            completed_text,
            generated_ids,
        )


    # ---------------------------------------------------------------------
    # Deterministic generation stream.
    # ---------------------------------------------------------------------

    if generation_seed is None:

        generation_seed = (
            SEED
            + 700000
        )


    generation_generator = torch.Generator(
        device=DEVICE
    )


    generation_generator.manual_seed(
        int(
            generation_seed
        )
    )


    # ---------------------------------------------------------------------
    # Replay sampling calls when resuming.
    #
    # One multinomial operation is performed for every generated token
    # in the standard generation path.
    #
    # The word-rescue path does not consume generator state because it
    # chooses a deterministic highest-scoring advancing candidate.
    # Therefore we need to replay only standard sampling operations.
    # To make resume exact, the caller stores standard_sample_count.
    # ---------------------------------------------------------------------


    # This function-level helper is used by the external progress runner.
    # For a clean generation with existing tokens but no sampling-count
    # metadata, replay one multinomial draw per token as a conservative
    # deterministic alignment.
    #
    # The external runner will normally provide fresh generation from zero
    # or use its stored exact state.


    replay_sample_count = min(
        len(generated_ids),
        max_new_tokens,
    )


    if replay_sample_count > 0:

        replay_probabilities = (
            torch.ones(
                VOCAB_SIZE,
                dtype=torch.float32,
                device=DEVICE,
            )
            / float(
                VOCAB_SIZE
            )
        )


        for _ in range(
            replay_sample_count
        ):

            torch.multinomial(
                replay_probabilities,
                num_samples=1,
                generator=generation_generator,
            )


        del replay_probabilities


    # ---------------------------------------------------------------------
    # Autoregressive TCN generation.
    # ---------------------------------------------------------------------

    stalled_tokens = 0
    last_word_count = (
        current_word_count
    )


    while (
        count_words_D(
            decode_generated_tokens_D(
                generated_ids
            )
        )
        < target_words
    ):

        if len(
            generated_ids
        ) >= max_new_tokens:

            raise RuntimeError(
                "Architecture-D TCN generation reached "
                "max_new_tokens before producing the requested "
                f"{target_words} words.\n"
                f"Generated tokens : "
                f"{len(generated_ids)}\n"
                f"Generated words  : "
                f"{current_word_count}\n"
                f"Maximum tokens   : "
                f"{max_new_tokens}"
            )


        # -------------------------------------------------------------
        # Build current causal context.
        # -------------------------------------------------------------

        current_ids = (
            prompt_ids
            + generated_ids
        )


        if len(
            current_ids
        ) > MODEL.context_length:

            current_ids = (
                current_ids[
                    -MODEL.context_length:
                ]
            )


        input_tensor = torch.tensor(
            [
                current_ids
            ],
            dtype=torch.long,
            device=DEVICE,
        )


        # -------------------------------------------------------------
        # Forward pass.
        # -------------------------------------------------------------

        with torch.amp.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(
                DEVICE.type
                == "cuda"
            ),
        ):

            logits = MODEL(
                input_tensor
            )


        next_logits = (
            logits[
                0,
                -1,
                :
            ]
            .float()
        )


        # -------------------------------------------------------------
        # Normal sampling.
        # -------------------------------------------------------------

        next_token_id = (
            sample_next_token_D(
                next_logits,
                generated_ids,
                generation_generator,
            )
        )


        candidate_generated_ids = (
            generated_ids
            + [
                next_token_id
            ]
        )


        candidate_text = (
            decode_generated_tokens_D(
                candidate_generated_ids
            )
        )


        candidate_word_count = (
            count_words_D(
                candidate_text
            )
        )


        # -------------------------------------------------------------
        # Word-boundary rescue.
        #
        # If the sampled token does not advance the whitespace-word
        # count for too long, inspect the strongest candidates.
        # -------------------------------------------------------------

        if (
            candidate_word_count
            <= current_word_count
        ):

            stalled_tokens += 1

        else:

            stalled_tokens = 0


        if (
            stalled_tokens
            >= 8
        ):

            rescue_token_id = (
                find_word_advancing_token_D(
                    next_logits,
                    generated_ids,
                    current_word_count,
                )
            )


            if rescue_token_id is not None:

                next_token_id = (
                    rescue_token_id
                )

                candidate_generated_ids = (
                    generated_ids
                    + [
                        next_token_id
                    ]
                )

                candidate_text = (
                    decode_generated_tokens_D(
                        candidate_generated_ids
                    )
                )

                candidate_word_count = (
                    count_words_D(
                        candidate_text
                    )
                )

                stalled_tokens = 0


        # -------------------------------------------------------------
        # Commit sampled token.
        # -------------------------------------------------------------

        generated_ids = (
            candidate_generated_ids
        )


        current_word_count = (
            candidate_word_count
        )


        # -------------------------------------------------------------
        # Stop exactly when enough words exist.
        # -------------------------------------------------------------

        if (
            current_word_count
            >= target_words
        ):

            break


        # -------------------------------------------------------------
        # Cleanup.
        # -------------------------------------------------------------

        del input_tensor
        del logits
        del next_logits

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    # ---------------------------------------------------------------------
    # Final exact-word conversion.
    # ---------------------------------------------------------------------

    decoded_text = (
        decode_generated_tokens_D(
            generated_ids
        )
    )


    continuation = (
        truncate_to_exact_words_D(
            decoded_text,
            target_words,
        )
    )


    assert continuation is not None, (
        "Generation finished without enough words."
    )


    final_word_count = (
        count_words_D(
            continuation
        )
    )


    assert (
        final_word_count
        == target_words
    ), (
        f"Expected {target_words} words, "
        f"got {final_word_count}."
    )


    assert continuation.strip()


    return (
        continuation,
        generated_ids,
    )


# =========================================================================
# 13. GENERATION FUNCTION TEST
# =========================================================================
#
# Test only the first active official example.
# This is a smoke test and does not yet write predictions.


TEST_EXAMPLE_D = (
    ACTIVE_TEST_RECORDS[
        0
    ]
)


TEST_EXAMPLE_ID_D = str(
    TEST_EXAMPLE_D[
        "id"
    ]
)

TEST_EXAMPLE_PROMPT_D = str(
    TEST_EXAMPLE_D[
        "prompt"
    ]
)


TEST_SEED_D = (
    stable_generation_seed_D(
        TEST_EXAMPLE_ID_D
    )
)


print("\n" + "-" * 70)
print("GENERATION FUNCTION SMOKE TEST")
print("-" * 70)

print(
    f"Test ID                 : "
    f"{TEST_EXAMPLE_ID_D}"
)

print(
    f"Prompt words            : "
    f"{count_words_D(TEST_EXAMPLE_PROMPT_D)}"
)

print(
    f"Generation seed         : "
    f"{TEST_SEED_D}"
)


TEST_CONTINUATION_D, TEST_GENERATED_IDS_D = (
    generate_architecture_D_continuation(
        prompt=TEST_EXAMPLE_PROMPT_D,
        target_words=TARGET_WORDS_D,
        max_new_tokens=MAX_NEW_TOKENS_D,
        temperature=TEMPERATURE_D,
        top_p=TOP_P_D,
        top_k=TOP_K_D,
        repetition_penalty=REPETITION_PENALTY_D,
        generation_seed=TEST_SEED_D,
    )
)


assert (
    count_words_D(
        TEST_CONTINUATION_D
    )
    == TARGET_WORDS_D
)


print(
    f"Generated tokens        : "
    f"{len(TEST_GENERATED_IDS_D)}"
)

print(
    f"Generated words         : "
    f"{count_words_D(TEST_CONTINUATION_D)}"
)

print(
    "Generated continuation:"
)

print(
    TEST_CONTINUATION_D
)


print(
    "\n✅ Generation function smoke test passed."
)


# =========================================================================
# 14. FINAL ENGINE MANIFEST
# =========================================================================

GENERATION_ENGINE_MANIFEST = {

    "status": "ready",

    "experiment_id": EXPERIMENT_ID,

    "architecture_name": ARCHITECTURE_NAME,

    "architecture_version": ARCHITECTURE_VERSION,

    "run_mode": RUN_MODE,

    "target_words": TARGET_WORDS_D,

    "max_new_tokens": MAX_NEW_TOKENS_D,

    "temperature": TEMPERATURE_D,

    "top_k": TOP_K_D,

    "top_p": TOP_P_D,

    "repetition_penalty": (
        REPETITION_PENALTY_D
    ),

    "word_rescue_top_k": (
        WORD_RESCUE_TOP_K_D
    ),

    "context_length": CONTEXT_LENGTH,

    "model_fingerprint": MODEL_FINGERPRINT,

    "test_smoke_id": TEST_EXAMPLE_ID_D,

    "test_smoke_generated_tokens": (
        len(TEST_GENERATED_IDS_D)
    ),

    "test_smoke_generated_words": (
        count_words_D(
            TEST_CONTINUATION_D
        )
    ),
}


GENERATION_ENGINE_MANIFEST_PATH = (
    CONFIGS_DIR
    / "generation_engine_D.json"
)


with GENERATION_ENGINE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        GENERATION_ENGINE_MANIFEST,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =========================================================================
# 15. CLEANUP
# =========================================================================

del TEST_CONTINUATION_D
del TEST_GENERATED_IDS_D
del TEST_EXAMPLE_D
del TEST_EXAMPLE_PROMPT_D

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# =========================================================================
# 16. FINAL OUTPUT
# =========================================================================

print(
    "\n"
    + "=" * 70
)

print(
    "✅ CELL 12 COMPLETE — GENERATION ENGINE READY"
)

print(
    "=" * 70
)

print(
    f"Target words             : "
    f"{TARGET_WORDS_D}"
)

print(
    f"Maximum new tokens       : "
    f"{MAX_NEW_TOKENS_D}"
)

print(
    f"Temperature              : "
    f"{TEMPERATURE_D}"
)

print(
    f"Top-k                    : "
    f"{TOP_K_D}"
)

print(
    f"Top-p                    : "
    f"{TOP_P_D}"
)

print(
    f"Word rescue top-k        : "
    f"{WORD_RESCUE_TOP_K_D}"
)

print(
    f"Context length           : "
    f"{CONTEXT_LENGTH}"
)

print(
    f"Generation engine        : "
    f"{GENERATION_ENGINE_MANIFEST_PATH}"
)

print(
    "\n✅ TCN generation function is ready."
)

print(
    "✅ No predictions were written yet."
)

print("=" * 70)

ARCHITECTURE-D TCN GENERATION ENGINE

----------------------------------------------------------------------
GENERATION CONFIGURATION
----------------------------------------------------------------------
Run mode               : production
Target words            : 128
Maximum new tokens      : 4096
Temperature             : 0.8
Top-k                   : 50
Top-p                   : 0.95
Repetition penalty     : 1.0
Word-rescue candidates  : 100
TCN context length      : 2048

----------------------------------------------------------------------
TOKENIZER
----------------------------------------------------------------------
Vocabulary size         : 50,000
✅ Architecture-D tokenizer loaded.

----------------------------------------------------------------------
GENERATION FUNCTION SMOKE TEST
----------------------------------------------------------------------
Test ID                 : 1
Prompt words            : 512
Generation seed         : 189574247
Generated tokens        : 149

In [13]:
# =========================================================================
# CELL 13 — ARCHITECTURE-D OFFICIAL TEST GENERATION + PERSISTED RESUME
# DECISION: REPLACE
# =========================================================================
#
# Guarantees:
#   - Architecture-D only
#   - Uses Cell-10 selected best checkpoint
#   - Uses Cell-11 official test records
#   - Uses Cell-12 generation engine
#   - NEVER includes the original prompt in prediction.json
#   - Exact whitespace-word count
#   - Automatically restores persisted D prediction/progress artifacts
#   - If a valid completed prediction already exists, NO regeneration
#   - Generates only missing/incomplete examples
#   - Saves progress after every generated token
#   - Production: 1000 examples × 128 words
#
# IMPORTANT:
# This cell can recover from:
#   /kaggle/working/architecture_D
#   OR a mounted Architecture-D dataset under /kaggle/input.
#
# It NEVER accesses Architecture-A/B/C artifacts.


import gc
import json
import shutil
import time
from pathlib import Path

import torch


print("=" * 70)
print("ARCHITECTURE-D OFFICIAL TEST GENERATION")
print("=" * 70)


# =========================================================================
# 1. PRECONDITIONS
# =========================================================================

assert "MODEL" in globals(), (
    "MODEL is unavailable. Run Cell 10 first."
)

assert "ACTIVE_TEST_RECORDS" in globals(), (
    "ACTIVE_TEST_RECORDS is unavailable. Run Cell 11 first."
)

assert "ACTIVE_TEST_COUNT" in globals(), (
    "ACTIVE_TEST_COUNT is unavailable. Run Cell 11 first."
)

assert "TOKENIZER_D" in globals(), (
    "TOKENIZER_D is unavailable. Run Cell 12 first."
)

assert callable(
    generate_architecture_D_continuation
), (
    "Architecture-D generation function is unavailable. "
    "Run Cell 12 first."
)

assert "stable_generation_seed_D" in globals(), (
    "stable_generation_seed_D is unavailable. Run Cell 12 first."
)

assert "TARGET_WORDS_D" in globals(), (
    "TARGET_WORDS_D is unavailable. Run Cell 12 first."
)

assert "MAX_NEW_TOKENS_D" in globals(), (
    "MAX_NEW_TOKENS_D is unavailable. Run Cell 12 first."
)

assert "TEMPERATURE_D" in globals(), (
    "TEMPERATURE_D is unavailable. Run Cell 12 first."
)

assert "TOP_K_D" in globals(), (
    "TOP_K_D is unavailable. Run Cell 12 first."
)

assert "TOP_P_D" in globals(), (
    "TOP_P_D is unavailable. Run Cell 12 first."
)

assert "REPETITION_PENALTY_D" in globals(), (
    "REPETITION_PENALTY_D is unavailable. Run Cell 12 first."
)

assert "MODEL_FINGERPRINT" in globals(), (
    "MODEL_FINGERPRINT is unavailable."
)

assert "DATA_FINGERPRINT" in globals(), (
    "DATA_FINGERPRINT is unavailable."
)

assert "EXPERIMENT_ID" in globals(), (
    "EXPERIMENT_ID is unavailable."
)

assert "ARCHITECTURE_NAME" in globals(), (
    "ARCHITECTURE_NAME is unavailable."
)

assert "ARCHITECTURE_VERSION" in globals(), (
    "ARCHITECTURE_VERSION is unavailable."
)

assert "RUN_MODE" in globals(), (
    "RUN_MODE is unavailable."
)

assert "PREDICTIONS_DIR" in globals(), (
    "PREDICTIONS_DIR is unavailable."
)

assert "CONFIGS_DIR" in globals(), (
    "CONFIGS_DIR is unavailable."
)


MODEL.eval()


# =========================================================================
# 2. LOCAL ARTIFACT PATHS
# =========================================================================

PREDICTIONS_ROOT = Path(
    PREDICTIONS_DIR
)

CONFIGS_ROOT = Path(
    CONFIGS_DIR
)


PREDICTIONS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

CONFIGS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


GENERATION_PROGRESS_PATH_D = (
    PREDICTIONS_ROOT
    / "generation_progress_D.json"
)

PARTIAL_PREDICTION_PATH_D = (
    PREDICTIONS_ROOT
    / "prediction_D_partial.json"
)

FINAL_PREDICTION_PATH_D = (
    PREDICTIONS_ROOT
    / "prediction_D.json"
)

GENERATION_SUMMARY_PATH_D = (
    CONFIGS_ROOT
    / "generation_summary_D.json"
)


# =========================================================================
# 3. ACTIVE OFFICIAL TEST IDS
# =========================================================================

ACTIVE_TEST_IDS_D = [
    str(
        record["id"]
    )
    for record
    in ACTIVE_TEST_RECORDS
]


assert (
    len(ACTIVE_TEST_IDS_D)
    == ACTIVE_TEST_COUNT
), (
    "Active test record count does not match ACTIVE_TEST_COUNT."
)


assert (
    len(set(ACTIVE_TEST_IDS_D))
    == ACTIVE_TEST_COUNT
), (
    "Duplicate active official-test IDs detected."
)


print(
    "\n" + "-" * 70
)

print(
    "GENERATION SET"
)

print(
    "-" * 70
)

print(
    f"Run mode             : {RUN_MODE}"
)

print(
    f"Active test examples : {ACTIVE_TEST_COUNT:,}"
)

print(
    f"Target words/example : {TARGET_WORDS_D}"
)

print(
    f"Maximum new tokens   : {MAX_NEW_TOKENS_D}"
)


# =========================================================================
# 4. SAFE JSON SAVE
# =========================================================================

def atomic_json_save_D(
    path,
    payload,
):

    path = Path(
        path
    )


    temporary_path = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            payload,
            file,
            ensure_ascii=False,
            indent=2,
        )


    temporary_path.replace(
        path
    )


# =========================================================================
# 5. DISCOVER MOUNTED ARCHITECTURE-D ARTIFACTS
# =========================================================================
#
# Search ONLY for directories containing architecture_D.
# Do not reuse A/B/C artifacts.
# =========================================================================

def discover_mounted_architecture_D_root():

    input_root = Path(
        "/kaggle/input"
    )


    if not input_root.exists():

        return None


    candidates = []


    try:

        for candidate in input_root.glob(
            "**/architecture_D"
        ):

            if candidate.is_dir():

                candidates.append(
                    candidate
                )

    except Exception:

        return None


    if not candidates:

        return None


    # Prefer candidates containing a completed D prediction.
    completed_candidates = [

        candidate

        for candidate
        in candidates

        if (
            candidate
            / "predictions"
            / "prediction_D.json"
        ).is_file()
    ]


    if completed_candidates:

        candidates = completed_candidates


    # Prefer candidates containing checkpoints too.
    checkpoint_candidates = [

        candidate

        for candidate
        in candidates

        if (
            candidate
            / "checkpoints"
            / "latest.pt"
        ).is_file()
    ]


    if checkpoint_candidates:

        candidates = checkpoint_candidates


    candidates.sort(
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )


    return candidates[0]


MOUNTED_ARCHITECTURE_D_ROOT = (
    discover_mounted_architecture_D_root()
)


if (
    MOUNTED_ARCHITECTURE_D_ROOT
    is not None
):

    print(
        "\n✅ Mounted Architecture-D dataset found:"
    )

    print(
        f"   {MOUNTED_ARCHITECTURE_D_ROOT}"
    )

else:

    print(
        "\nℹ️ No mounted Architecture-D dataset found."
    )


# =========================================================================
# 6. RESTORE PERSISTED GENERATION ARTIFACTS
# =========================================================================

def restore_from_mounted_D(
    local_path,
    relative_path,
):

    local_path = Path(
        local_path
    )


    if local_path.is_file():

        return False


    if (
        MOUNTED_ARCHITECTURE_D_ROOT
        is None
    ):

        return False


    mounted_path = (
        MOUNTED_ARCHITECTURE_D_ROOT
        / relative_path
    )


    if not mounted_path.is_file():

        return False


    shutil.copy2(
        mounted_path,
        local_path,
    )


    print(
        "♻️ Restored persisted D artifact:"
    )

    print(
        f"   Mounted : {mounted_path}"
    )

    print(
        f"   Local   : {local_path}"
    )


    return True


# Restore final prediction FIRST.
# This is the most important recovery path because a completed final
# prediction means generation does not need to happen again.

restore_from_mounted_D(
    FINAL_PREDICTION_PATH_D,
    Path(
        "predictions"
    )
    / "prediction_D.json",
)


restore_from_mounted_D(
    PARTIAL_PREDICTION_PATH_D,
    Path(
        "predictions"
    )
    / "prediction_D_partial.json",
)


restore_from_mounted_D(
    GENERATION_PROGRESS_PATH_D,
    Path(
        "predictions"
    )
    / "generation_progress_D.json",
)


# =========================================================================
# 7. VALIDATE EXISTING FINAL PREDICTION
# =========================================================================
#
# If a complete valid prediction is already present, use it directly
# and DO NOT generate again.
# =========================================================================

def validate_complete_prediction_D(
    path,
):

    path = Path(
        path
    )


    if not path.is_file():

        return None


    try:

        with path.open(
            "r",
            encoding="utf-8",
        ) as file:

            candidate = json.load(
                file
            )

    except Exception:

        return None


    if not isinstance(
        candidate,
        dict,
    ):

        return None


    if (
        list(candidate.keys())
        != ACTIVE_TEST_IDS_D
    ):

        return None


    for test_id in (
        ACTIVE_TEST_IDS_D
    ):

        value = candidate.get(
            test_id
        )


        if not isinstance(
            value,
            dict,
        ):

            return None


        if set(
            value.keys()
        ) != {
            "text"
        }:

            return None


        text = value[
            "text"
        ]


        if not isinstance(
            text,
            str,
        ):

            return None


        if (
            len(
                text.strip().split()
            )
            != TARGET_WORDS_D
        ):

            return None


    return candidate


existing_final_prediction_D = (
    validate_complete_prediction_D(
        FINAL_PREDICTION_PATH_D
    )
)


# =========================================================================
# 8. COMPLETE PREDICTION ALREADY EXISTS
# =========================================================================

if (
    existing_final_prediction_D
    is not None
):

    predictions_D = (
        existing_final_prediction_D
    )


    print(
        "\n" + "=" * 70
    )

    print(
        "✅ EXISTING COMPLETE PREDICTION DETECTED"
    )

    print(
        "=" * 70
    )

    print(
        f"Prediction file      : "
        f"{FINAL_PREDICTION_PATH_D}"
    )

    print(
        f"Examples              : "
        f"{len(predictions_D):,}"
    )

    print(
        f"Words/example         : "
        f"{TARGET_WORDS_D}"
    )

    print(
        "Generation required   : NO"
    )

    print(
        "Regeneration          : SKIPPED"
    )

    print(
        "\n✅ Previous D prediction will be reused."
    )


    # -------------------------------------------------------------
    # Recreate a valid progress artifact locally if it is absent.
    # -------------------------------------------------------------

    generation_progress_D = {

        "status":
            "complete",

        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "model_fingerprint":
            MODEL_FINGERPRINT,

        "data_fingerprint":
            DATA_FINGERPRINT,

        "target_words":
            TARGET_WORDS_D,

        "max_new_tokens":
            MAX_NEW_TOKENS_D,

        "temperature":
            TEMPERATURE_D,

        "top_k":
            TOP_K_D,

        "top_p":
            TOP_P_D,

        "repetition_penalty":
            REPETITION_PENALTY_D,

        "active_test_count":
            ACTIVE_TEST_COUNT,

        "active_test_ids":
            list(
                ACTIVE_TEST_IDS_D
            ),

        "completed_ids":
            list(
                ACTIVE_TEST_IDS_D
            ),

        "completed_count":
            ACTIVE_TEST_COUNT,

        "final_prediction_path":
            str(
                FINAL_PREDICTION_PATH_D
            ),

        "recovered_from_completed_prediction":
            True,

        "updated_unix_time":
            time.time(),

    }


    atomic_json_save_D(
        GENERATION_PROGRESS_PATH_D,
        generation_progress_D,
    )


    # -------------------------------------------------------------
    # Save a consistent partial prediction too.
    # -------------------------------------------------------------

    atomic_json_save_D(
        PARTIAL_PREDICTION_PATH_D,
        predictions_D,
    )


# =========================================================================
# 9. OTHERWISE LOAD / INITIALIZE PROGRESS
# =========================================================================

else:

    generation_progress_D = None

    predictions_D = {}


    # -------------------------------------------------------------
    # Try persisted generation progress.
    # -------------------------------------------------------------

    if (
        GENERATION_PROGRESS_PATH_D.is_file()
        and
        PARTIAL_PREDICTION_PATH_D.is_file()
    ):

        try:

            with GENERATION_PROGRESS_PATH_D.open(
                "r",
                encoding="utf-8",
            ) as file:

                candidate_progress = json.load(
                    file
                )


            with PARTIAL_PREDICTION_PATH_D.open(
                "r",
                encoding="utf-8",
            ) as file:

                candidate_predictions = json.load(
                    file
                )


            assert (
                candidate_progress[
                    "experiment_id"
                ]
                == EXPERIMENT_ID
            )


            assert (
                candidate_progress[
                    "architecture_name"
                ]
                == ARCHITECTURE_NAME
            )


            assert (
                candidate_progress[
                    "architecture_version"
                ]
                == ARCHITECTURE_VERSION
            )


            assert (
                candidate_progress[
                    "run_mode"
                ]
                == RUN_MODE
            )


            assert (
                candidate_progress[
                    "model_fingerprint"
                ]
                == MODEL_FINGERPRINT
            )


            assert (
                candidate_progress[
                    "data_fingerprint"
                ]
                == DATA_FINGERPRINT
            )


            assert (
                int(
                    candidate_progress[
                        "target_words"
                    ]
                )
                == TARGET_WORDS_D
            )


            assert (
                int(
                    candidate_progress[
                        "max_new_tokens"
                    ]
                )
                == MAX_NEW_TOKENS_D
            )


            assert isinstance(
                candidate_predictions,
                dict,
            )


            # Only active official IDs are allowed.
            assert (
                set(
                    candidate_predictions.keys()
                )
                <= set(
                    ACTIVE_TEST_IDS_D
                )
            )


            # Validate partial predictions and discard malformed entries.
            valid_predictions = {}


            for test_id, value in (
                candidate_predictions.items()
            ):

                if not isinstance(
                    value,
                    dict,
                ):

                    continue


                if set(
                    value.keys()
                ) != {
                    "text"
                }:

                    continue


                text_value = value[
                    "text"
                ]


                if not isinstance(
                    text_value,
                    str,
                ):

                    continue


                if not text_value.strip():

                    continue


                if (
                    len(
                        text_value.strip().split()
                    )
                    != TARGET_WORDS_D
                ):

                    continue


                valid_predictions[
                    str(test_id)
                ] = {
                    "text": text_value
                }


            generation_progress_D = (
                candidate_progress
            )

            predictions_D = (
                valid_predictions
            )


            print(
                "\n♻️ Existing D generation state restored."
            )

            print(
                f"   Valid completed predictions : "
                f"{len(predictions_D):,}"
            )

            print(
                f"   Active examples             : "
                f"{ACTIVE_TEST_COUNT:,}"
            )


        except Exception as exc:

            print(
                "\n⚠️ Existing generation state "
                "could not be safely reused."
            )

            print(
                f"   Reason: {exc}"
            )

            generation_progress_D = None

            predictions_D = {}


    # -------------------------------------------------------------
    # Initialize new state if necessary.
    # -------------------------------------------------------------

    if generation_progress_D is None:

        generation_progress_D = {

            "status":
                "initialized",

            "experiment_id":
                EXPERIMENT_ID,

            "architecture_name":
                ARCHITECTURE_NAME,

            "architecture_version":
                ARCHITECTURE_VERSION,

            "run_mode":
                RUN_MODE,

            "model_fingerprint":
                MODEL_FINGERPRINT,

            "data_fingerprint":
                DATA_FINGERPRINT,

            "target_words":
                TARGET_WORDS_D,

            "max_new_tokens":
                MAX_NEW_TOKENS_D,

            "temperature":
                TEMPERATURE_D,

            "top_k":
                TOP_K_D,

            "top_p":
                TOP_P_D,

            "repetition_penalty":
                REPETITION_PENALTY_D,

            "active_test_count":
                ACTIVE_TEST_COUNT,

            "active_test_ids":
                list(
                    ACTIVE_TEST_IDS_D
                ),

            "completed_ids":
                [],

            "completed_count":
                0,

            "started_unix_time":
                time.time(),

            "updated_unix_time":
                time.time(),

        }


        atomic_json_save_D(
            GENERATION_PROGRESS_PATH_D,
            generation_progress_D,
        )


        atomic_json_save_D(
            PARTIAL_PREDICTION_PATH_D,
            predictions_D,
        )


        print(
            "\n✅ New D generation state initialized."
        )


# =========================================================================
# 10. BUILD COMPLETED-ID SET
# =========================================================================

completed_ids_D = set(
    str(test_id)
    for test_id
    in predictions_D.keys()
)


# Also incorporate progress completed IDs.
for test_id in (
    generation_progress_D.get(
        "completed_ids",
        [],
    )
):

    test_id = str(
        test_id
    )


    if test_id in (
        ACTIVE_TEST_IDS_D
    ):

        if test_id in predictions_D:

            completed_ids_D.add(
                test_id
            )


generation_progress_D[
    "completed_ids"
] = sorted(
    completed_ids_D,
    key=lambda value: (
        int(value)
        if value.isdigit()
        else value
    ),
)


generation_progress_D[
    "completed_count"
] = len(
    completed_ids_D
)


atomic_json_save_D(
    GENERATION_PROGRESS_PATH_D,
    generation_progress_D,
)


# =========================================================================
# 11. REPORT RESUME STATUS
# =========================================================================

print(
    "\n" + "-" * 70
)

print(
    "RESUME STATUS"
)

print(
    "-" * 70
)

print(
    f"Previously completed : "
    f"{len(completed_ids_D):,}/"
    f"{ACTIVE_TEST_COUNT:,}"
)

print(
    f"Remaining examples   : "
    f"{ACTIVE_TEST_COUNT - len(completed_ids_D):,}"
)


# =========================================================================
# 12. EXACT WORD COUNT
# =========================================================================

def exact_word_count_D(
    text,
):

    return len(
        str(
            text
        )
        .strip()
        .split()
    )


# =========================================================================
# 13. GENERATE ONLY MISSING EXAMPLES
# =========================================================================

generation_wall_start = (
    time.time()
)


for index, record in enumerate(
    ACTIVE_TEST_RECORDS,
    start=1,
):

    test_id = str(
        record["id"]
    )

    prompt = str(
        record["prompt"]
    )


    # -------------------------------------------------------------
    # Already complete.
    # -------------------------------------------------------------

    if test_id in completed_ids_D:

        print(
            f"\n[{index:>4}/"
            f"{ACTIVE_TEST_COUNT:<4}] "
            f"ID={test_id} "
            "already complete — skipped."
        )

        continue


    print(
        f"\n[{index:>4}/"
        f"{ACTIVE_TEST_COUNT:<4}] "
        f"GENERATING ID={test_id}"
    )


    # -------------------------------------------------------------
    # Generation seed.
    # -------------------------------------------------------------

    generation_seed = (
        stable_generation_seed_D(
            test_id
        )
    )


    # -------------------------------------------------------------
    # Generate exact continuation.
    # -------------------------------------------------------------

    continuation = (
        generate_architecture_D_continuation(
            prompt,
            target_words=TARGET_WORDS_D,
            max_new_tokens=MAX_NEW_TOKENS_D,
            temperature=TEMPERATURE_D,
            top_k=TOP_K_D,
            top_p=TOP_P_D,
            repetition_penalty=(
                REPETITION_PENALTY_D
            ),
        )
    )


    continuation = str(
        continuation
    ).strip()


    assert continuation, (
        f"Test ID {test_id}: "
        "empty continuation generated."
    )


    generated_word_count = (
        exact_word_count_D(
            continuation
        )
    )


    assert (
        generated_word_count
        == TARGET_WORDS_D
    ), (
        f"Test ID {test_id}: expected "
        f"{TARGET_WORDS_D} words, got "
        f"{generated_word_count}."
    )


    # -------------------------------------------------------------
    # Store only generated continuation.
    # -------------------------------------------------------------

    predictions_D[
        test_id
    ] = {

        "text":
            continuation

    }


    completed_ids_D.add(
        test_id
    )


    # -------------------------------------------------------------
    # Persist immediately.
    # -------------------------------------------------------------

    generation_progress_D[
        "completed_ids"
    ] = sorted(
        completed_ids_D,
        key=lambda value: (
            int(value)
            if value.isdigit()
            else value
        ),
    )


    generation_progress_D[
        "completed_count"
    ] = len(
        completed_ids_D
    )


    generation_progress_D[
        "last_completed_id"
    ] = test_id


    generation_progress_D[
        "updated_unix_time"
    ] = time.time()


    atomic_json_save_D(
        PARTIAL_PREDICTION_PATH_D,
        predictions_D,
    )


    atomic_json_save_D(
        GENERATION_PROGRESS_PATH_D,
        generation_progress_D,
    )


    print(
        f"✅ ID={test_id} complete "
        f"({generated_word_count} words)"
    )


    # -------------------------------------------------------------
    # GPU cleanup.
    # -------------------------------------------------------------

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# =========================================================================
# 14. FINAL VALIDATION
# =========================================================================

EXPECTED_ACTIVE_IDS_D = set(
    ACTIVE_TEST_IDS_D
)


ACTUAL_PREDICTION_IDS_D = set(
    str(test_id)
    for test_id in predictions_D.keys()
)


MISSING_PREDICTION_IDS_D = (
    EXPECTED_ACTIVE_IDS_D
    - ACTUAL_PREDICTION_IDS_D
)


UNEXPECTED_PREDICTION_IDS_D = (
    ACTUAL_PREDICTION_IDS_D
    - EXPECTED_ACTIVE_IDS_D
)


assert (
    ACTUAL_PREDICTION_IDS_D
    == EXPECTED_ACTIVE_IDS_D
), (
    "Prediction IDs do not exactly match "
    "the active official test IDs.\n"
    f"Missing: "
    f"{sorted(MISSING_PREDICTION_IDS_D)}\n"
    f"Unexpected: "
    f"{sorted(UNEXPECTED_PREDICTION_IDS_D)}"
)


for test_id in (
    ACTIVE_TEST_IDS_D
):

    prediction = predictions_D[
        test_id
    ]


    assert isinstance(
        prediction,
        dict,
    )


    assert (
        set(
            prediction.keys()
        )
        == {
            "text"
        }
    )


    continuation = prediction[
        "text"
    ]


    assert isinstance(
        continuation,
        str,
    )


    assert continuation.strip()


    assert (
        exact_word_count_D(
            continuation
        )
        == TARGET_WORDS_D
    ), (
        f"Prediction {test_id}: expected "
        f"{TARGET_WORDS_D} words, found "
        f"{exact_word_count_D(continuation)}."
    )


# =========================================================================
# 15. PRESERVE OFFICIAL TEST ORDER
# =========================================================================

ordered_predictions_D = {

    test_id:
        predictions_D[
            test_id
        ]

    for test_id
    in ACTIVE_TEST_IDS_D

}


# =========================================================================
# 16. WRITE FINAL PREDICTION
# =========================================================================

atomic_json_save_D(
    FINAL_PREDICTION_PATH_D,
    ordered_predictions_D,
)


# =========================================================================
# 17. FINAL GENERATION STATE
# =========================================================================

generation_progress_D[
    "status"
] = "complete"


generation_progress_D[
    "completed_ids"
] = list(
    ACTIVE_TEST_IDS_D
)


generation_progress_D[
    "completed_count"
] = ACTIVE_TEST_COUNT


generation_progress_D[
    "final_prediction_path"
] = str(
    FINAL_PREDICTION_PATH_D
)


generation_progress_D[
    "completed_unix_time"
] = time.time()


atomic_json_save_D(
    GENERATION_PROGRESS_PATH_D,
    generation_progress_D,
)


# =========================================================================
# 18. GENERATION SUMMARY
# =========================================================================

generation_elapsed_seconds = (
    time.time()
    - generation_wall_start
)


generation_summary_D = {

    "status":
        "complete",

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "model_fingerprint":
        MODEL_FINGERPRINT,

    "data_fingerprint":
        DATA_FINGERPRINT,

    "official_test_count":
        1000,

    "active_test_count":
        ACTIVE_TEST_COUNT,

    "target_words":
        TARGET_WORDS_D,

    "max_new_tokens":
        MAX_NEW_TOKENS_D,

    "temperature":
        TEMPERATURE_D,

    "top_k":
        TOP_K_D,

    "top_p":
        TOP_P_D,

    "repetition_penalty":
        REPETITION_PENALTY_D,

    "prediction_path":
        str(
            FINAL_PREDICTION_PATH_D
        ),

    "partial_prediction_path":
        str(
            PARTIAL_PREDICTION_PATH_D
        ),

    "progress_path":
        str(
            GENERATION_PROGRESS_PATH_D
        ),

    "elapsed_seconds":
        generation_elapsed_seconds,

    "predictions_completed":
        len(
            ordered_predictions_D
        ),

    "generation_reused_existing_prediction":
        (
            existing_final_prediction_D
            is not None
        ),

}


atomic_json_save_D(
    GENERATION_SUMMARY_PATH_D,
    generation_summary_D,
)


# =========================================================================
# 19. FINAL OUTPUT
# =========================================================================

print(
    "\n"
    + "=" * 70
)

print(
    "✅ CELL 13 COMPLETE — "
    "OFFICIAL GENERATION FINISHED"
)

print(
    "=" * 70
)

print(
    f"Examples in final prediction : "
    f"{len(ordered_predictions_D):,}"
)

print(
    f"Words/example                : "
    f"{TARGET_WORDS_D}"
)

print(
    f"Generation reused old output : "
    f"{existing_final_prediction_D is not None}"
)

print(
    f"Prediction file              : "
    f"{FINAL_PREDICTION_PATH_D}"
)

print(
    f"Generation progress          : "
    f"{GENERATION_PROGRESS_PATH_D}"
)

print(
    f"Generation summary           : "
    f"{GENERATION_SUMMARY_PATH_D}"
)

print(
    f"Elapsed time                 : "
    f"{generation_elapsed_seconds:.2f} seconds"
)

print(
    "\n✅ Exact active official-test IDs verified."
)

print(
    "✅ Exact word counts verified."
)

print(
    "✅ Only generated continuation text is stored."
)

print(
    "✅ Persisted Architecture-D recovery supported."
)

print(
    "✅ Architecture-D remains independent of A/B/C."
)

print(
    "=" * 70
)

ARCHITECTURE-D OFFICIAL TEST GENERATION

----------------------------------------------------------------------
GENERATION SET
----------------------------------------------------------------------
Run mode             : production
Active test examples : 1,000
Target words/example : 128
Maximum new tokens   : 4096

✅ Mounted Architecture-D dataset found:
   /kaggle/input/datasets/ruchikhemka/data-archd/architecture_D
♻️ Restored persisted D artifact:
   Mounted : /kaggle/input/datasets/ruchikhemka/data-archd/architecture_D/predictions/prediction_D.json
   Local   : /kaggle/working/architecture_D/predictions/prediction_D.json
♻️ Restored persisted D artifact:
   Mounted : /kaggle/input/datasets/ruchikhemka/data-archd/architecture_D/predictions/prediction_D_partial.json
   Local   : /kaggle/working/architecture_D/predictions/prediction_D_partial.json
♻️ Restored persisted D artifact:
   Mounted : /kaggle/input/datasets/ruchikhemka/data-archd/architecture_D/predictions/generation_progress

In [14]:
# Cell 14 — Architecture-D prediction validation and submission packaging
# DECISION: ADD
#
# This cell:
#   1. Loads the Architecture-D official test manifest.
#   2. Loads prediction_D.json.
#   3. Verifies prediction IDs exactly match the active official IDs.
#   4. Verifies every prediction contains ONLY the "text" field.
#   5. Verifies every continuation is non-empty.
#   6. Verifies exact whitespace-separated word count.
#   7. Preserves official test ID order.
#   8. Writes canonical prediction.json.
#   9. Creates submission.zip.
#  10. Verifies prediction.json is directly at ZIP root.
#  11. Verifies ZIP contents match prediction.json.
#  12. Saves a complete Architecture-D submission manifest.
#
# FAST:
#     2 active predictions × 16 words
#
# PRODUCTION:
#     1000 active predictions × 128 words
#
# No training occurs.
# No generation occurs.
# No other architecture directories are accessed.


import json
import zipfile
from pathlib import Path


print("=" * 70)
print("ARCHITECTURE-D PREDICTION VALIDATION + SUBMISSION")
print("=" * 70)


# =========================================================================
# 1. ARCHITECTURE-D-ONLY PATHS
# =========================================================================

ARCHITECTURE_D_ROOT = Path(
    "/kaggle/working/architecture_D"
)

PREDICTIONS_DIR_D = (
    ARCHITECTURE_D_ROOT
    / "predictions"
)

CONFIGS_DIR_D = (
    ARCHITECTURE_D_ROOT
    / "configs"
)

SUBMISSION_DIR_D = (
    ARCHITECTURE_D_ROOT
    / "submission_package"
)

PREDICTION_PATH_D = (
    PREDICTIONS_DIR_D
    / "prediction_D.json"
)

OFFICIAL_TEST_MANIFEST_PATH_D = (
    CONFIGS_DIR_D
    / "official_test_manifest_D.json"
)

SUBMISSION_PREDICTION_PATH_D = (
    SUBMISSION_DIR_D
    / "prediction.json"
)

SUBMISSION_ZIP_PATH_D = (
    ARCHITECTURE_D_ROOT
    / "submission.zip"
)

SUBMISSION_MANIFEST_PATH_D = (
    CONFIGS_DIR_D
    / "submission_manifest_D.json"
)


# =========================================================================
# 2. PATH CHECKS
# =========================================================================

assert ARCHITECTURE_D_ROOT.is_dir(), (
    f"Architecture-D root not found:\n"
    f"{ARCHITECTURE_D_ROOT}"
)

assert PREDICTION_PATH_D.is_file(), (
    f"Architecture-D prediction file not found:\n"
    f"{PREDICTION_PATH_D}\n"
    "Run Cell 13 successfully first."
)

assert OFFICIAL_TEST_MANIFEST_PATH_D.is_file(), (
    f"Architecture-D official-test manifest not found:\n"
    f"{OFFICIAL_TEST_MANIFEST_PATH_D}\n"
    "Run Cell 11 successfully first."
)


PREDICTIONS_DIR_D.mkdir(
    parents=True,
    exist_ok=True,
)

CONFIGS_DIR_D.mkdir(
    parents=True,
    exist_ok=True,
)


# =========================================================================
# 3. LOAD OFFICIAL TEST MANIFEST
# =========================================================================

with OFFICIAL_TEST_MANIFEST_PATH_D.open(
    "r",
    encoding="utf-8",
) as file:

    official_test_manifest_D = json.load(
        file
    )


assert (
    official_test_manifest_D[
        "status"
    ]
    == "validated"
), (
    "Official-test manifest is not marked validated."
)


OFFICIAL_TEST_IDS_D = [
    str(test_id)
    for test_id
    in official_test_manifest_D[
        "active_test_ids"
    ]
]


ACTIVE_TEST_COUNT_D = int(
    official_test_manifest_D[
        "active_test_count"
    ]
)


TARGET_WORDS_D_VALIDATION = int(
    official_test_manifest_D[
        "generation_target_words"
    ]
)


assert (
    len(OFFICIAL_TEST_IDS_D)
    == ACTIVE_TEST_COUNT_D
)


assert (
    len(OFFICIAL_TEST_IDS_D)
    == len(
        set(
            OFFICIAL_TEST_IDS_D
        )
    )
)


# =========================================================================
# 4. LOAD PREDICTIONS
# =========================================================================

with PREDICTION_PATH_D.open(
    "r",
    encoding="utf-8",
) as file:

    predictions_D_validation = json.load(
        file
    )


print("\n" + "-" * 70)
print("PREDICTION FILE")
print("-" * 70)

print(
    f"Prediction path          : "
    f"{PREDICTION_PATH_D}"
)

print(
    f"Prediction JSON type     : "
    f"{type(predictions_D_validation).__name__}"
)

print(
    f"Prediction count         : "
    f"{len(predictions_D_validation):,}"
)

print(
    f"Expected active count    : "
    f"{ACTIVE_TEST_COUNT_D:,}"
)

print(
    f"Target words             : "
    f"{TARGET_WORDS_D_VALIDATION}"
)


# =========================================================================
# 5. TOP-LEVEL FORMAT
# =========================================================================
#
# Official Phase-1 format:
#
# {
#     "1": {
#         "text": "..."
#     },
#     "2": {
#         "text": "..."
#     }
# }


assert isinstance(
    predictions_D_validation,
    dict,
), (
    "Architecture-D prediction_D.json must be "
    "a dictionary keyed by official test ID."
)


# =========================================================================
# 6. PREDICTION COUNT
# =========================================================================

assert (
    len(
        predictions_D_validation
    )
    == ACTIVE_TEST_COUNT_D
), (
    f"Expected {ACTIVE_TEST_COUNT_D} predictions, "
    f"found {len(predictions_D_validation)}."
)


# =========================================================================
# 7. NORMALIZE ID TYPES
# =========================================================================

prediction_ids_D = {
    str(test_id)
    for test_id
    in predictions_D_validation.keys()
}


expected_id_set_D = set(
    OFFICIAL_TEST_IDS_D
)


assert (
    prediction_ids_D
    == expected_id_set_D
), (
    "Prediction IDs do not exactly match the active official-test IDs.\n"
    f"Missing IDs   : "
    f"{sorted(expected_id_set_D - prediction_ids_D)}\n"
    f"Unexpected IDs: "
    f"{sorted(prediction_ids_D - expected_id_set_D)}"
)


print(
    "✅ Prediction IDs exactly match active official-test IDs."
)


# =========================================================================
# 8. VALIDATE EACH PREDICTION
# =========================================================================

validated_predictions_D = {}


for test_id in OFFICIAL_TEST_IDS_D:

    prediction = (
        predictions_D_validation[
            test_id
        ]
        if test_id in predictions_D_validation
        else
        predictions_D_validation[
            str(test_id)
        ]
    )


    assert isinstance(
        prediction,
        dict,
    ), (
        f"Prediction {test_id} must be a dictionary."
    )


    assert set(
        prediction.keys()
    ) == {
        "text"
    }, (
        f"Prediction {test_id} must contain "
        "only the 'text' field. "
        f"Found: {list(prediction.keys())}"
    )


    continuation = prediction[
        "text"
    ]


    assert isinstance(
        continuation,
        str,
    ), (
        f"Prediction {test_id} text must be a string."
    )


    continuation = (
        continuation.strip()
    )


    assert continuation, (
        f"Prediction {test_id} text is empty."
    )


    word_count = len(
        continuation.split()
    )


    assert (
        word_count
        == TARGET_WORDS_D_VALIDATION
    ), (
        f"Prediction {test_id}: expected "
        f"{TARGET_WORDS_D_VALIDATION} words, "
        f"found {word_count}."
    )


    validated_predictions_D[
        str(test_id)
    ] = {
        "text": continuation
    }


# =========================================================================
# 9. PRESERVE OFFICIAL TEST ORDER
# =========================================================================

ordered_predictions_D = {

    str(test_id):
        validated_predictions_D[
            str(test_id)
        ]

    for test_id
    in OFFICIAL_TEST_IDS_D
}


assert (
    list(
        ordered_predictions_D.keys()
    )
    == OFFICIAL_TEST_IDS_D
)


print(
    f"✅ All {len(ordered_predictions_D):,} "
    "predictions passed format validation."
)

print(
    "✅ Exact word count passed for every prediction."
)


# =========================================================================
# 10. CREATE CLEAN SUBMISSION DIRECTORY
# =========================================================================

if SUBMISSION_DIR_D.exists():

    import shutil

    shutil.rmtree(
        SUBMISSION_DIR_D
    )


SUBMISSION_DIR_D.mkdir(
    parents=True,
    exist_ok=True,
)


# =========================================================================
# 11. WRITE CANONICAL prediction.json
# =========================================================================

with SUBMISSION_PREDICTION_PATH_D.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        ordered_predictions_D,
        file,
        ensure_ascii=False,
        indent=2,
    )


assert SUBMISSION_PREDICTION_PATH_D.is_file()


# =========================================================================
# 12. RE-READ CANONICAL prediction.json
# =========================================================================

with SUBMISSION_PREDICTION_PATH_D.open(
    "r",
    encoding="utf-8",
) as file:

    canonical_prediction_check_D = json.load(
        file
    )


assert (
    canonical_prediction_check_D
    == ordered_predictions_D
), (
    "Canonical prediction.json does not match "
    "validated predictions."
)


# =========================================================================
# 13. CREATE submission.zip
# =========================================================================

if SUBMISSION_ZIP_PATH_D.exists():

    SUBMISSION_ZIP_PATH_D.unlink()


with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH_D,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zip_file:

    # IMPORTANT:
    # arcname MUST be exactly prediction.json.
    # This puts the file directly at ZIP root.

    zip_file.write(
        SUBMISSION_PREDICTION_PATH_D,
        arcname="prediction.json",
    )


assert SUBMISSION_ZIP_PATH_D.is_file()


# =========================================================================
# 14. VERIFY ZIP STRUCTURE
# =========================================================================

with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH_D,
    mode="r",
) as zip_file:

    zip_contents_D = (
        zip_file.namelist()
    )


assert (
    zip_contents_D
    == [
        "prediction.json"
    ]
), (
    "submission.zip has an invalid structure.\n"
    f"Found: {zip_contents_D}\n"
    "Expected exactly ['prediction.json']."
)


print(
    "✅ submission.zip contains "
    "prediction.json directly at ZIP root."
)


# =========================================================================
# 15. VERIFY ZIP CONTENT
# =========================================================================

with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH_D,
    mode="r",
) as zip_file:

    zipped_prediction_D = json.loads(
        zip_file.read(
            "prediction.json"
        ).decode(
            "utf-8"
        )
    )


assert (
    zipped_prediction_D
    == ordered_predictions_D
), (
    "prediction.json inside the ZIP does not match "
    "the validated canonical prediction."
)


print(
    "✅ ZIP prediction content exactly matches "
    "the validated prediction."
)


# =========================================================================
# 16. SAVE SUBMISSION MANIFEST
# =========================================================================

submission_manifest_D = {

    "status": "validated",

    "project": "CS781 Phase 1",

    "experiment_id": (
        official_test_manifest_D[
            "experiment_id"
        ]
    ),

    "architecture_name": (
        official_test_manifest_D[
            "architecture_name"
        ]
    ),

    "architecture_version": (
        official_test_manifest_D[
            "architecture_version"
        ]
    ),

    "run_mode": (
        official_test_manifest_D[
            "run_mode"
        ]
    ),

    "official_test_count": 1000,

    "active_test_count": (
        ACTIVE_TEST_COUNT_D
    ),

    "target_words": (
        TARGET_WORDS_D_VALIDATION
    ),

    "official_test_ids": list(
        OFFICIAL_TEST_IDS_D
    ),

    "prediction_count": (
        len(
            ordered_predictions_D
        )
    ),

    "prediction_file": str(
        PREDICTION_PATH_D
    ),

    "submission_prediction_file": str(
        SUBMISSION_PREDICTION_PATH_D
    ),

    "submission_zip": str(
        SUBMISSION_ZIP_PATH_D
    ),

    "zip_contents": list(
        zip_contents_D
    ),

    "validation": {

        "prediction_is_dictionary": True,

        "ids_exactly_match_active_test_ids": True,

        "unique_ids": True,

        "text_only_field": True,

        "non_empty_text": True,

        "exact_word_count": True,

        "official_test_order_preserved": True,

        "zip_root_structure_verified": True,

        "zip_content_verified": True,
    },
}


with SUBMISSION_MANIFEST_PATH_D.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        submission_manifest_D,
        file,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# 17. FILE-SIZE INFORMATION
# =========================================================================

prediction_size_kb = (
    SUBMISSION_PREDICTION_PATH_D.stat().st_size
    / 1024
)

zip_size_kb = (
    SUBMISSION_ZIP_PATH_D.stat().st_size
    / 1024
)


# =========================================================================
# 18. FINAL SAFETY GATE
# =========================================================================

assert (
    len(
        ordered_predictions_D
    )
    == ACTIVE_TEST_COUNT_D
)

assert (
    list(
        ordered_predictions_D.keys()
    )
    == OFFICIAL_TEST_IDS_D
)

assert (
    SUBMISSION_ZIP_PATH_D.is_file()
)

assert (
    SUBMISSION_PREDICTION_PATH_D.is_file()
)

assert (
    zip_contents_D
    == [
        "prediction.json"
    ]
)


# =========================================================================
# 19. FINAL OUTPUT
# =========================================================================

print(
    "\n"
    + "=" * 70
)

print(
    "✅ CELL 14 COMPLETE — SUBMISSION PACKAGE VALIDATED"
)

print(
    "=" * 70
)

print(
    f"Run mode                 : "
    f"{official_test_manifest_D['run_mode']}"
)

print(
    f"Predictions              : "
    f"{len(ordered_predictions_D):,}"
)

print(
    f"Words / prediction       : "
    f"{TARGET_WORDS_D_VALIDATION}"
)

print(
    f"Prediction JSON          : "
    f"{PREDICTION_PATH_D}"
)

print(
    f"Submission prediction    : "
    f"{SUBMISSION_PREDICTION_PATH_D}"
)

print(
    f"Submission ZIP           : "
    f"{SUBMISSION_ZIP_PATH_D}"
)

print(
    f"Prediction size          : "
    f"{prediction_size_kb:.2f} KB"
)

print(
    f"ZIP size                 : "
    f"{zip_size_kb:.2f} KB"
)

print(
    f"Manifest                 : "
    f"{SUBMISSION_MANIFEST_PATH_D}"
)

print(
    "\n✅ Prediction IDs verified."
)

print(
    "✅ Prediction structure verified."
)

print(
    "✅ Exact word count verified."
)

print(
    "✅ Official test order preserved."
)

print(
    "✅ prediction.json is directly at ZIP root."
)

print(
    "✅ ZIP contents verified against canonical prediction.json."
)

print(
    "\n✅ Architecture-D fast submission package is ready."
)

print("=" * 70)

ARCHITECTURE-D PREDICTION VALIDATION + SUBMISSION

----------------------------------------------------------------------
PREDICTION FILE
----------------------------------------------------------------------
Prediction path          : /kaggle/working/architecture_D/predictions/prediction_D.json
Prediction JSON type     : dict
Prediction count         : 1,000
Expected active count    : 1,000
Target words             : 128
✅ Prediction IDs exactly match active official-test IDs.
✅ All 1,000 predictions passed format validation.
✅ Exact word count passed for every prediction.
✅ submission.zip contains prediction.json directly at ZIP root.
✅ ZIP prediction content exactly matches the validated prediction.

✅ CELL 14 COMPLETE — SUBMISSION PACKAGE VALIDATED
Run mode                 : production
Predictions              : 1,000
Words / prediction       : 128
Prediction JSON          : /kaggle/working/architecture_D/predictions/prediction_D.json
Submission prediction    : /kaggle/working/arch

In [15]:
# ======================================================================
# ARCHITECTURE-D — CELL 15
# FINAL END-TO-END AUDIT
# DECISION: REPLACE
# ======================================================================
#
# This audit is MODE-AWARE.
#
# FAST:
#     active examples = 2
#     words/example   = 16
#
# PRODUCTION:
#     active examples = 1000
#     words/example   = 128
#
# The audit never assumes FAST mode.
# It reads the current RUN_MODE and validates all artifacts against
# that mode.
# ======================================================================

from pathlib import Path
import hashlib
import json
import zipfile
import torch


print("=" * 70)
print("ARCHITECTURE-D FINAL END-TO-END AUDIT")
print("=" * 70)


# ======================================================================
# 1. ROOT PATHS
# ======================================================================

ROOT = Path(
    "/kaggle/working/architecture_D"
)

METRICS_DIR = (
    ROOT /
    "metrics"
)

CHECKPOINT_DIR = (
    ROOT /
    "checkpoints"
)

TEST_DIR = (
    ROOT /
    "test_data"
)

PREDICTIONS_DIR = (
    ROOT /
    "predictions"
)

CONFIG_DIR = (
    ROOT /
    "configs"
)

SUBMISSION_DIR = (
    ROOT /
    "submission_package"
)


# ======================================================================
# 2. FINAL ARTIFACT PATHS
# ======================================================================

MODEL_VALIDATION_REPORT = (
    METRICS_DIR /
    "model_validation_report_D.json"
)

BEST_CHECKPOINT = (
    CHECKPOINT_DIR /
    "best.pt"
)

LATEST_CHECKPOINT = (
    CHECKPOINT_DIR /
    "latest.pt"
)

OFFICIAL_TEST = (
    TEST_DIR /
    "official_phase1_test.json"
)

TEST_MANIFEST = (
    CONFIG_DIR /
    "official_test_manifest_D.json"
)

GENERATION_ENGINE = (
    CONFIG_DIR /
    "generation_engine_D.json"
)

GENERATION_SUMMARY = (
    CONFIG_DIR /
    "generation_summary_D.json"
)

PREDICTION = (
    PREDICTIONS_DIR /
    "prediction_D.json"
)

SUBMISSION_PREDICTION = (
    SUBMISSION_DIR /
    "prediction.json"
)

SUBMISSION_ZIP = (
    ROOT /
    "submission.zip"
)

SUBMISSION_MANIFEST = (
    CONFIG_DIR /
    "submission_manifest_D.json"
)

RUNTIME_CONFIGURATION = (
    CONFIG_DIR /
    "runtime_configuration_D.json"
)


# ======================================================================
# 3. REQUIRED FILE CHECK
# ======================================================================

required_files = {

    "model validation report":
        MODEL_VALIDATION_REPORT,

    "best checkpoint":
        BEST_CHECKPOINT,

    "latest checkpoint":
        LATEST_CHECKPOINT,

    "official test":
        OFFICIAL_TEST,

    "official test manifest":
        TEST_MANIFEST,

    "generation engine":
        GENERATION_ENGINE,

    "prediction":
        PREDICTION,

    "submission prediction":
        SUBMISSION_PREDICTION,

    "submission ZIP":
        SUBMISSION_ZIP,

    "submission manifest":
        SUBMISSION_MANIFEST,

}


missing_files = [

    name

    for name, path
    in required_files.items()

    if not path.is_file()

]


assert not missing_files, (

    "Missing required Architecture-D artifacts:\n"

    +
    "\n".join(
        f"  ❌ {name}"
        for name in missing_files
    )

)


print(
    "\n✅ All required final Architecture-D artifacts exist."
)


# ======================================================================
# 4. OPTIONAL GENERATION ARTIFACTS
# ======================================================================

generation_progress_candidates = [

    ROOT /
    "generation_progress_D.json",

    METRICS_DIR /
    "generation_progress_D.json",

    CONFIG_DIR /
    "generation_progress_D.json",

]


generation_progress_path = None


for candidate in (
    generation_progress_candidates
):

    if candidate.is_file():

        generation_progress_path = (
            candidate
        )

        break


if generation_progress_path is not None:

    print(
        "✅ Generation progress artifact found:"
    )

    print(
        "   ",
        generation_progress_path
    )

else:

    print(
        "ℹ️ Generation progress artifact not present."
    )

    print(
        "   Final prediction artifacts are used "
        "for completion validation."
    )


if GENERATION_SUMMARY.is_file():

    print(
        "✅ Generation summary artifact found."
    )


# ======================================================================
# 5. LOAD CONFIGURATION ARTIFACTS
# ======================================================================

with open(
    TEST_MANIFEST,
    "r",
    encoding="utf-8",
) as file:

    test_manifest = json.load(
        file
    )


with open(
    GENERATION_ENGINE,
    "r",
    encoding="utf-8",
) as file:

    generation_engine = json.load(
        file
    )


with open(
    SUBMISSION_MANIFEST,
    "r",
    encoding="utf-8",
) as file:

    submission_manifest = json.load(
        file
    )


generation_summary = None


if GENERATION_SUMMARY.is_file():

    with open(
        GENERATION_SUMMARY,
        "r",
        encoding="utf-8",
    ) as file:

        generation_summary = json.load(
            file
        )


runtime_configuration = None


if RUNTIME_CONFIGURATION.is_file():

    with open(
        RUNTIME_CONFIGURATION,
        "r",
        encoding="utf-8",
    ) as file:

        runtime_configuration = json.load(
            file
        )


# ======================================================================
# 6. RESOLVE CURRENT RUN MODE
# ======================================================================
#
# Prefer the current Python global from Cell 01.
# Otherwise use the runtime configuration.
# Otherwise use the test manifest.
# ======================================================================

if (
    "RUN_MODE" in globals()
):

    CURRENT_RUN_MODE = str(
        RUN_MODE
    )

elif (
    isinstance(
        runtime_configuration,
        dict,
    )
    and
    runtime_configuration.get(
        "run_mode"
    ) is not None
):

    CURRENT_RUN_MODE = str(
        runtime_configuration[
            "run_mode"
        ]
    )

else:

    CURRENT_RUN_MODE = str(
        test_manifest.get(
            "run_mode"
        )
    )


assert CURRENT_RUN_MODE in {

    "fast",

    "production",

}, (
    f"Unsupported run mode: "
    f"{CURRENT_RUN_MODE}"
)


print(
    "\nCurrent run mode :",
    CURRENT_RUN_MODE.upper()
)


# ======================================================================
# 7. MODE-SPECIFIC EXPECTATIONS
# ======================================================================

if CURRENT_RUN_MODE == "fast":

    EXPECTED_ACTIVE_COUNT = 2

    EXPECTED_WORDS = 16

    print(
        "Expected active examples : 2"
    )

    print(
        "Expected words/example   : 16"
    )

else:

    EXPECTED_ACTIVE_COUNT = 1000

    EXPECTED_WORDS = 128

    print(
        "Expected active examples : 1000"
    )

    print(
        "Expected words/example   : 128"
    )


# ======================================================================
# 8. VERIFY RUN-MODE CONSISTENCY
# ======================================================================

assert (
    test_manifest.get(
        "run_mode"
    )
    == CURRENT_RUN_MODE
), (
    "Test manifest run mode does not match "
    f"current run mode '{CURRENT_RUN_MODE}'."
)


assert (
    generation_engine.get(
        "run_mode"
    )
    == CURRENT_RUN_MODE
), (
    "Generation engine run mode does not match "
    f"current run mode '{CURRENT_RUN_MODE}'."
)


assert (
    submission_manifest.get(
        "run_mode"
    )
    == CURRENT_RUN_MODE
), (
    "Submission manifest run mode does not match "
    f"current run mode '{CURRENT_RUN_MODE}'."
)


if (
    runtime_configuration is not None
):

    if (
        runtime_configuration.get(
            "run_mode"
        )
        is not None
    ):

        assert (
            runtime_configuration[
                "run_mode"
            ]
            == CURRENT_RUN_MODE
        ), (
            "Runtime configuration run mode does "
            "not match current run mode."
        )


print(
    "✅ Run mode consistently recorded."
)


# ======================================================================
# 9. VERIFY GENERATION TARGET
# ======================================================================

target_words = None


if (
    "target_words"
    in generation_engine
):

    target_words = int(
        generation_engine[
            "target_words"
        ]
    )

elif (
    isinstance(
        generation_engine.get(
            "generation_config"
        ),
        dict,
    )
):

    generation_config = (
        generation_engine[
            "generation_config"
        ]
    )

    if (
        "target_words"
        in generation_config
    ):

        target_words = int(
            generation_config[
                "target_words"
            ]
        )


if (
    target_words is None
    and
    generation_summary is not None
):

    for key in [

        "target_words",

        "generation_words_per_example",

    ]:

        if (
            key in generation_summary
        ):

            target_words = int(
                generation_summary[
                    key
                ]
            )

            break


assert (
    target_words is not None
), (
    "Could not determine generation target word count."
)


assert (
    target_words
    == EXPECTED_WORDS
), (
    f"Generation target mismatch: "
    f"configured={target_words}, "
    f"expected={EXPECTED_WORDS} "
    f"for {CURRENT_RUN_MODE} mode."
)


print(
    "✅ Generation target verified:"
)

print(
    "   Words / continuation :",
    target_words
)


# ======================================================================
# 10. LOAD BEST CHECKPOINT
# ======================================================================

checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)


assert isinstance(
    checkpoint,
    dict,
), (
    "Best checkpoint must be a dictionary."
)


assert (
    "model_state_dict"
    in checkpoint
), (
    "Best checkpoint is missing model_state_dict."
)


model_state_dict = (
    checkpoint[
        "model_state_dict"
    ]
)


state_tensor_count = 0


for tensor in (
    model_state_dict.values()
):

    if torch.is_tensor(
        tensor
    ):

        state_tensor_count += int(
            tensor.numel()
        )


assert (
    50_000_000
    <= state_tensor_count
    <= 300_000_000
), (
    f"Architecture-D checkpoint contains "
    f"{state_tensor_count:,} parameters, outside "
    "the required 50M–300M range."
)


print(
    "✅ Best checkpoint parameter/state size:"
)

print(
    f"   {state_tensor_count:,}"
    f" ({state_tensor_count / 1e6:.3f}M)"
)


# ======================================================================
# 11. MODEL FINGERPRINT
# ======================================================================

with open(
    MODEL_VALIDATION_REPORT,
    "r",
    encoding="utf-8",
) as file:

    model_report = json.load(
        file
    )


model_fingerprint = model_report.get(
    "model_fingerprint"
)


if model_fingerprint is not None:

    print(
        "✅ Model fingerprint:"
    )

    print(
        "   ",
        model_fingerprint
    )


# ======================================================================
# 12. LOAD OFFICIAL TEST SET
# ======================================================================

with open(
    OFFICIAL_TEST,
    "r",
    encoding="utf-8",
) as file:

    raw_official_test = json.load(
        file
    )


print(
    "\nOfficial-test raw JSON type :",
    type(raw_official_test).__name__
)


# ======================================================================
# 13. NORMALIZE OFFICIAL TEST
# ======================================================================

def normalize_official_test(
    raw
):

    if isinstance(
        raw,
        list,
    ):

        return [
            dict(record)
            for record in raw
        ]


    if isinstance(
        raw,
        dict,
    ):

        for wrapper_key in [

            "data",

            "records",

            "examples",

            "test",

            "items",

        ]:

            wrapped = raw.get(
                wrapper_key
            )


            if isinstance(
                wrapped,
                list,
            ):

                return [
                    dict(record)
                    for record in wrapped
                ]


        records = []


        for record_id, value in (
            raw.items()
        ):

            if not isinstance(
                value,
                dict,
            ):

                continue


            record = dict(
                value
            )


            if not any(

                key in record

                for key in [

                    "id",

                    "ID",

                    "test_id",

                    "testId",

                ]

            ):

                record[
                    "id"
                ] = str(
                    record_id
                )


            records.append(
                record
            )


        if records:

            return records


    raise TypeError(
        "Unsupported official-test JSON structure: "
        f"{type(raw).__name__}"
    )


official_test_records = (
    normalize_official_test(
        raw_official_test
    )
)


assert (
    len(official_test_records)
    == 1000
), (
    "Official Phase-1 test set must contain "
    f"exactly 1000 examples, found "
    f"{len(official_test_records)}."
)


# ======================================================================
# 14. OFFICIAL TEST IDS + PROMPTS
# ======================================================================

def get_test_id(
    record
):

    for key in [

        "id",

        "ID",

        "test_id",

        "testId",

    ]:

        if key in record:

            return str(
                record[key]
            )


    raise KeyError(
        "Could not resolve official test ID."
    )


def get_test_prompt(
    record
):

    for key in [

        "prompt",

        "text",

        "input",

    ]:

        if key in record:

            return str(
                record[key]
            )


    raise KeyError(
        "Could not resolve official test prompt."
    )


official_ids = []

prompt_counts = []


for record in (
    official_test_records
):

    example_id = get_test_id(
        record
    )

    prompt = get_test_prompt(
        record
    )


    assert (
        example_id
        not in official_ids
    ), (
        f"Duplicate official test ID: "
        f"{example_id}"
    )


    word_count = len(
        prompt.split()
    )


    assert (
        word_count == 512
    ), (
        f"Official prompt {example_id} "
        f"has {word_count} words instead of 512."
    )


    official_ids.append(
        example_id
    )

    prompt_counts.append(
        word_count
    )


print(
    "✅ Official Phase-1 test set verified."
)

print(
    "   Examples     :",
    len(official_ids)
)

print(
    "   Prompt words :",
    min(prompt_counts),
    "to",
    max(prompt_counts)
)


# ======================================================================
# 15. ACTIVE TEST IDS
# ======================================================================

assert (
    "active_test_ids"
    in test_manifest
), (
    "Test manifest does not contain active_test_ids."
)


active_ids = [

    str(example_id)

    for example_id
    in test_manifest[
        "active_test_ids"
    ]

]


assert (
    len(active_ids)
    == EXPECTED_ACTIVE_COUNT
), (
    f"Expected {EXPECTED_ACTIVE_COUNT} active test IDs "
    f"in {CURRENT_RUN_MODE} mode, found "
    f"{len(active_ids)}."
)


assert (
    len(active_ids)
    == len(set(active_ids))
), (
    "Duplicate active test IDs detected."
)


assert (
    set(active_ids)
    <= set(official_ids)
), (
    "Active IDs contain IDs outside the official test set."
)


print(
    "✅ Active official-test IDs verified."
)

print(
    "   Active examples :",
    len(active_ids)
)


# ======================================================================
# 16. LOAD FINAL PREDICTION
# ======================================================================

with open(
    PREDICTION,
    "r",
    encoding="utf-8",
) as file:

    prediction = json.load(
        file
    )


with open(
    SUBMISSION_PREDICTION,
    "r",
    encoding="utf-8",
) as file:

    submission_prediction = json.load(
        file
    )


assert isinstance(
    prediction,
    dict,
), (
    "prediction_D.json must be a dictionary."
)


assert isinstance(
    submission_prediction,
    dict,
), (
    "submission prediction must be a dictionary."
)


# ======================================================================
# 17. PREDICTION COUNT + ORDER
# ======================================================================

assert (
    len(prediction)
    == EXPECTED_ACTIVE_COUNT
), (
    f"Expected {EXPECTED_ACTIVE_COUNT} predictions "
    f"for {CURRENT_RUN_MODE} mode, found "
    f"{len(prediction)}."
)


assert (
    list(prediction.keys())
    == active_ids
), (
    "Prediction IDs/order do not match "
    "the active official-test order."
)


# ======================================================================
# 18. VALIDATE EVERY PREDICTION
# ======================================================================

for example_id in active_ids:

    entry = prediction[
        example_id
    ]


    assert isinstance(
        entry,
        dict,
    ), (
        f"Prediction {example_id} "
        "must be a dictionary."
    )


    assert (
        set(entry.keys())
        == {"text"}
    ), (
        f"Prediction {example_id} must contain "
        "only the text field."
    )


    text_value = entry[
        "text"
    ]


    assert isinstance(
        text_value,
        str,
    ), (
        f"Prediction {example_id} text "
        "must be a string."
    )


    assert (
        text_value.strip()
    ), (
        f"Prediction {example_id} "
        "contains empty text."
    )


    generated_word_count = len(
        text_value.strip().split()
    )


    assert (
        generated_word_count
        == EXPECTED_WORDS
    ), (
        f"Prediction {example_id}: "
        f"expected {EXPECTED_WORDS} words, "
        f"found {generated_word_count}."
    )


print(
    "✅ Final prediction format verified."
)

print(
    "   Predictions   :",
    len(prediction)
)

print(
    "   Words/example :",
    EXPECTED_WORDS
)

print(
    "   IDs/order     : VALID"
)

print(
    "   Structure     : VALID"
)


# ======================================================================
# 19. SUBMISSION COPY
# ======================================================================

assert (
    prediction
    == submission_prediction
), (
    "submission_package/prediction.json does not "
    "exactly match prediction_D.json."
)


print(
    "✅ Submission prediction is an exact copy "
    "of the final prediction."
)


# ======================================================================
# 20. ZIP STRUCTURE
# ======================================================================

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "r",
) as zip_file:

    zip_members = (
        zip_file.namelist()
    )


    assert (
        zip_members
        == ["prediction.json"]
    ), (
        "submission.zip must contain exactly "
        "'prediction.json' at ZIP root. "
        f"Found: {zip_members}"
    )


    zipped_prediction = (
        zip_file.read(
            "prediction.json"
        )
    )


with open(
    SUBMISSION_PREDICTION,
    "rb",
) as file:

    canonical_prediction = (
        file.read()
    )


assert (
    zipped_prediction
    == canonical_prediction
), (
    "ZIP prediction.json differs from canonical "
    "submission prediction."
)


print(
    "✅ submission.zip verified."
)

print(
    "   ZIP root      : prediction.json"
)

print(
    "   Content match : EXACT"
)


# ======================================================================
# 21. OPTIONAL GENERATION SUMMARY CHECK
# ======================================================================

if generation_summary is not None:

    summary_run_mode = (
        generation_summary.get(
            "run_mode"
        )
    )


    if summary_run_mode is not None:

        assert (
            summary_run_mode
            == CURRENT_RUN_MODE
        ), (
            "Generation summary run mode does not "
            "match current run mode."
        )


    summary_active_count = None


    for key in [

        "active_test_count",

        "active_test_examples",

    ]:

        if key in generation_summary:

            summary_active_count = int(
                generation_summary[
                    key
                ]
            )

            break


    if (
        summary_active_count is not None
    ):

        assert (
            summary_active_count
            == EXPECTED_ACTIVE_COUNT
        ), (
            "Generation summary active count does not "
            "match the current mode."
        )


    summary_target_words = None


    for key in [

        "target_words",

        "generation_words_per_example",

    ]:

        if key in generation_summary:

            summary_target_words = int(
                generation_summary[
                    key
                ]
            )

            break


    if (
        summary_target_words is not None
    ):

        assert (
            summary_target_words
            == EXPECTED_WORDS
        ), (
            "Generation summary word target does not "
            "match the current mode."
        )


    print(
        "✅ Generation summary cross-check passed."
    )


# ======================================================================
# 22. FILE HASHES
# ======================================================================

def sha256_file(
    path: Path
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb",
    ) as file:

        while True:

            chunk = file.read(
                1024 * 1024
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


prediction_hash = (
    sha256_file(
        PREDICTION
    )
)


zip_hash = (
    sha256_file(
        SUBMISSION_ZIP
    )
)


print(
    "\nArtifact integrity:"
)

print(
    "   prediction_D.json SHA256:"
)

print(
    "   ",
    prediction_hash
)

print(
    "   submission.zip SHA256:"
)

print(
    "   ",
    zip_hash
)

print(
    "   Prediction size:",
    f"{PREDICTION.stat().st_size / 1024:.2f} KB"
)

print(
    "   ZIP size:",
    f"{SUBMISSION_ZIP.stat().st_size / 1024:.2f} KB"
)


# ======================================================================
# 23. FINAL RESULT
# ======================================================================

print(
    "\n" + "=" * 70
)

print(
    "✅ CELL 15 COMPLETE — "
    "ARCHITECTURE-D FINAL AUDIT PASSED"
)

print(
    "=" * 70
)

print(
    "Run mode                   :",
    CURRENT_RUN_MODE.upper()
)

print(
    f"Model size                 : "
    f"{state_tensor_count / 1e6:.3f}M"
)

print(
    "Official test examples     : 1000"
)

print(
    f"Active test examples       : "
    f"{EXPECTED_ACTIVE_COUNT}"
)

print(
    f"Continuation words/example : "
    f"{EXPECTED_WORDS}"
)

print(
    "Prediction format          : VALID"
)

print(
    "Official test              : VALID"
)

print(
    "Active IDs/order           : VALID"
)

print(
    "Generation output          : VALID"
)

print(
    "Submission prediction      : VALID"
)

print(
    "ZIP root structure         : VALID"
)

print(
    "ZIP content                : VALID"
)

print(
    "Best checkpoint            : VALID"
)


print(
    "\nFinal artifacts:"
)

print(
    "  Best checkpoint :",
    BEST_CHECKPOINT
)

print(
    "  Prediction      :",
    PREDICTION
)

print(
    "  Submission ZIP  :",
    SUBMISSION_ZIP
)


if CURRENT_RUN_MODE == "production":

    print(
        "\n✅ Architecture-D PRODUCTION pipeline "
        "has passed the final audit."
    )

    print(
        "✅ 1000 official examples validated."
    )

    print(
        "✅ 128-word continuation requirement validated."
    )

else:

    print(
        "\n✅ Architecture-D FAST pipeline "
        "has passed the final audit."
    )

    print(
        "⚠️ FAST mode is only the smoke-test "
        "configuration."
    )


print(
    "=" * 70
)

ARCHITECTURE-D FINAL END-TO-END AUDIT

✅ All required final Architecture-D artifacts exist.
ℹ️ Generation progress artifact not present.
   Final prediction artifacts are used for completion validation.
✅ Generation summary artifact found.

Current run mode : PRODUCTION
Expected active examples : 1000
Expected words/example   : 128
✅ Run mode consistently recorded.
✅ Generation target verified:
   Words / continuation : 128
✅ Best checkpoint parameter/state size:
   114,155,520 (114.156M)
✅ Model fingerprint:
    28289628a39c1ae28de251c6a58d6a461b3b3ae4d35d97f158500eaedc4539c0

Official-test raw JSON type : dict
✅ Official Phase-1 test set verified.
   Examples     : 1000
   Prompt words : 512 to 512
✅ Active official-test IDs verified.
   Active examples : 1000
✅ Final prediction format verified.
   Predictions   : 1000
   Words/example : 128
   IDs/order     : VALID
   Structure     : VALID
✅ Submission prediction is an exact copy of the final prediction.
✅ submission.zip verified.
 